# CS781 Capstone: Strided GQA decoder, Phases 1 to 7 (full notebook)

This notebook trains the **Strided GQA decoder** from scratch (Phase 1) and then runs every downstream phase on it: Phase 2 NER, Phase 3 rhetorical roles, Phase 4 court judgment prediction, Phase 5 statute identification, Phase 6 summarisation and Phase 7 prior case retrieval. Each phase starts at a heading with its name. Run the cells **top to bottom**; later phases reuse the tokenizer, model classes and helper functions defined earlier. See the repository README for set-up (Colab GPU, Drive, data access secret) and the per-phase folders for the scores of every run.

## 1. Data Access Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from your Colab Secrets (make sure a secret named 'HF' is saved in your notebook settings)
HF_TOKEN = userdata.get('HF')
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful on Google Colab.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful on Google Colab.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU count: 1


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Updated training files list (Supreme Court, Acts, and Tribunals)
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json"
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 1_500_000_000
VAL_TOKENS = 10_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 4096

# Model architecture configured to match the previous run (n_layer = 8)
N_EMBD = 768
N_LAYER = 8
N_HEAD = 12

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.6
TOP_P = 0.9

print("Configuration updated successfully.")

Configuration updated successfully.


## 4. Load Tokenizer

### Tokenizer implementation (from scratch)

Byte-level BPE written by hand: word splitting, merge training, encoding and decoding. The cells below load the saved merge table with it.

In [ ]:
# ============================================================
# Byte-level BPE tokenizer, implemented from scratch
#   - pretokenize : split text into words with a regular expression, map every byte to a printable symbol
#   - train_bpe   : learn the merge table (counts word frequencies, then repeatedly merges the most frequent pair)
#   - ByteBPE     : encode / decode with a learned merge table
# ============================================================
import json, heapq, time
from collections import Counter, defaultdict
try:
    import regex as re          # needed for the Unicode classes \p{L} and \p{N} (installed on Colab)
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
except ImportError:
    import re
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
PAT = re.compile(PAT)

def bytes_to_unicode():
    """Map each of the 256 byte values to one printable character (so every token is a readable string)."""
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}
B2U = bytes_to_unicode()
U2B = {v: k for k, v in B2U.items()}
SPECIALS = ["<|endoftext|>", "[PAD]"]
SPECIAL_SPLIT = re.compile("(" + "|".join(re.escape(s) for s in SPECIALS) + ")")

def pretokenize(text):
    return ["".join(B2U[b] for b in w.encode("utf-8")) for w in PAT.findall(text)]

def train_bpe(texts, vocab_size=32000, min_frequency=2):
    wc = Counter()
    for t in texts:
        wc.update(pretokenize(t))
    alphabet = sorted({c for w in wc for c in w})
    vocab = {s: i for i, s in enumerate(SPECIALS)}
    for c in alphabet: vocab[c] = len(vocab)
    words = [[vocab[c] for c in w] for w in wc]
    freqs = list(wc.values())
    pair_cnt = defaultdict(int); where = defaultdict(set)
    for wi, w in enumerate(words):
        for p in zip(w, w[1:]):
            pair_cnt[p] += freqs[wi]; where[p].add(wi)
    heap = [(-c, p) for p, c in pair_cnt.items()]; heapq.heapify(heap)
    id2tok = {i: s for s, i in vocab.items()}; merges = []
    while len(vocab) < vocab_size and heap:
        negc, pair = heapq.heappop(heap)
        if pair_cnt.get(pair, 0) != -negc:
            if pair_cnt.get(pair, 0) > 0: heapq.heappush(heap, (-pair_cnt[pair], pair))
            continue
        if -negc < min_frequency: break
        a, b = pair; new_tok = id2tok[a] + id2tok[b]
        new_id = len(vocab); vocab[new_tok] = new_id; id2tok[new_id] = new_tok
        merges.append((id2tok[a], id2tok[b]))
        touched = set()
        for wi in list(where[pair]):
            w, f = words[wi], freqs[wi]
            for p in zip(w, w[1:]):
                pair_cnt[p] -= f
                if p != pair: touched.add(p)
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: out.append(new_id); i += 2
                else: out.append(w[i]); i += 1
            words[wi] = out
            for p in zip(out, out[1:]):
                pair_cnt[p] += f; where[p].add(wi); touched.add(p)
        pair_cnt[pair] = 0; del where[pair]
        for p in touched:
            if pair_cnt[p] > 0: heapq.heappush(heap, (-pair_cnt[p], p))
    return vocab, merges

class _Enc:                                   # small result object so callers can write  tokenizer.encode(text).ids
    def __init__(self, ids): self.ids = ids

class ByteBPE:
    def __init__(self, vocab, merges):
        self.vocab, self.id2tok = vocab, {i: t for t, i in vocab.items()}
        self.merges = [tuple(m) for m in merges]
        self.rank = {m: r for r, m in enumerate(self.merges)}; self.cache = {}
    def _parts(self, w):
        parts = list(w)
        while len(parts) > 1:
            best = min((self.rank.get(p, 1 << 30), i) for i, p in enumerate(zip(parts, parts[1:])))
            if best[0] == 1 << 30: break
            i = best[1]; parts[i:i + 2] = [parts[i] + parts[i + 1]]
        return parts
    def _bpe(self, w):
        if w not in self.cache:
            self.cache[w] = [self.vocab.get(p, 0) for p in self._parts(w)]   # a symbol never seen in training maps to id 0
        return self.cache[w]
    def get_vocab_size(self): return len(self.vocab)
    def token_to_id(self, t): return self.vocab.get(t)
    def encode_ids(self, text):
        ids = []
        for piece in SPECIAL_SPLIT.split(text):
            if piece in self.vocab and piece in SPECIALS: ids.append(self.vocab[piece])
            elif piece:
                for w in pretokenize(piece): ids += self._bpe(w)
        return ids
    def encode(self, text):
        """Like encode_ids, but also returns the character span (start, end) of every token in `text`.
        Leading and trailing spaces are trimmed from a token's span, so the span covers only visible characters."""
        ids, toks, offs = [], [], []
        pos = 0
        for piece in SPECIAL_SPLIT.split(text):
            if piece in SPECIALS:
                ids.append(self.vocab[piece]); toks.append(piece); offs.append((pos, pos + len(piece)))
            elif piece:
                for m in PAT.finditer(piece):
                    word = m.group(0); cstart = pos + m.start()
                    b2c = [k for k, ch in enumerate(word) for _ in ch.encode("utf-8")]   # byte index -> character index
                    w = "".join(B2U[b] for b in word.encode("utf-8")); b = 0
                    for p in self._parts(w):
                        s0, e0 = cstart + b2c[b], cstart + b2c[b + len(p) - 1] + 1
                        lead = len(p) - len(p.lstrip("Ġ")); trail = 0   # Ġ is the symbol for a space byte
                        s1 = min(s0 + lead, e0)
                        ids.append(self.vocab.get(p, 0)); toks.append(p); offs.append((s1, e0)); b += len(p)
            pos += len(piece)
        e = _Enc(ids); e.tokens, e.offsets = toks, offs
        return e
    def encode_batch(self, texts):
        out = []
        for t in texts:
            e = _Enc(self.encode_ids(t)); out.append(e)      # batch use only needs ids; use encode() when offsets are needed
        return out
    def decode(self, ids):
        out = bytearray()
        for i in ids:
            t = self.id2tok[int(i)]
            out += t.encode() if t in SPECIALS else bytes(U2B[c] for c in t)
        return out.decode("utf-8", errors="replace")
    @classmethod
    def from_json(cls, path):
        d = json.load(open(path)); m = d.get("model", d)       # accepts the saved merge-table file
        return cls(m["vocab"], [tuple(x.split(" ")) if isinstance(x, str) else tuple(x) for x in m["merges"]])
    def save(self, path):
        json.dump({"vocab": self.vocab, "merges": [list(m) for m in self.merges]}, open(path, "w"))


In [4]:
import os, json, glob, shutil, torch
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shutil.copy(f"{DRIVE_PROJECT_DIR}/legal_bpe_32k_FINAL.json", "legal_bpe_32k_FINAL.json")
tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")
assert tokenizer.get_vocab_size() == 32000
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# HF_TOKEN = "..."   # set it as before (the Phase-2 setup cell used it)
print("device:", device, "| vocab:", tokenizer.get_vocab_size())
for p in sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True)): print(" ", p)

Mounted at /content/drive
device: cuda | vocab: 32000
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt
  /content/drive/M

In [ ]:
import os, json, glob, shutil, time
import torch
from datasets import load_from_disk

# ============================================================
# 0. Settings
# ============================================================
RETRAIN_TOKENIZER = False   # keep False: we already trained and verified legal_bpe_32k_FINAL.json
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
TOKENIZER_NAME = "legal_bpe_32k_FINAL.json"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"   # needed again for the test-file download

# ============================================================
# 1. Mount Drive + device
# ============================================================
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
assert os.path.isdir(DRIVE_PROJECT_DIR), f"Project folder not found: {DRIVE_PROJECT_DIR}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# 2. Load the tokenizer (no training)
# ============================================================
tok_path = os.path.join(DRIVE_PROJECT_DIR, TOKENIZER_NAME)
if not os.path.exists(tok_path):
    # We may have saved it somewhere else; look for it, but never silently pick an old file
    hits = [p for p in glob.glob("/content/drive/MyDrive/**/legal_bpe_32k_FINAL.json", recursive=True)]
    print("Tokenizer not at the expected path. Found:", hits)
    assert hits, "legal_bpe_32k_FINAL.json is not on Drive. Re-run the tokenizer cell once, then copy it to Drive."
    tok_path = hits[0]

shutil.copy(tok_path, TOKENIZER_NAME)          # local copy for fast loading
tokenizer = ByteBPE.from_json(TOKENIZER_NAME)
vocab_sz = tokenizer.get_vocab_size()
print(f"Tokenizer loaded from {tok_path} | vocab size: {vocab_sz}")

endoftext_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("[PAD]")
print(f"<|endoftext|> id = {endoftext_id} | [PAD] id = {pad_id}")

# ============================================================
# 3. Sanity checks (same ones as before)
# ============================================================
for text in ["The Supreme Court held that statutory interpretation must adhere to the text.",
             "the court held that"]:
    enc = tokenizer.encode(text)
    dec = tokenizer.decode(enc.ids)
    print(f"{len(enc.ids):3d} tokens | {dec}")
    assert dec.strip() != "", "Decode returned an empty string, so stop and debug."
assert vocab_sz == 32000, f"Expected 32000, got {vocab_sz}"

# ============================================================
# 4. Optional retraining, only runs if you flip the flag
# ============================================================
if RETRAIN_TOKENIZER:
    from huggingface_hub import hf_hub_download
    TRAIN_FILES = ["Phase_1_Pretraining/Train/train_acts.json",
                   "Phase_1_Pretraining/Train/train_supreme_court.json"]

    def training_text_generator():
        for file_path in TRAIN_FILES:
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=file_path, repo_type="dataset")
            with open(local_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                for record in (list(data.values()) if isinstance(data, dict) else data):
                    if isinstance(record, dict) and "text" in record:
                        yield record["text"]
                    elif isinstance(record, str):
                        yield record

    t0 = time.time()
    new_vocab, new_merges = train_bpe(training_text_generator(), vocab_size=32000, min_frequency=2)
    print(f"Retrained in {time.time() - t0:.0f}s")
    ByteBPE(new_vocab, new_merges).save("legal_bpe_32k_NEW.json")   # new filename on purpose, so the verified file is never overwritten
    print("Saved as legal_bpe_32k_NEW.json. If you use it, you must re-pack the data and retrain the model.")

# ============================================================
# 5. Load the packed datasets (copy to local disk first: much faster than reading from Drive)
# ============================================================
drive_data = os.path.join(DRIVE_PROJECT_DIR, "packed_pretrain_data")
local_data = "packed_pretrain_data"
if os.path.isdir(os.path.join(drive_data, "train")) and not os.path.isdir(os.path.join(local_data, "train")):
    shutil.copytree(drive_data, local_data)
    print(f"Copied packed data from Drive to {local_data}/")
assert os.path.isdir(os.path.join(local_data, "train")), \
    "Packed data not found on Drive or locally. Re-run the packing script, then copy it to Drive."

train_dataset = load_from_disk(f"{local_data}/train")
val_dataset = load_from_disk(f"{local_data}/val")
train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])
print(f"Train blocks: {len(train_dataset):,} | Val blocks: {len(val_dataset):,} | block length: {len(train_dataset[0]['input_ids'])}")

sample = tokenizer.decode(train_dataset[0]["input_ids"][:50].tolist())
assert sample.strip() != "", "Packed data decodes to empty, so the data and tokenizer don't match."
assert int(train_dataset[0]["input_ids"].max()) < vocab_sz, "Packed data contains ids outside the tokenizer's vocab."
print(f"Data matches tokenizer. Sample: {sample[:100]}...")

# ============================================================
# 6. Find saved model checkpoints (if any)
# ============================================================
ckpts = sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True))
print("\nCheckpoints on Drive:" if ckpts else "\nNo checkpoints on Drive yet.")
for p in ckpts:
    print(f"  {p}  ({os.path.getsize(p) / 1e6:.0f} MB)")

# ============================================================
# 7. Test file (needs your HF token)
# ============================================================
if "HF_TOKEN" in globals():
    from huggingface_hub import hf_hub_download
    test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename="Phase_1_Pretraining/Test/test.json",
                                     repo_type="dataset", token=HF_TOKEN)
    with open(test_file_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    print(f"\nTest examples: {len(test_data)}")
else:
    print("\nHF_TOKEN is not defined, so test.json was skipped. Set HF_TOKEN and re-run this cell to load test_data.")

print("\n✅ Ready: tokenizer, train_dataset, val_dataset, device, vocab_sz are defined.")

Mounted at /content/drive
Device: cuda
Tokenizer loaded from /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/legal_bpe_32k_FINAL.json | vocab size: 32000
<|endoftext|> id = 0 | [PAD] id = 1
 13 tokens | The Supreme Court held that statutory interpretation must adhere to the text.
  4 tokens | the court held that
Copied packed data from Drive to packed_pretrain_data/
Train blocks: 190,794 | Val blocks: 10,207 | block length: 2048
Data matches tokenizer. Sample: The State of Karnataka (Special Responsibility of Governor for Hyderabad-Karnataka Region) Order, 20...

Checkpoints on Drive:
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt  (446 MB)
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt  (1050 MB)
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt  (1050 MB)
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkp

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]


Test examples: 1000

✅ Ready: tokenizer, train_dataset, val_dataset, device, vocab_sz are defined.


In [ ]:
for name in ["tokenizer", "train_dataset", "val_dataset", "vocab_sz", "device"]:
    print(name, "->", "defined" if name in globals() else "MISSING")

tokenizer -> defined
train_dataset -> defined
val_dataset -> defined
vocab_sz -> defined
device -> defined


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Pick a clear destination folder - adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
os.makedirs(DEST_DIR, exist_ok=True)

src_path = "legal_bpe_32k_FINAL.json"
dest_path = os.path.join(DEST_DIR, "legal_bpe_32k_FINAL.json")

shutil.copy(src_path, dest_path)
print(f"Saved to: {dest_path}")

# Quick confirmation it actually copied correctly
check = ByteBPE.from_json(dest_path)
print(f"Verified — loads fine from Drive, vocab size: {check.get_vocab_size()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/legal_bpe_32k_FINAL.json
Verified — loads fine from Drive, vocab size: 32000


## 5. Download Training Data

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = []

for file_path in TRAIN_FILES:
    print(f"Downloading '{file_path}' from repository '{HF_DATASET_REPO}'...")
    downloaded_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    train_file_paths.append(downloaded_path)
    print(f"✅ Downloaded to: {downloaded_path}\n")

print("All selected training files downloaded successfully!")
print(train_file_paths)

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json



Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json



Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

All selected training files downloaded successfully!
['/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json']


## 6. Create Train and Validation Tokens Set

In [ ]:
import json
from huggingface_hub import hf_hub_download
from datasets import Dataset
import numpy as np
import os
import time
from tqdm import tqdm

# 1. Load your verified tokenizer — MAKE SURE THIS IS THE CORRECT FILE
tokenizer_path = "legal_bpe_32k_FINAL.json"
tokenizer = ByteBPE.from_json(tokenizer_path)
print(f"✅ Loaded tokenizer from {tokenizer_path} (Vocab size: {tokenizer.get_vocab_size()})")

_test = tokenizer.encode("the court held that")
assert tokenizer.decode(_test.ids).strip() != "", "Tokenizer decode failed sanity check — STOP."
print("Tokenizer sanity check passed.")

# 2. Gather raw text — everything except high courts
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
]

all_texts = []
for file_path in tqdm(TRAIN_FILES, desc="Downloading & Reading Dataset Files"):
    local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=file_path, repo_type="dataset")
    with open(local_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        records = list(data.values()) if isinstance(data, dict) else data
        for record in records:
            text = record.get("text") if isinstance(record, dict) else record
            if text and isinstance(text, str):
                all_texts.append(text)

print(f"Total raw texts collected: {len(all_texts)}")

# 3. Train/Val split
raw_dataset = Dataset.from_dict({"text": all_texts})
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train samples: {len(split_dataset['train'])} | Val samples: {len(split_dataset['test'])}")

# 4. Tokenize + pack — SMALL batch_size and writer_batch_size to avoid memory pressure/swapping
BLOCK_SIZE = 2048
endoftext_id = tokenizer.token_to_id("<|endoftext|>")
if endoftext_id is None:
    endoftext_id = 256

def tokenize_and_pack(examples):
    batch_encodings = tokenizer.encode_batch(examples["text"])
    all_token_ids = []
    for encoding in batch_encodings:
        all_token_ids.extend(encoding.ids)
        all_token_ids.append(endoftext_id)

    total_length = (len(all_token_ids) // BLOCK_SIZE) * BLOCK_SIZE
    all_token_ids = all_token_ids[:total_length]
    chunked_ids = np.array(all_token_ids, dtype=np.int32).reshape(-1, BLOCK_SIZE)

    return {"input_ids": chunked_ids.tolist(), "labels": chunked_ids.tolist()}

PACK_BATCH_SIZE = 500      # small, avoids huge in-RAM lists per batch
WRITER_BATCH_SIZE = 500    # flush to disk often, don't hoard memory
NUM_PROC = 2                # safe default for Colab; don't blindly trust os.cpu_count()

print(f"Detected CPU count: {os.cpu_count()} (using NUM_PROC={NUM_PROC} regardless)")

print(f"⚙️ Tokenizing and packing training set...")
t0 = time.time()
tokenized_train = split_dataset["train"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Train Split"
)
print(f"Train packing took {time.time() - t0:.1f}s")

print("⚙️ Tokenizing and packing validation set...")
t0 = time.time()
tokenized_val = split_dataset["test"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Val Split"
)
print(f"Val packing took {time.time() - t0:.1f}s")

# 5. Save
output_dir = "packed_pretrain_data"
os.makedirs(output_dir, exist_ok=True)
tokenized_train.save_to_disk(os.path.join(output_dir, "train"))
tokenized_val.save_to_disk(os.path.join(output_dir, "val"))

print(f"\n🎉 Saved to '{output_dir}/'")
print(f"   - Train blocks: {len(tokenized_train)} (~{len(tokenized_train) * BLOCK_SIZE / 1e6:.2f}M tokens)")
print(f"   - Val blocks:   {len(tokenized_val)} (~{len(tokenized_val) * BLOCK_SIZE / 1e6:.2f}M tokens)")

# 6. Verify packed data decodes correctly
sample_ids = tokenized_train[0]["input_ids"]
decoded_sample = tokenizer.decode(sample_ids[:50])
print(f"\n--- Packed block decode check ---\n{decoded_sample}")
assert decoded_sample.strip() != "", "Packed data decodes to empty — tokenizer/data mismatch!"
print("Packed data verified readable.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Destination folder on Drive — adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data"
os.makedirs(DEST_DIR, exist_ok=True)

src_dir = "packed_pretrain_data"  # the local folder your script already saved to

for split in ["train", "val"]:
    src_path = os.path.join(src_dir, split)
    dest_path = os.path.join(DEST_DIR, split)

    if os.path.exists(dest_path):
        shutil.rmtree(dest_path)  # avoid stale leftovers from a previous run

    shutil.copytree(src_path, dest_path)
    print(f"Copied {src_path} -> {dest_path}")

print("\nDone. Verifying by reloading from Drive...")

# Verify both splits reload correctly straight from Drive
from datasets import load_from_disk

train_check = load_from_disk(os.path.join(DEST_DIR, "train"))
val_check = load_from_disk(os.path.join(DEST_DIR, "val"))

print(f"Train blocks on Drive: {len(train_check)}")
print(f"Val blocks on Drive:   {len(val_check)}")
print(f"Sample input_ids length: {len(train_check[0]['input_ids'])}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied packed_pretrain_data/train -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data/train
Copied packed_pretrain_data/val -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data/val

Done. Verifying by reloading from Drive...
Train blocks on Drive: 190794
Val blocks on Drive:   10207
Sample input_ids length: 2048


## 7. Create Token Blocks

In [ ]:
from datasets import load_from_disk

# Load your pre-packed training and validation blocks from disk
train_blocks = load_from_disk("packed_pretrain_data/train")
val_blocks = load_from_disk("packed_pretrain_data/val")

print(f"✅ Loaded train_blocks: {len(train_blocks):,} sequences")
print(f"✅ Loaded val_blocks:   {len(val_blocks):,} sequences")

✅ Loaded train_blocks: 190,794 sequences
✅ Loaded val_blocks:   10,207 sequences





## 8. Create Datasets

In [ ]:
# train_blocks and val_blocks are already Hugging Face Datasets!
print(train_blocks)
print(val_blocks)

# You can inspect columns directly:
print(train_blocks[0])

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 190794
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 10207
})
{'input_ids': [1443, 484, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 5512, 2173, 1934, 2900, 1753, 13, 235, 1589, 492, 2899, 13, 3714, 4064, 241, 2073, 2183, 2368, 308, 19616, 1703, 10, 372, 15, 280, 15, 342, 15, 24340, 308, 38, 922, 655, 325, 1050, 1616, 521, 292, 235, 2266, 294, 2401, 295, 1672, 1633, 25828, 36, 15, 395, 15, 14177, 3, 1443, 484, 4385, 3520, 308, 9017, 19669, 241, 2632, 1318, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 440, 1365, 241, 235, 1301, 2226, 292, 1110, 308, 18, 10, 241, 2749, 14869, 43, 241, 235, 991, 13, 235, 2266, 2458, 3056, 13, 343, 803, 259, 235, 484, 241, 3520, 13, 235, 1050, 1616, 13, 1618, 3549, 180, 18, 15, 4477, 2312, 268, 2083, 15, 180, 9, 18, 10, 948, 1616, 414, 274, 1847, 235, 1008, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 161

## 9. Define The Model

In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. CORE BUILDING BLOCKS & GQA
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


def precompute_rope_params(head_dim: int, max_seq_len: int = 4096, theta: float = 10000.0):
    assert head_dim % 2 == 0, "Head dimension must be even for RoPE"
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cos = torch.cos(freqs).repeat_interleave(2, dim=-1)
    freqs_sin = torch.sin(freqs).repeat_interleave(2, dim=-1)
    return freqs_cos, freqs_sin


def apply_rotary_emb(x, cos, sin):
    # x: [B, heads, T, head_dim]; cos/sin: [T, head_dim]
    def rotate_half(tensor):
        x1 = tensor[..., 0::2]
        x2 = tensor[..., 1::2]
        return torch.stack([-x2, x1], dim=-1).flatten(-2)
    cos = cos.to(x.dtype).unsqueeze(0).unsqueeze(1)   # cast so bf16 stays bf16
    sin = sin.to(x.dtype).unsqueeze(0).unsqueeze(1)
    return (x * cos) + (rotate_half(x) * sin)


class SwiGLU(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        hidden_dim = int(2.5 * d_model)
        hidden_dim = ((hidden_dim + 255) // 256) * 256
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class GroupedQueryAttention(nn.Module):
    """GQA with RoPE. Attention is explicit matmul + softmax (no flash / no SDPA).
    Optional strided-causal pattern: each token sees the previous `window` tokens
    plus every `stride`-th token further back. window=None means dense causal."""
    def __init__(self, d_model: int, n_head: int, n_kv_head: int = 4, window=None, stride=None):
        super().__init__()
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert (window is None) == (stride is None), "set both window and stride, or neither"
        assert stride is None or (stride >= 1 and window >= 1), "window and stride must be >= 1"
        self.d_model = d_model
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = d_model // n_head
        self.window, self.stride = window, stride      # plain attributes, so no extra parameters

        self.q_proj = nn.Linear(d_model, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_head * self.head_dim, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # [B, H, T, hd]
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)   # [B, Hkv, T, hd]
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)   # [B, Hkv, T, hd]

        past_len = past_kv[0].shape[2] if past_kv is not None else 0
        total_len = past_len + T

        q = apply_rotary_emb(q, cos[past_len:total_len], sin[past_len:total_len])
        k = apply_rotary_emb(k, cos[past_len:total_len], sin[past_len:total_len])

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present_kv = (k, v)   # cache holds only the 4 KV heads, 3x smaller than full MHA

        # Group query heads under the KV head they share: [B, Hkv, n_rep, T, hd]
        q = q.reshape(B, self.n_kv_head, self.n_rep, T, self.head_dim)
        # K broadcasts across the n_rep dimension, so no expanded copy is made
        scores = torch.matmul(q, k.unsqueeze(2).transpose(-2, -1)) / math.sqrt(self.head_dim)

        # A single decoded token still needs the mask when the pattern is strided
        if T > 1 or self.window is not None:
            q_pos = torch.arange(past_len, total_len, device=x.device)[:, None]
            k_pos = torch.arange(total_len, device=x.device)[None, :]
            dist = q_pos - k_pos
            allowed = dist >= 0                                            # causal
            if self.window is not None:
                allowed = allowed & ((dist < self.window) | (dist % self.stride == 0))
            scores = scores.masked_fill(~allowed, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        y = torch.matmul(attn_weights, v.unsqueeze(2))                    # [B, Hkv, n_rep, T, hd]
        y = y.reshape(B, self.n_head, T, self.head_dim).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(y), present_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, n_kv_head: int = 4, window=None, stride=None):
        super().__init__()
        self.attention_norm = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, n_head, n_kv_head, window=window, stride=stride)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, cos, sin, past_kv=None):
        attn_out, present_kv = self.attn(self.attention_norm(x), cos, sin, past_kv=past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, present_kv


# ==========================================
# 2. MODEL WRAPPER & INITIALIZATION
# ==========================================
def alternating_patterns(n_layer, cycle):
    """Cycle through `cycle` across the layers. Each entry is None (dense) or a (window, stride) tuple.
    Example: [(256, 32), (256, 128)] gives stride 32 on even layers and stride 128 on odd layers."""
    return [cycle[i % len(cycle)] for i in range(n_layer)]


class CustomDecoderLM(nn.Module):
    """Decoder-Only Language Model from Scratch with GQA + KV Caching + per-layer strided attention."""
    def __init__(self, vocab_size=32000, d_model=768, n_layer=10, n_head=12,
                 n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=None):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len

        if layer_patterns is None:
            layer_patterns = [None] * n_layer                  # all dense = the original baseline
        assert len(layer_patterns) == n_layer, "layer_patterns must have one entry per layer"

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        blocks = []
        for p in layer_patterns:
            w, s = p if p is not None else (None, None)
            blocks.append(TransformerBlock(d_model, n_head, n_kv_head, window=w, stride=s))
        self.layers = nn.ModuleList(blocks)
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if tie_weights:
            self.token_embedding.weight = self.lm_head.weight

        # RoPE is applied to the full head dimension in GQA
        cos, sin = precompute_rope_params(d_model // n_head, max_seq_len)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self._init_weights()

    def set_patterns(self, layer_patterns):
        """Change the attention pattern of each layer without rebuilding the model."""
        assert len(layer_patterns) == len(self.layers)
        for layer, p in zip(self.layers, layer_patterns):
            w, s = p if p is not None else (None, None)
            assert (w is None) == (s is None)
            layer.attn.window, layer.attn.stride = w, s

    def describe_patterns(self):
        return ["dense" if l.attn.window is None else f"window={l.attn.window}, stride={l.attn.stride}"
                for l in self.layers]

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None, past_key_values=None):
        B, T = input_ids.shape
        x = self.token_embedding(input_ids)

        new_present_key_values = []
        for idx, layer in enumerate(self.layers):
            layer_past = past_key_values[idx] if past_key_values is not None else None
            x, present_kv = layer(x, self.cos, self.sin, past_kv=layer_past)
            new_present_key_values.append(present_kv)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, self.vocab_size), shift_labels.view(-1))

        return {
            "loss": loss,
            "logits": logits,
            "past_key_values": tuple(new_present_key_values)
        }


# ==========================================
# 3. DETERMINE VOCAB SIZE — explicit, no silent fallback
# ==========================================
assert 'tokenizer' in globals(), "Load your verified tokenizer (e.g. legal_bpe_32k_FINAL.json) before building the model."
vocab_sz = tokenizer.get_vocab_size()
print(f"Using tokenizer vocab size: {vocab_sz}")

target_device = DEVICE if 'DEVICE' in globals() else (device if 'device' in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# ==========================================
# 4. BUILD MODEL — ~87.5M parameters (patterns add no parameters)
# ==========================================
N_LAYER = 10

# Alternate two strided patterns: even layers stride 32, odd layers stride 128 (window 256 in both).
# Use None for a dense layer, e.g. [None, (256, 64)] alternates dense and strided layers.
# For the dense baseline use [None]; for a single pattern use [(256, 64)].
PATTERN_CYCLE = [(256, 32), (256, 128)]
LAYER_PATTERNS = alternating_patterns(N_LAYER, PATTERN_CYCLE)

model = CustomDecoderLM(
    vocab_size=vocab_sz,
    d_model=768,
    n_layer=N_LAYER,
    n_head=12,
    n_kv_head=4,
    max_seq_len=2048,
    tie_weights=True,
    layer_patterns=LAYER_PATTERNS
).to(target_device)

model_size = sum(p.numel() for p in model.parameters())
print(f"🚀 Custom GQA Decoder Model Size: {model_size / 1e6:.2f}M parameters")
for i, desc in enumerate(model.describe_patterns()):
    print(f"  layer {i}: {desc}")

# ==========================================
# 5. QUICK SHAPE + CACHE + PATTERN CHECK (tiny patterns, so the mask actually bites on 24 tokens)
# ==========================================
with torch.no_grad():
    _ids = torch.randint(0, vocab_sz, (2, 24), device=target_device)
    model.set_patterns(alternating_patterns(N_LAYER, [None]))                 # all dense
    _dense = model(_ids)["logits"]
    model.set_patterns(alternating_patterns(N_LAYER, [(4, 2), (4, 4)]))       # tiny alternating patterns
    _full = model(_ids)["logits"]
    _o1 = model(_ids[:, :12])
    _o2 = model(_ids[:, 12:], past_key_values=_o1["past_key_values"])
    _diff = (_full[:, 12:] - _o2["logits"]).abs().max().item()
    _eff = (_full - _dense).abs().max().item()
model.set_patterns(LAYER_PATTERNS)                                           # restore the real patterns
print(f"Cached vs uncached max logit diff: {_diff:.2e} (should be ~1e-5 or smaller)")
print(f"Strided vs dense max logit diff:   {_eff:.2e} (should be clearly above 0)")

Using tokenizer vocab size: 32000
🚀 Custom GQA Decoder Model Size: 87.51M parameters
  layer 0: window=256, stride=32
  layer 1: window=256, stride=128
  layer 2: window=256, stride=32
  layer 3: window=256, stride=128
  layer 4: window=256, stride=32
  layer 5: window=256, stride=128
  layer 6: window=256, stride=32
  layer 7: window=256, stride=128
  layer 8: window=256, stride=32
  layer 9: window=256, stride=128
Cached vs uncached max logit diff: 3.58e-06 (should be ~1e-5 or smaller)
Strided vs dense max logit diff:   1.80e+00 (should be clearly above 0)


## 10. Training Arguments

In [ ]:
from datasets import load_from_disk

# 1. Load the pre-packed datasets from disk
train_dataset = load_from_disk("packed_pretrain_data/train")
val_dataset = load_from_disk("packed_pretrain_data/val")

# 2. Set format to PyTorch tensors
train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

In [ ]:
from datasets import load_from_disk
import torch

DATA_DIR = "packed_pretrain_data"  # or your Drive path if this is a fresh session
train_dataset = load_from_disk(f"{DATA_DIR}/train")
val_dataset = load_from_disk(f"{DATA_DIR}/val")

train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

print(f"Train blocks: {len(train_dataset)} | Val blocks: {len(val_dataset)}")

Train blocks: 190794 | Val blocks: 10207


## 11. Training

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
import os
import math
import time
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

BLOCK_LEN = len(train_dataset[0]["input_ids"])
assert BLOCK_LEN <= model.max_seq_len, f"Block length {BLOCK_LEN} > model.max_seq_len {model.max_seq_len}"

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 8
    MAX_STEPS_PER_EPOCH = 40            # micro-batches per epoch
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 3
    COOLDOWN_STEPS = 3
    EVAL_EVERY = 5
    LOG_EVERY = 1
    RESUME = False
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 4
    BATCH_SIZE = 4                      # if OOM: BATCH_SIZE = 2 and GRADIENT_ACCUMULATION_STEPS = 8
    MAX_STEPS_PER_EPOCH = 20000         # micro-batches per epoch (None = whole dataset)
    GRADIENT_ACCUMULATION_STEPS = 4
    WARMUP_STEPS = 500                  # optimizer steps
    COOLDOWN_STEPS = 1000               # optimizer steps
    EVAL_EVERY = 500
    LOG_EVERY = 25
    RESUME = True

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 2. Checkpoint paths (checks Drive is really mounted)
# ============================================================
CKPT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive is not mounted — saving checkpoints locally (lost if the runtime disconnects).")
    CKPT_DIR = "gqa_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
suffix = "_debug" if DEBUG_MODE else ""
BEST_PATH = f"{CKPT_DIR}/gqa_best{suffix}.pt"
LAST_PATH = f"{CKPT_DIR}/gqa_last{suffix}.pt"

# ============================================================
# 3. Dataloaders
# ============================================================
def custom_collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          collate_fn=custom_collate_fn)
val_blocks = min(len(val_dataset), 400)
val_loader = DataLoader(Subset(val_dataset, range(val_blocks)), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=custom_collate_fn)

micro_per_epoch = min(len(train_loader), MAX_STEPS_PER_EPOCH) if MAX_STEPS_PER_EPOCH else len(train_loader)
steps_per_epoch = max(1, micro_per_epoch // GRADIENT_ACCUMULATION_STEPS)   # optimizer steps
total_steps = steps_per_epoch * EPOCHS
tokens_per_step = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * BLOCK_LEN

print(f"Train blocks: {len(train_dataset):,} | Val blocks used: {val_blocks}")
print(f"Micro-batches/epoch: {micro_per_epoch} | Optimizer steps/epoch: {steps_per_epoch} | Total: {total_steps}")
print(f"Tokens/optimizer step: {tokens_per_step:,} | Tokens seen in total: {total_steps * tokens_per_step / 1e6:.0f}M")

# ============================================================
# 4. Optimizer & schedule (step counts are in OPTIMIZER steps)
# ============================================================
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]    # norm weights
optimizer = AdamW(
    [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95)
)

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 5. Evaluation + checkpoint helpers
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

def save_ckpt(path, epoch):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "global_step": global_step, "epoch": epoch,
                "best_val": best_val, "val_loss_history": val_loss_history}, path)

global_step, best_val, start_epoch, val_loss_history = 0, float("inf"), 0, []
if RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    global_step, best_val, val_loss_history = ck["global_step"], ck["best_val"], ck["val_loss_history"]
    start_epoch = min(global_step // steps_per_epoch, EPOCHS)
    print(f"Resumed from optimizer step {global_step} (starting at epoch {start_epoch + 1})")

# ============================================================
# 6. Training loop
# ============================================================
model.train()
print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

t0, start_global = time.time(), global_step

for epoch in range(start_epoch, EPOCHS):
    if global_step >= total_steps:
        break
    progress_bar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = torch.tensor(0.0, device=device)
    epoch_micro = 0
    win_loss_sum = torch.tensor(0.0, device=device)
    win_micro = 0

    for step, batch in enumerate(train_loader):
        if step >= micro_per_epoch or global_step >= total_steps:
            break
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        micro_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS    # stays on GPU, no sync
        epoch_loss_sum += micro_loss; epoch_micro += 1
        win_loss_sum += micro_loss;   win_micro += 1

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            progress_bar.update(1)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                done = global_step - start_global
                tok_s = done * tokens_per_step / elapsed
                eta_h = (total_steps - global_step) * elapsed / max(1, done) / 3600
                mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
                progress_bar.set_postfix({
                    "loss": f"{(win_loss_sum / win_micro).item():.4f}",
                    "val": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                    "lr": f"{scheduler.get_last_lr()[0]:.6f}",
                    "tok/s": f"{tok_s:,.0f}",
                    "eta_h": f"{eta_h:.2f}",
                    "mem_GB": f"{mem:.1f}",
                })
                win_loss_sum = torch.tensor(0.0, device=device); win_micro = 0

            if global_step % EVAL_EVERY == 0:
                v = evaluate()
                val_loss_history.append(v)
                print(f"\nstep {global_step}: val_loss={v:.4f} (ppl {math.exp(v):.1f})")
                if v < best_val:
                    best_val = v
                    save_ckpt(BEST_PATH, epoch)
                save_ckpt(LAST_PATH, epoch)

    progress_bar.close()
    avg_train_loss = (epoch_loss_sum / max(1, epoch_micro)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    if avg_val_loss < best_val:
        best_val = avg_val_loss
        save_ckpt(BEST_PATH, epoch + 1)
    save_ckpt(LAST_PATH, epoch + 1)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"(ppl {math.exp(avg_val_loss):.1f})\n")

print("✅ Training complete!")
print(f"Best val loss: {best_val:.4f} | Best checkpoint: {BEST_PATH}")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

# Reload the best weights before generating / evaluating:
# model.load_state_dict(torch.load(BEST_PATH, map_location=device)["model"])

Train blocks: 190,794 | Val blocks used: 400
Micro-batches/epoch: 20000 | Optimizer steps/epoch: 5000 | Total: 20000
Tokens/optimizer step: 32,768 | Tokens seen in total: 655M
Using bf16: True

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...


Epoch 1/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 500: val_loss=4.3661 (ppl 78.7)

step 1000: val_loss=3.6533 (ppl 38.6)

step 1500: val_loss=3.2694 (ppl 26.3)

step 2000: val_loss=3.0202 (ppl 20.5)

step 2500: val_loss=2.8833 (ppl 17.9)

step 3000: val_loss=2.7986 (ppl 16.4)

step 3500: val_loss=2.7346 (ppl 15.4)

step 4000: val_loss=2.6811 (ppl 14.6)

step 4500: val_loss=2.6459 (ppl 14.1)

step 5000: val_loss=2.6095 (ppl 13.6)

📊 Epoch 1 -> Train Loss: 3.2116 | Val Loss: 2.6095 (ppl 13.6)



Epoch 2/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 5500: val_loss=2.5804 (ppl 13.2)

step 6000: val_loss=2.5518 (ppl 12.8)

step 6500: val_loss=2.5379 (ppl 12.7)

step 7000: val_loss=2.5225 (ppl 12.5)

step 7500: val_loss=2.4982 (ppl 12.2)

step 8000: val_loss=2.4830 (ppl 12.0)

step 8500: val_loss=2.4624 (ppl 11.7)

step 9000: val_loss=2.4536 (ppl 11.6)

step 9500: val_loss=2.4439 (ppl 11.5)

step 10000: val_loss=2.4282 (ppl 11.3)

📊 Epoch 2 -> Train Loss: 2.4509 | Val Loss: 2.4282 (ppl 11.3)



Epoch 3/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 10500: val_loss=2.4097 (ppl 11.1)

step 11000: val_loss=2.4036 (ppl 11.1)

step 11500: val_loss=2.4004 (ppl 11.0)

step 12000: val_loss=2.3927 (ppl 10.9)

step 12500: val_loss=2.3858 (ppl 10.9)

step 13000: val_loss=2.3775 (ppl 10.8)

step 13500: val_loss=2.3600 (ppl 10.6)

step 14000: val_loss=2.3578 (ppl 10.6)

step 14500: val_loss=2.3539 (ppl 10.5)

step 15000: val_loss=2.3473 (ppl 10.5)

📊 Epoch 3 -> Train Loss: 2.3194 | Val Loss: 2.3473 (ppl 10.5)



Epoch 4/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 15500: val_loss=2.3374 (ppl 10.4)

step 16000: val_loss=2.3448 (ppl 10.4)

step 16500: val_loss=2.3346 (ppl 10.3)

step 17000: val_loss=2.3303 (ppl 10.3)

step 17500: val_loss=2.3279 (ppl 10.3)

step 18000: val_loss=2.3208 (ppl 10.2)

step 18500: val_loss=2.3110 (ppl 10.1)

step 19000: val_loss=2.3105 (ppl 10.1)

step 19500: val_loss=2.2450 (ppl 9.4)

step 20000: val_loss=2.1949 (ppl 9.0)

📊 Epoch 4 -> Train Loss: 2.2324 | Val Loss: 2.1949 (ppl 9.0)

✅ Training complete!
Best val loss: 2.1949 | Best checkpoint: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt


In [ ]:
import os
import math
import time
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

BLOCK_LEN = len(train_dataset[0]["input_ids"])
assert BLOCK_LEN <= model.max_seq_len, f"Block length {BLOCK_LEN} > model.max_seq_len {model.max_seq_len}"

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 8
    MAX_STEPS_PER_EPOCH = 40            # micro-batches per epoch
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 3
    COOLDOWN_STEPS = 3
    EVAL_EVERY = 5
    LOG_EVERY = 1
    RESUME = False
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 4
    BATCH_SIZE = 4                      # if OOM: BATCH_SIZE = 2 and GRADIENT_ACCUMULATION_STEPS = 8
    MAX_STEPS_PER_EPOCH = 20000         # micro-batches per epoch (None = whole dataset)
    GRADIENT_ACCUMULATION_STEPS = 4
    WARMUP_STEPS = 500                  # optimizer steps
    COOLDOWN_STEPS = 1000               # optimizer steps
    EVAL_EVERY = 500
    LOG_EVERY = 25
    RESUME = True

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 2. Checkpoint paths (checks Drive is really mounted)
# ============================================================
CKPT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive is not mounted — saving checkpoints locally (lost if the runtime disconnects).")
    CKPT_DIR = "gqa_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
suffix = "_debug" if DEBUG_MODE else ""
BEST_PATH = f"{CKPT_DIR}/gqa_best{suffix}.pt"
LAST_PATH = f"{CKPT_DIR}/gqa_last{suffix}.pt"

# ============================================================
# 3. Dataloaders
# ============================================================
def custom_collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          collate_fn=custom_collate_fn)
val_blocks = min(len(val_dataset), 400)
val_loader = DataLoader(Subset(val_dataset, range(val_blocks)), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=custom_collate_fn)

micro_per_epoch = min(len(train_loader), MAX_STEPS_PER_EPOCH) if MAX_STEPS_PER_EPOCH else len(train_loader)
steps_per_epoch = max(1, micro_per_epoch // GRADIENT_ACCUMULATION_STEPS)   # optimizer steps
total_steps = steps_per_epoch * EPOCHS
tokens_per_step = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * BLOCK_LEN

print(f"Train blocks: {len(train_dataset):,} | Val blocks used: {val_blocks}")
print(f"Micro-batches/epoch: {micro_per_epoch} | Optimizer steps/epoch: {steps_per_epoch} | Total: {total_steps}")
print(f"Tokens/optimizer step: {tokens_per_step:,} | Tokens seen in total: {total_steps * tokens_per_step / 1e6:.0f}M")

# ============================================================
# 4. Optimizer & schedule (step counts are in OPTIMIZER steps)
# ============================================================
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]    # norm weights
optimizer = AdamW(
    [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95)
)

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 5. Evaluation + checkpoint helpers
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

def save_ckpt(path, epoch):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "global_step": global_step, "epoch": epoch,
                "best_val": best_val, "val_loss_history": val_loss_history}, path)

global_step, best_val, start_epoch, val_loss_history = 0, float("inf"), 0, []
if RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    global_step, best_val, val_loss_history = ck["global_step"], ck["best_val"], ck["val_loss_history"]
    start_epoch = min(global_step // steps_per_epoch, EPOCHS)
    print(f"Resumed from optimizer step {global_step} (starting at epoch {start_epoch + 1})")

# ============================================================
# 6. Training loop
# ============================================================
model.train()
print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

t0, start_global = time.time(), global_step

for epoch in range(start_epoch, EPOCHS):
    if global_step >= total_steps:
        break
    progress_bar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = torch.tensor(0.0, device=device)
    epoch_micro = 0
    win_loss_sum = torch.tensor(0.0, device=device)
    win_micro = 0

    for step, batch in enumerate(train_loader):
        if step >= micro_per_epoch or global_step >= total_steps:
            break
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        micro_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS    # stays on GPU, no sync
        epoch_loss_sum += micro_loss; epoch_micro += 1
        win_loss_sum += micro_loss;   win_micro += 1

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            progress_bar.update(1)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                done = global_step - start_global
                tok_s = done * tokens_per_step / elapsed
                eta_h = (total_steps - global_step) * elapsed / max(1, done) / 3600
                mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
                progress_bar.set_postfix({
                    "loss": f"{(win_loss_sum / win_micro).item():.4f}",
                    "val": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                    "lr": f"{scheduler.get_last_lr()[0]:.6f}",
                    "tok/s": f"{tok_s:,.0f}",
                    "eta_h": f"{eta_h:.2f}",
                    "mem_GB": f"{mem:.1f}",
                })
                win_loss_sum = torch.tensor(0.0, device=device); win_micro = 0

            if global_step % EVAL_EVERY == 0:
                v = evaluate()
                val_loss_history.append(v)
                print(f"\nstep {global_step}: val_loss={v:.4f} (ppl {math.exp(v):.1f})")
                if v < best_val:
                    best_val = v
                    save_ckpt(BEST_PATH, epoch)
                save_ckpt(LAST_PATH, epoch)

    progress_bar.close()
    avg_train_loss = (epoch_loss_sum / max(1, epoch_micro)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    if avg_val_loss < best_val:
        best_val = avg_val_loss
        save_ckpt(BEST_PATH, epoch + 1)
    save_ckpt(LAST_PATH, epoch + 1)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"(ppl {math.exp(avg_val_loss):.1f})\n")

print("✅ Training complete!")
print(f"Best val loss: {best_val:.4f} | Best checkpoint: {BEST_PATH}")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

# Reload the best weights before generating / evaluating:
# model.load_state_dict(torch.load(BEST_PATH, map_location=device)["model"])

Train blocks: 190,794 | Val blocks used: 400
Micro-batches/epoch: 20000 | Optimizer steps/epoch: 5000 | Total: 20000
Tokens/optimizer step: 32,768 | Tokens seen in total: 655M
Using bf16: True
Resumed from optimizer step 20000 (starting at epoch 5)

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...
✅ Training complete!
Best val loss: 2.1949 | Best checkpoint: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt


In [ ]:
import os
import math
import time
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# CHANGED: name this experiment. It decides the checkpoint folder, so runs can't collide.
EXPERIMENT_NAME = "alt_stride_32_128"

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."
assert hasattr(model, "describe_patterns"), \
    "The model in memory is the old class without strided attention. Re-run the updated model cell."   # CHANGED

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

BLOCK_LEN = len(train_dataset[0]["input_ids"])
assert BLOCK_LEN <= model.max_seq_len, f"Block length {BLOCK_LEN} > model.max_seq_len {model.max_seq_len}"

print(f"Experiment: {EXPERIMENT_NAME}")                                     # CHANGED
for i, desc in enumerate(model.describe_patterns()):                        # CHANGED
    print(f"  layer {i}: {desc}")

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False    # CHANGED: the strided code is new, so smoke-test first

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 8
    MAX_STEPS_PER_EPOCH = 40            # micro-batches per epoch
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 3
    COOLDOWN_STEPS = 3
    EVAL_EVERY = 5
    LOG_EVERY = 1
    RESUME = False
    print("\n*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 3
    BATCH_SIZE = 4                      # if OOM: BATCH_SIZE = 2 and GRADIENT_ACCUMULATION_STEPS = 8
    MAX_STEPS_PER_EPOCH = 15000         # micro-batches per epoch (None = whole dataset)
    GRADIENT_ACCUMULATION_STEPS = 4
    WARMUP_STEPS = 500                  # optimizer steps
    COOLDOWN_STEPS = 1000               # optimizer steps
    EVAL_EVERY = 500
    LOG_EVERY = 25
    RESUME = True

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 2. Checkpoint paths (checks Drive is really mounted)
# ============================================================
CKPT_DIR = f"/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/{EXPERIMENT_NAME}_checkpoints"   # CHANGED
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive is not mounted — saving checkpoints locally (lost if the runtime disconnects).")
    CKPT_DIR = f"{EXPERIMENT_NAME}_checkpoints"                                                      # CHANGED
os.makedirs(CKPT_DIR, exist_ok=True)
suffix = "_debug" if DEBUG_MODE else ""
BEST_PATH = f"{CKPT_DIR}/best{suffix}.pt"                                                            # CHANGED
LAST_PATH = f"{CKPT_DIR}/last{suffix}.pt"                                                            # CHANGED

# ============================================================
# 3. Dataloaders
# ============================================================
def custom_collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          collate_fn=custom_collate_fn)
val_blocks = min(len(val_dataset), 400)
val_loader = DataLoader(Subset(val_dataset, range(val_blocks)), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=custom_collate_fn)

micro_per_epoch = min(len(train_loader), MAX_STEPS_PER_EPOCH) if MAX_STEPS_PER_EPOCH else len(train_loader)
steps_per_epoch = max(1, micro_per_epoch // GRADIENT_ACCUMULATION_STEPS)   # optimizer steps
total_steps = steps_per_epoch * EPOCHS
tokens_per_step = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * BLOCK_LEN

print(f"Train blocks: {len(train_dataset):,} | Val blocks used: {val_blocks}")
print(f"Micro-batches/epoch: {micro_per_epoch} | Optimizer steps/epoch: {steps_per_epoch} | Total: {total_steps}")
print(f"Tokens/optimizer step: {tokens_per_step:,} | Tokens seen in total: {total_steps * tokens_per_step / 1e6:.0f}M")

# ============================================================
# 4. Optimizer & schedule (step counts are in OPTIMIZER steps)
# ============================================================
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]    # norm weights
optimizer = AdamW(
    [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95)
)

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 5. Evaluation + checkpoint helpers
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

def save_ckpt(path, epoch):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "global_step": global_step, "epoch": epoch,
                "best_val": best_val, "val_loss_history": val_loss_history,
                "layer_patterns": model.describe_patterns(),          # CHANGED: patterns aren't in the state_dict
                "experiment": EXPERIMENT_NAME}, path)                 # CHANGED

global_step, best_val, start_epoch, val_loss_history = 0, float("inf"), 0, []
if RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    saved_patterns = ck.get("layer_patterns", ["dense"] * len(model.layers))   # CHANGED: dense-baseline checkpoints have no key
    assert saved_patterns == model.describe_patterns(), (                       # CHANGED
        f"Checkpoint was trained with different attention patterns:\n  checkpoint: {saved_patterns}\n"
        f"  model now:  {model.describe_patterns()}\nUse a different EXPERIMENT_NAME or rebuild the model to match.")
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    global_step, best_val, val_loss_history = ck["global_step"], ck["best_val"], ck["val_loss_history"]
    start_epoch = min(global_step // steps_per_epoch, EPOCHS)
    print(f"Resumed from optimizer step {global_step} (starting at epoch {start_epoch + 1})")

# ============================================================
# 6. Training loop
# ============================================================
model.train()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()        # CHANGED: mem_GB now reflects this run only
print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

t0, start_global = time.time(), global_step

for epoch in range(start_epoch, EPOCHS):
    if global_step >= total_steps:
        break
    progress_bar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = torch.tensor(0.0, device=device)
    epoch_micro = 0
    win_loss_sum = torch.tensor(0.0, device=device)
    win_micro = 0

    for step, batch in enumerate(train_loader):
        if step >= micro_per_epoch or global_step >= total_steps:
            break
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        micro_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS    # stays on GPU, no sync
        epoch_loss_sum += micro_loss; epoch_micro += 1
        win_loss_sum += micro_loss;   win_micro += 1

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            progress_bar.update(1)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                done = global_step - start_global
                tok_s = done * tokens_per_step / elapsed
                eta_h = (total_steps - global_step) * elapsed / max(1, done) / 3600
                mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
                progress_bar.set_postfix({
                    "loss": f"{(win_loss_sum / win_micro).item():.4f}",
                    "val": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                    "lr": f"{scheduler.get_last_lr()[0]:.6f}",
                    "tok/s": f"{tok_s:,.0f}",
                    "eta_h": f"{eta_h:.2f}",
                    "mem_GB": f"{mem:.1f}",
                })
                win_loss_sum = torch.tensor(0.0, device=device); win_micro = 0

            if global_step % EVAL_EVERY == 0:
                v = evaluate()
                val_loss_history.append(v)
                print(f"\nstep {global_step}: val_loss={v:.4f} (ppl {math.exp(v):.1f})")
                if v < best_val:
                    best_val = v
                    save_ckpt(BEST_PATH, epoch)
                save_ckpt(LAST_PATH, epoch)

    progress_bar.close()
    avg_train_loss = (epoch_loss_sum / max(1, epoch_micro)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    if avg_val_loss < best_val:
        best_val = avg_val_loss
        save_ckpt(BEST_PATH, epoch + 1)
    save_ckpt(LAST_PATH, epoch + 1)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"(ppl {math.exp(avg_val_loss):.1f})\n")

print("✅ Training complete!")
print(f"Experiment: {EXPERIMENT_NAME} | Best val loss: {best_val:.4f} | Best checkpoint: {BEST_PATH}")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

# Reload the best weights before generating / evaluating:
# model.load_state_dict(torch.load(BEST_PATH, map_location=device)["model"])

Experiment: alt_stride_32_128
  layer 0: window=256, stride=32
  layer 1: window=256, stride=128
  layer 2: window=256, stride=32
  layer 3: window=256, stride=128
  layer 4: window=256, stride=32
  layer 5: window=256, stride=128
  layer 6: window=256, stride=32
  layer 7: window=256, stride=128
  layer 8: window=256, stride=32
  layer 9: window=256, stride=128
Train blocks: 190,794 | Val blocks used: 400
Micro-batches/epoch: 15000 | Optimizer steps/epoch: 3750 | Total: 11250
Tokens/optimizer step: 32,768 | Tokens seen in total: 369M
Using bf16: True

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...


Epoch 1/3:   0%|          | 0/3750 [00:00<?, ?it/s]


step 500: val_loss=4.3386 (ppl 76.6)

step 1000: val_loss=3.6140 (ppl 37.1)

step 1500: val_loss=3.2620 (ppl 26.1)

step 2000: val_loss=3.0624 (ppl 21.4)

step 2500: val_loss=2.9489 (ppl 19.1)

step 3000: val_loss=2.8499 (ppl 17.3)

step 3500: val_loss=2.7911 (ppl 16.3)

📊 Epoch 1 -> Train Loss: 3.4293 | Val Loss: 2.7661 (ppl 15.9)



Epoch 2/3:   0%|          | 0/3750 [00:00<?, ?it/s]


step 4000: val_loss=2.7422 (ppl 15.5)

step 4500: val_loss=2.7212 (ppl 15.2)

step 5000: val_loss=2.6808 (ppl 14.6)

step 5500: val_loss=2.6419 (ppl 14.0)

step 6000: val_loss=2.6205 (ppl 13.7)

step 6500: val_loss=2.6056 (ppl 13.5)

step 7000: val_loss=2.5804 (ppl 13.2)

step 7500: val_loss=2.5606 (ppl 12.9)

📊 Epoch 2 -> Train Loss: 2.6012 | Val Loss: 2.5606 (ppl 12.9)



Epoch 3/3:   0%|          | 0/3750 [00:00<?, ?it/s]


step 8000: val_loss=2.5480 (ppl 12.8)

step 8500: val_loss=2.5378 (ppl 12.7)

step 9000: val_loss=2.5151 (ppl 12.4)

step 9500: val_loss=2.5134 (ppl 12.3)

step 10000: val_loss=2.5016 (ppl 12.2)

step 10500: val_loss=2.4637 (ppl 11.7)

step 11000: val_loss=2.3849 (ppl 10.9)

📊 Epoch 3 -> Train Loss: 2.4291 | Val Loss: 2.3650 (ppl 10.6)

✅ Training complete!
Experiment: alt_stride_32_128 | Best val loss: 2.3650 | Best checkpoint: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt


In [ ]:
import torch
ck = torch.load(BEST_PATH, map_location="cpu")
out = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt"
torch.save(ck["model"], out)
print(f"Saved weights-only model to {out} (best val loss {ck['best_val']:.4f})")

# To load later, after building the model with the GQA model cell:
# model.load_state_dict(torch.load(out, map_location=device))

Saved weights-only model to /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt (best val loss 2.1949)


In [ ]:
import os, torch

WEIGHTS_PATH = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt"
assert os.path.exists(WEIGHTS_PATH), f"Weights file not found: {WEIGHTS_PATH}"
for name in ["CustomDecoderLM", "tokenizer", "device"]:
    assert name in globals(), f"'{name}' is not defined, so run the loader cell and the GQA model cell first."

# Rebuild the exact architecture the weights were trained with
model = CustomDecoderLM(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=768, n_layer=10, n_head=12, n_kv_head=4,
    max_seq_len=2048, tie_weights=True
).to(device)

state = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(state, strict=True)    # strict=True fails loudly on any shape or key mismatch
model.eval()
print(f"Loaded {WEIGHTS_PATH} | {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

# Quick sanity check: a prompt should continue with plausible legal text
ids = tokenizer.encode("The Supreme Court held that").ids
x = torch.tensor([ids], device=device)
with torch.no_grad():
    for _ in range(20):
        nxt = model(x)["logits"][:, -1].argmax(-1, keepdim=True)
        x = torch.cat([x, nxt], dim=1)
print(tokenizer.decode(x[0].tolist()))

Loaded /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt | 87.51M parameters
The Supreme Court held that the High Court was not right in holding that the order of the Tribunal was not a nullity. The


## 12. Perplexity

In [ ]:
import math

# After your training loop finishes:
final_val_loss = val_loss_history[-1] # Grabs the last epoch's validation loss
perplexity = math.exp(final_val_loss)

print(f"📊 Final Validation Loss: {final_val_loss:.6f}")
print(f"📈 Final Validation Perplexity: {perplexity:.4f}")

📊 Final Validation Loss: 2.364988
📈 Final Validation Perplexity: 10.6439


## 13. Plot Training and Validation Loss

## 16. Load Test Data

In [ ]:
from huggingface_hub import hf_hub_download

# Make sure filename points to the correct nested directory path
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file downloaded to:", test_file_path)

Test file downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


## 18. Load Final Model for Generation

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
import os, math, torch

CKPT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints"
for f in sorted(os.listdir(CKPT_DIR)):
    print(f"{f:28s} {os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6:7.0f} MB")

ck = torch.load(f"{CKPT_DIR}/gqa_best.pt", map_location="cpu")
print(f"\nBest val loss: {ck['best_val']:.4f} (perplexity {math.exp(ck['best_val']):.1f})")
print(f"Optimizer steps completed: {ck['global_step']}")
print("Val loss history:", [round(v, 3) for v in ck["val_loss_history"]])

In [ ]:
import os, json, time
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ============================================================
# 0. Settings
# ============================================================
EXPERIMENT_NAME = "alt_stride_32_128"     # CHANGED: must match the name used in the training cell
LIMIT = None        # 10 = timed test on 10 prompts. Set to None for the full 1000 run.
LOAD_WEIGHTS = True
OUT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
CKPT_PATH = f"{OUT_DIR}/{EXPERIMENT_NAME}_checkpoints/best.pt"       # CHANGED
PRESET = "greedy"                         # CHANGED: same decoding for every experiment, so comparisons are fair
SEED = 42
USE_BF16 = True
TARGET_WORDS = 128

PRESETS = {
    "greedy":       dict(do_sample=False, temperature=1.0, top_k=0,  top_p=1.0, repetition_penalty=1.0, no_repeat_ngram=4),
    "sample_top_p": dict(do_sample=True,  temperature=0.7, top_k=0,  top_p=0.9, repetition_penalty=1.1, no_repeat_ngram=0),
    "sample_top_k": dict(do_sample=True,  temperature=0.7, top_k=40, top_p=1.0, repetition_penalty=1.1, no_repeat_ngram=0),
}
cfg = PRESETS[PRESET]                     # CHANGED: was hard-coded to greedy while PRESET said something else

# ============================================================
# 1. Guards + load trained weights
# ============================================================
for name in ["tokenizer", "model", "device"]:
    assert name in globals(), f"'{name}' is not defined, so run the setup cells first."
assert hasattr(model, "describe_patterns"), \
    "The model in memory is the old class without strided attention. Re-run the updated model cell."   # CHANGED
assert tokenizer.decode(tokenizer.encode("the court held that").ids).strip() != "", "Tokenizer sanity check failed."

print(f"Experiment: {EXPERIMENT_NAME}")                                                               # CHANGED
for i, desc in enumerate(model.describe_patterns()):
    print(f"  layer {i}: {desc}")

if LOAD_WEIGHTS:
    assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
    ck = torch.load(CKPT_PATH, map_location=device)
    is_full = isinstance(ck, dict) and "model" in ck
    if is_full:                                                                                       # CHANGED
        saved = ck.get("layer_patterns", ["dense"] * len(model.layers))   # dense-baseline checkpoints have no key
        assert saved == model.describe_patterns(), (
            f"Attention patterns differ.\n  checkpoint: {saved}\n  model now:  {model.describe_patterns()}\n"
            "Rebuild the model with the checkpoint's patterns, or fix CKPT_PATH.")
    else:
        print("⚠️ Weights-only file: patterns can't be verified. Make sure the model was built with the same patterns.")
    model.load_state_dict(ck["model"] if is_full else ck)
    if is_full and "best_val" in ck:
        print(f"Loaded {CKPT_PATH} (best val loss {ck['best_val']:.4f})")
model.eval()

# ============================================================
# 2. Test data (reuses test_data if it's already loaded)
# ============================================================
if "test_data" not in globals():
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename="Phase_1_Pretraining/Test/test.json",
                        repo_type="dataset", token=HF_TOKEN)
    with open(p, "r", encoding="utf-8") as f:
        test_data = json.load(f)
print("Number of test examples:", len(test_data))

# ============================================================
# 3. Generation (KV cache, top-k / top-p, repetition control)
# ============================================================
eos_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("[PAD]")
assert eos_id is not None, "<|endoftext|> missing from tokenizer"
BAN_IDS = [i for i in (eos_id, pad_id) if i is not None]

MAX_NEW_TOKENS = int(TARGET_WORDS * 2.2)
MAX_PROMPT_TOKENS = model.max_seq_len - MAX_NEW_TOKENS     # prompt + generation must fit the RoPE table

def pick_token(logits, generated, blocked, cfg):
    logits = logits.float().clone()
    logits[BAN_IDS] = float("-inf")                         # never stop early
    rp = cfg["repetition_penalty"]
    if rp != 1.0 and generated:                             # penalise only tokens generated so far
        idx = torch.tensor(sorted(set(generated)), device=logits.device)
        sel = logits[idx]
        logits[idx] = torch.where(sel > 0, sel / rp, sel * rp)
    if blocked:
        logits[list(blocked)] = float("-inf")
    if not cfg["do_sample"]:
        return logits.argmax().view(1)
    logits = logits / cfg["temperature"]
    if cfg["top_k"] > 0:
        kth = torch.topk(logits, min(cfg["top_k"], logits.numel())).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if cfg["top_p"] < 1.0:
        sl, si = torch.sort(logits, descending=True)
        pr = F.softmax(sl, dim=-1)
        sl = sl.masked_fill((pr.cumsum(-1) - pr) > cfg["top_p"], float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(0, si, sl)
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

@torch.no_grad()
def generate_continuation(prompt_text, cfg):
    ids = tokenizer.encode(prompt_text).ids[-MAX_PROMPT_TOKENS:]       # keep the END of long prompts
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    n = cfg["no_repeat_ngram"]
    ngram_seen, generated = {}, []
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(USE_BF16 and device.type == "cuda")):
        out = model(input_ids, past_key_values=None)                    # prime the KV cache with the prompt
        for step in range(MAX_NEW_TOKENS):
            logits = out["logits"][0, -1]
            blocked = ngram_seen.get(tuple(generated[-(n - 1):])) if (n > 1 and len(generated) >= n - 1) else None
            nxt = pick_token(logits, generated, blocked, cfg)
            tok = nxt.item()
            generated.append(tok)
            if n > 1 and len(generated) >= n:
                ngram_seen.setdefault(tuple(generated[-n:-1]), set()).add(tok)
            if len(generated) % 10 == 0 and len(tokenizer.decode(generated).split()) >= TARGET_WORDS:
                break
            if step == MAX_NEW_TOKENS - 1:
                break
            out = model(nxt.view(1, 1), past_key_values=out["past_key_values"])
    return " ".join(tokenizer.decode(generated).split()[:TARGET_WORDS])

# ============================================================
# 4. Run (with resume for the full run)
# ============================================================
torch.manual_seed(SEED)
items = list(test_data.items())
if LIMIT:
    items = items[:LIMIT]

drive_ok = os.path.isdir(OUT_DIR)
PARTIAL_PATH = os.path.join(OUT_DIR, f"prediction_partial_{EXPERIMENT_NAME}.json")                 # CHANGED
sig = {"experiment": EXPERIMENT_NAME, "preset": PRESET, "cfg": cfg,                                # CHANGED
       "patterns": model.describe_patterns(),
       "ckpt_mtime": os.path.getmtime(CKPT_PATH) if os.path.exists(CKPT_PATH) else None}

predictions = {}
if LIMIT is None and drive_ok and os.path.exists(PARTIAL_PATH):
    with open(PARTIAL_PATH, "r", encoding="utf-8") as f:
        saved_run = json.load(f)
    if saved_run.get("sig") == sig:                 # only resume if experiment, preset, patterns AND checkpoint are unchanged
        predictions = saved_run["preds"]
        print(f"Resuming: {len(predictions)} predictions already done.")

t0, n_done_now = time.time(), 0
for example_id, record in tqdm(items, desc="Generating predictions"):
    key = str(example_id)
    if key in predictions:
        continue
    prompt_text = record["text"] if isinstance(record, dict) else record
    try:
        text = generate_continuation(prompt_text, cfg)
    except Exception as e:
        print(f"⚠️ Generation failed for ID {key}: {e}")
        text = ""
    predictions[key] = {"text": text}
    n_done_now += 1
    if LIMIT is None and drive_ok and n_done_now % 100 == 0:
        with open(PARTIAL_PATH, "w", encoding="utf-8") as f:
            json.dump({"sig": sig, "preds": predictions}, f, ensure_ascii=False)

elapsed = time.time() - t0
if n_done_now:
    print(f"\n{elapsed:.0f}s for {n_done_now} examples ({elapsed / n_done_now:.2f}s each)")
    if LIMIT:
        print(f"Estimated full run for {len(test_data)} prompts: {elapsed / n_done_now * len(test_data) / 60:.1f} minutes")

# ============================================================
# 5. Checks before saving
# ============================================================
texts = [predictions[str(k)]["text"] for k, _ in items]
wc = [len(t.split()) for t in texts]
uniq = [len(set(t.split())) / max(1, len(t.split())) for t in texts]
print(f"Empty predictions:     {sum(1 for t in texts if not t.strip())} / {len(texts)}")
print(f"Under {TARGET_WORDS} words:        {sum(1 for c in wc if c < TARGET_WORDS)} / {len(texts)}")
print(f"Avg unique-word ratio: {sum(uniq) / len(uniq):.2f} (heuristic; very low values suggest repetition loops)")
print(f"\n--- Sample prediction (ID {items[0][0]}) ---\n{texts[0][:500]}")
if LIMIT is None:
    assert set(predictions) == set(str(k) for k in test_data), "Some test IDs are missing from predictions."

# ============================================================
# 6. Save (never overwrites your baseline prediction.json)
# ============================================================
fname = f"prediction_{EXPERIMENT_NAME}.json" if LIMIT is None else f"prediction_test_subset_{EXPERIMENT_NAME}.json"   # CHANGED
with open(fname, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print(f"\nSaved {len(predictions)} predictions to '{fname}'")
if drive_ok:
    import shutil
    shutil.copy(fname, os.path.join(OUT_DIR, fname))
    print(f"Copied to Drive: {os.path.join(OUT_DIR, fname)}")
# To submit one of these, copy it to prediction.json first.
# To download to your laptop:
# from google.colab import files; files.download(fname)

Experiment: alt_stride_32_128
  layer 0: window=256, stride=32
  layer 1: window=256, stride=128
  layer 2: window=256, stride=32
  layer 3: window=256, stride=128
  layer 4: window=256, stride=32
  layer 5: window=256, stride=128
  layer 6: window=256, stride=32
  layer 7: window=256, stride=128
  layer 8: window=256, stride=32
  layer 9: window=256, stride=128
Loaded /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt (best val loss 2.3650)
Number of test examples: 1000


Generating predictions:   0%|          | 0/1000 [00:00<?, ?it/s]


2754s for 1000 examples (2.75s each)
Empty predictions:     0 / 1000
Under 128 words:        4 / 1000
Avg unique-word ratio: 0.45 (heuristic; very low values suggest repetition loops)

--- Sample prediction (ID 1) ---
a matter of discretion and discretion of the court to be exercised judiciously and not arbitrarily. The court should not be swayed by irrelevant considerations and should not be guided by irrelevant considerations. The court must not only be careful in its duty to consider the relevant facts and circumstances of the case but must also bear in mind that the delay in the acquisition proceedings is not fatal to the acquisition proceedings. The court has to be satisfied that the delay was not proper

Saved 1000 predictions to 'prediction_alt_stride_32_128.json'
Copied to Drive: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/prediction_alt_stride_32_128.json


In [ ]:
needed = ["model", "tokenizer", "val_dataset", "device", "generate_continuation", "PRESETS", "TARGET_WORDS", "eos_id"]
missing = [n for n in needed if n not in globals()]
print("Missing:", missing if missing else "none")

if not missing:
    # Same construction as the comparison cell, and it's deterministic, so the baseline stays comparable
    N_EVAL, PROMPT_WORDS = 60, 512
    stride = max(1, len(val_dataset) // (N_EVAL * 2))
    pairs = []
    for i in range(0, len(val_dataset), stride):
        seg = []
        for t in val_dataset[i]["input_ids"].tolist() + [eos_id]:
            if t == eos_id:
                words = tokenizer.decode(seg).split()
                if len(words) >= PROMPT_WORDS + TARGET_WORDS:
                    pairs.append((" ".join(words[:PROMPT_WORDS]),
                                  " ".join(words[PROMPT_WORDS:PROMPT_WORDS + TARGET_WORDS])))
                    break
                seg = []
            else:
                seg.append(t)
        if len(pairs) >= N_EVAL:
            break
    refs = [r for _, r in pairs]
    print(f"Rebuilt {len(pairs)} pairs")

Missing: none
Rebuilt 60 pairs


In [ ]:
print(model.vocab_size == tokenizer.get_vocab_size())  # 32000 == 32000, True — but doesn't prove it's the SAME tokenizer

# This does prove it — decode real IDs and check for sensible output:
test_enc = tokenizer.encode("the Supreme Court held that the appellant must comply with the statute")
print(test_enc.ids)
print(tokenizer.decode(test_enc.ids))

True
[1380, 2986, 384, 790, 298, 235, 535, 993, 3817, 343, 235, 2675]
the Supreme Court held that the appellant must comply with the statute


In [ ]:
tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")
print(f"Reloaded. Vocab size: {tokenizer.get_vocab_size()}")

_test = tokenizer.encode("the court held that")
print(tokenizer.decode(_test.ids))
assert tokenizer.decode(_test.ids).strip() != ""

Reloaded. Vocab size: 32000
the court held that


In [ ]:
import json
import torch
import os

# Ensure device is set
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

@torch.no_grad()
def generate_single(prompt, model, tokenizer, device="cuda", target_words=128):
    # Encode prompt
    if hasattr(tokenizer, "encode"):
        encoded = tokenizer.encode(prompt)
        encoded_ids = encoded.ids if hasattr(encoded, "ids") else encoded
    else:
        encoded_ids = [0]

    input_ids = torch.tensor([encoded_ids], dtype=torch.long, device=device)
    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)

    prompt_len = input_ids.shape[1]
    generated = input_ids
    past_key_values = None

    # 1. Initial pass to populate KV cache
    outputs = model(input_ids, past_key_values=None)
    logits = outputs["logits"]
    past_key_values = outputs["past_key_values"]

    next_token_logits = logits[:, -1, :]
    next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True).to(device)
    generated = torch.cat([generated, next_token], dim=1)

    max_tokens_to_generate = int(target_words * 1.3)

    # 2. Autoregressive generation loop
    for step in range(max_tokens_to_generate - 1):
        outputs = model(next_token, past_key_values=past_key_values)
        logits = outputs["logits"]
        past_key_values = outputs["past_key_values"]

        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True).to(device)
        generated = torch.cat([generated, next_token], dim=1)

        token_val = next_token.item()

        # Diagnostic print for the first few steps to see what token it's hitting
        if step < 5:
            print(f"Step {step}: generated token ID = {token_val}")

        if hasattr(tokenizer, "eos_token_id") and token_val == tokenizer.eos_token_id:
            print(f"🛑 Stopped early: hit EOS token ID {token_val} at step {step}")
            break
        if token_val == 256: # Custom endoftext fallback
            print(f"🛑 Stopped early: hit endoftext token ID 256 at step {step}")
            break

    # Extract continuation tokens only
    continuation_tokens = generated[0, prompt_len:].tolist()

    # 🔍 Diagnostic print for raw token IDs
    print("DEBUG -> Raw generated token IDs:", continuation_tokens)

    # Decode
    if hasattr(tokenizer, "decode"):
        decoded_text = tokenizer.decode(continuation_tokens)
    else:
        decoded_text = ""

    return decoded_text

# ==========================================
# Load Test Data
# ==========================================
if 'test_data' not in globals():
    test_file_path = "test.json"
    if os.path.exists(test_file_path):
        with open(test_file_path, "r", encoding="utf-8") as f:
            test_data = json.load(f)
    else:
        raise FileNotFoundError("⚠️ 'test_data' not found in memory or disk.")

# ==========================================
# Single Example Debug Run & Print
# ==========================================
items = list(test_data.items()) if isinstance(test_data, dict) else [(str(i), {"text": x}) for i, x in enumerate(test_data)]

# Grab exactly 1 example
example_id, record = items[0]
prompt_text = record["text"] if isinstance(record, dict) else record

print(f"🐞 Running single-example debug generation for ID: {example_id}...\n")

predicted_text = generate_single(
    prompt=prompt_text,
    model=model,
    tokenizer=tokenizer,
    device=device,
    target_words=128
)

# 🖨️ Explicitly print the single prediction clearly
print("=" * 60)
print("✨ SINGLE PREDICTION RESULT:")
print("=" * 60)
print(predicted_text)
print("=" * 60)

# Save to prediction.json
output_file = "prediction.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump({str(example_id): {"text": predicted_text}}, f, ensure_ascii=False, indent=2)

print(f"\nSaved result to '{output_file}'.")

🐞 Running single-example debug generation for ID: 1...

Step 0: generated token ID = 3204
Step 1: generated token ID = 32
Step 2: generated token ID = 3204
Step 3: generated token ID = 32
Step 4: generated token ID = 3776
DEBUG -> Raw generated token IDs: [32, 3204, 32, 3204, 32, 3776, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3776, 32, 3666, 32, 3666, 32, 2692, 32, 3666, 32, 3666, 32, 2692, 32, 3204, 32, 3204, 32, 2692, 32, 3204, 32, 3204, 32, 3666, 32, 3666, 32, 3666, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3204, 32, 3204, 32, 3666, 32, 3666, 32, 2692, 32, 3776, 32, 3666, 32, 3666, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3666, 32, 2692, 32, 3666, 32, 3204, 32, 3666, 32, 2692, 32, 3204, 32, 3666, 32, 3204, 32, 3666, 32, 3204, 32, 3776, 32, 3204, 32, 3204, 32, 3204, 32, 3204, 32,

## Fine Tuning

In [ ]:
need = {
    "setup (tokenizer, data, token)": ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "train_dataset", "val_dataset"],
    "strict scorer cell":             ["strict_scores", "print_scores"],
    "Phase-2 cell 1 (toolkit)":       ["train_ner", "NERModel", "write_test_predictions", "train_docs", "val_docs", "NL", "P2_DIR", "predict_docs"],
    "attached model cell":            ["CustomDecoderLM", "alternating_patterns"],
}
for step, names in need.items():
    miss = [n for n in names if n not in globals()]
    print(f"{'✅' if not miss else '❌'} {step}" + (f"   missing: {miss}" if miss else ""))

✅ setup (tokenizer, data, token)
✅ strict scorer cell
✅ Phase-2 cell 1 (toolkit)
✅ attached model cell


In [6]:
import collections

def strict_scores(gold, pred, labels):
    """
    gold, pred: {doc_id: [[start, end, label, (text)], ...]}
    Exact match on (doc_id, start, end, label). Macro over `labels`.
    Returns dict with macro_p / macro_r / macro_f1, per-label dict, and micro_f1.
    """
    def to_set(d):
        out = set()
        for did, ents in d.items():
            for e in ents:
                out.add((str(did), int(e[0]), int(e[1]), e[2]))
        return out

    G, P = to_set(gold), to_set(pred)
    tp, gp, pp = collections.Counter(), collections.Counter(), collections.Counter()
    for x in G: gp[x[3]] += 1
    for x in P: pp[x[3]] += 1
    for x in G & P: tp[x[3]] += 1

    per = {}
    for lab in labels:
        p = tp[lab] / pp[lab] if pp[lab] else 0.0
        r = tp[lab] / gp[lab] if gp[lab] else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        per[lab] = {"p": p, "r": r, "f1": f, "support": gp[lab], "pred": pp[lab], "tp": tp[lab]}

    n = len(labels)
    macro_p = sum(v["p"] for v in per.values()) / n
    macro_r = sum(v["r"] for v in per.values()) / n
    macro_f1 = sum(v["f1"] for v in per.values()) / n
    TP, PP, GG = sum(tp.values()), sum(pp.values()), sum(gp.values())
    mp, mr = (TP / PP if PP else 0.0), (TP / GG if GG else 0.0)
    micro_f1 = 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0
    return {"macro_p": macro_p, "macro_r": macro_r, "macro_f1": macro_f1,
            "micro_f1": micro_f1, "per_label": per}

def print_scores(res):
    print(f"MACRO  P={res['macro_p']:.4f}  R={res['macro_r']:.4f}  F1={res['macro_f1']:.4f}  | micro F1={res['micro_f1']:.4f}")
    print(f"{'label':10s} {'P':>7s} {'R':>7s} {'F1':>7s} {'gold':>6s} {'pred':>6s} {'tp':>6s}")
    for lab, v in sorted(res["per_label"].items(), key=lambda kv: -kv[1]["support"]):
        print(f"{lab:10s} {v['p']:7.3f} {v['r']:7.3f} {v['f1']:7.3f} {v['support']:6d} {v['pred']:6d} {v['tp']:6d}")

# ---------- self-test on toy data ----------
_L = ["JUDGE", "COURT", "DATE"]
_gold = {"d1": [[0, 5, "JUDGE", "Smith"], [10, 20, "COURT", "High Court"]],
         "d2": [[3, 9, "DATE", "1 May"]]}

# perfect predictions (with and without optional text field)
_r = strict_scores(_gold, {"d1": [[0, 5, "JUDGE"], [10, 20, "COURT", "x"]], "d2": [[3, 9, "DATE"]]}, _L)
assert abs(_r["macro_f1"] - 1.0) < 1e-9, _r

# boundary off by one and wrong label must both count as wrong
_r = strict_scores(_gold, {"d1": [[0, 6, "JUDGE"], [10, 20, "JUDGE"]], "d2": [[3, 9, "DATE"]]}, _L)
assert _r["per_label"]["JUDGE"]["tp"] == 0 and _r["per_label"]["COURT"]["tp"] == 0
assert _r["per_label"]["DATE"]["f1"] == 1.0
assert abs(_r["macro_f1"] - 1/3) < 1e-9, _r["macro_f1"]

# empty predictions -> zeros; duplicate predictions must not inflate counts
assert strict_scores(_gold, {}, _L)["macro_f1"] == 0.0
_r = strict_scores(_gold, {"d2": [[3, 9, "DATE"], [3, 9, "DATE"]]}, _L)
assert _r["per_label"]["DATE"]["pred"] == 1 and _r["per_label"]["DATE"]["p"] == 1.0
print("✅ strict scorer self-test passed")

✅ strict scorer self-test passed


In [7]:
import os, json, math, time, random, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

for name in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "strict_scores", "print_scores"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cell that defines it."

# ============================================================
# 0. Config
# ============================================================
WINDOW, STRIDE = 1024, 768            # tokens per window / step (256 overlap)
VAL_FRACTION, SEED = 0.15, 42
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0

P2_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive not mounted — saving Phase-2 files locally (lost if the runtime disconnects).")
    P2_DIR = "CS781_Capstone_Project_Phase_2"
os.makedirs(P2_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if use_bf16 else torch.float16
PAD_ID = tokenizer.token_to_id("[PAD]")
PAD_ID = 1 if PAD_ID is None else PAD_ID
def amp(): return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)

# ============================================================
# 1. Load Phase-2 files
# ============================================================
BASE = "Phase_2_LNER"
def _path(fn): return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"{BASE}/{fn}", repo_type="dataset", token=HF_TOKEN)
def _json(fn):
    with open(_path(fn), "r", encoding="utf-8") as f: return json.load(f)

text1, gold1 = _json("Train/fold1-text.json"), _json("Train/fold1-gold.json")
text2, gold2 = _json("Train/fold2-text.json"), _json("Train/fold2-gold.json")
text3 = _json("Test/fold3-text.json")
with open(_path("Train/labels.txt"), "r", encoding="utf-8") as f:
    LABELS = [l.split()[0] for l in f if l.strip()]
print(f"Labels ({len(LABELS)}): {LABELS}")
assert len(LABELS) == 12, "labels.txt did not parse into 12 labels — paste me the first lines of the file."

def as_text(v): return v.get("text") if isinstance(v, dict) else v

def parse_entity(e):
    if isinstance(e, (list, tuple)):
        return int(e[0]), int(e[1]), e[2], (e[3] if len(e) > 3 else None)
    if isinstance(e, dict):
        g = lambda *ks: next((e[k] for k in ks if k in e), None)
        return (int(g("start", "start_char", "begin")), int(g("end", "end_char")),
                g("label", "type", "entity"), g("text", "entity_text", "span"))
    raise ValueError(f"Unknown annotation format: {e!r}")

def doc_entities(gold, doc_id):
    v = gold.get(doc_id, [])
    if isinstance(v, dict):
        v = next((x for x in v.values() if isinstance(x, list)), [])
    return [parse_entity(e) for e in v]

# ---- offset convention: is the gold `end` exclusive (python slice) or inclusive? ----
excl = incl = 0
for tx, gd in [(text1, gold1), (text2, gold2)]:
    for did, v in tx.items():
        t = as_text(v)
        for s, e, lab, et in doc_entities(gd, did):
            if et is None: continue
            if t[s:e] == et: excl += 1
            elif t[s:e + 1] == et: incl += 1
print(f"Offset test -> text[s:e]==entity: {excl} | text[s:e+1]==entity: {incl}")
if excl == 0 and incl == 0:
    print("⚠️ Could not verify the convention (no entity-text field matched). Assuming END-EXCLUSIVE.")
END_EXCL = excl >= incl
print("Gold end offset is", "EXCLUSIVE" if END_EXCL else "INCLUSIVE")

# ============================================================
# 2. BIO label space + token labelling / span decoding
# ============================================================
L2I = {"O": 0}
for lab in LABELS:
    L2I["B-" + lab] = len(L2I); L2I["I-" + lab] = len(L2I)
I2L = {v: k for k, v in L2I.items()}
NL = len(L2I)
print(f"Classes: {NL} (O + B/I for each of {len(LABELS)} labels)")

def label_tokens(offsets, ents):
    """ents: [(start, end_exclusive, label)] -> BIO ids per token. Overlapping entities: first (longest) wins."""
    n = len(offsets)
    starts = np.array([o[0] for o in offsets]); ends = np.array([o[1] for o in offsets])
    y = [0] * n; conflicts = 0
    for s, e, lab in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        i = int(np.searchsorted(ends, s, side="right"))   # first token ending after s
        first = True
        while i < n and starts[i] < e:
            if ends[i] > starts[i]:
                if y[i] != 0: conflicts += 1
                else:
                    y[i] = L2I[("B-" if first else "I-") + lab]; first = False
            i += 1
    return y, conflicts

def decode_spans(text, offsets, tags):
    """Token tag ids -> [(start, end_exclusive, label)], whitespace trimmed."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        a, b = offsets[i]
        if b <= a: continue
        name = I2L[int(t)]
        if name == "O":
            if cur: spans.append(cur); cur = None
            continue
        pre, lab = name.split("-", 1)
        if pre == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out

def spans_out(text, offsets, tags):
    return [[int(s), int(e if END_EXCL else e - 1), lab, text[s:e]] for s, e, lab in decode_spans(text, offsets, tags)]

# ============================================================
# 3. Build docs, split by DOCUMENT, windows
# ============================================================
def build_docs(text_d, gold_d, prefix=""):
    docs = []
    for did, v in text_d.items():
        t = as_text(v); enc = tokenizer.encode(t)
        d = {"id": prefix + str(did), "text": t, "ids": enc.ids, "offsets": enc.offsets}
        if gold_d is not None:
            raw = doc_entities(gold_d, did)
            bad = {lab for _, _, lab, _ in raw if lab not in LABELS}
            assert not bad, f"Gold labels not in labels.txt: {bad}"
            d["gold_raw"] = [[s, e, lab] for s, e, lab, _ in raw]
            d["labels"], d["conflicts"] = label_tokens(
                d["offsets"], [(s, e if END_EXCL else e + 1, lab) for s, e, lab, _ in raw])
        docs.append(d)
    return docs

t0 = time.time()
all_docs = build_docs(text1, gold1, "f1:") + build_docs(text2, gold2, "f2:")
test_docs = build_docs(text3, None)
print(f"Tokenized {len(all_docs)} train docs + {len(test_docs)} test docs in {time.time()-t0:.0f}s")

order = list(range(len(all_docs))); random.Random(SEED).shuffle(order)
n_val = max(1, int(VAL_FRACTION * len(all_docs)))
val_docs = [all_docs[i] for i in order[:n_val]]
train_docs = [all_docs[i] for i in order[n_val:]]

def make_windows(n, W=WINDOW, S=STRIDE):
    if n <= W: return [0]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n: starts.append(n - W)
    return starts

train_windows = [(d["ids"][s:s + WINDOW], d["labels"][s:s + WINDOW])
                 for d in train_docs for s in make_windows(len(d["ids"]))]

tok_tr = sum(len(d["ids"]) for d in train_docs)
print(f"Train docs: {len(train_docs)} ({tok_tr:,} tokens) -> {len(train_windows)} windows of ≤{WINDOW} "
      f"(~{sum(len(x) for x,_ in train_windows)/1e6:.2f}M tokens/epoch)")
print(f"Val docs: {len(val_docs)} | Test docs: {len(test_docs)} | "
      f"docs longer than WINDOW: {sum(len(d['ids'])>WINDOW for d in all_docs)} | max tokens: {max(len(d['ids']) for d in all_docs)}")
print(f"Entity-token conflicts (overlapping gold): {sum(d['conflicts'] for d in all_docs)}")
val_cnt = collections.Counter(l for d in val_docs for _, _, l in d["gold_raw"])
print("Val entity counts:", dict(val_cnt.most_common()))

# class weights (sqrt inverse frequency, clipped)
cnt = np.bincount(np.concatenate([np.array(d["labels"]) for d in train_docs]), minlength=NL).astype(np.float64)
w = np.ones(NL)
if USE_CLASS_WEIGHTS:
    for c in range(1, NL): w[c] = float(np.clip(math.sqrt(cnt[0] / max(cnt[c], 1.0)), 1.0, CLASS_W_CLIP))
CLASS_W = torch.tensor(w, dtype=torch.float32, device=device) if USE_CLASS_WEIGHTS else None
print(f"O tokens: {100*cnt[0]/cnt.sum():.1f}% | class weights (min/max entity): {w[1:].min():.1f}/{w[1:].max():.1f}")

def collate_windows(items):
    L = max(len(x) for x, _ in items)
    X = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    Y = torch.full((len(items), L), -100, dtype=torch.long)
    for i, (x, y) in enumerate(items):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X, Y

# ============================================================
# 4. ORACLE round-trip: gold -> token labels -> spans -> strict score  (the ceiling)
# ============================================================
o_docs = train_docs + val_docs
o_gold = {d["id"]: d["gold_raw"] for d in o_docs}
o_pred = {d["id"]: spans_out(d["text"], d["offsets"], d["labels"]) for d in o_docs}
o_res = strict_scores(o_gold, o_pred, LABELS)
print("\n=== ORACLE (ceiling for any model with this tokenizer + labelling) ===")
print_scores(o_res)
_lost = []
for d in o_docs:
    got = {(s, e, l) for s, e, l, _ in o_pred[d["id"]]}
    for s, e, l in d["gold_raw"]:
        if (s, e, l) not in got and len(_lost) < 5: _lost.append((d["id"], s, e, l, d["text"][s:e if END_EXCL else e + 1]))
print("Examples of gold spans the round-trip cannot reproduce:", _lost)
assert o_res["macro_f1"] > 0.8, "Oracle ceiling too low — offsets/tokenizer alignment is off. Paste this output to me."

# ============================================================
# 5. NER wrapper around the GQA decoder (no change to the LM class)
# ============================================================
class NERModel(nn.Module):
    """Token classifier: GQA decoder backbone (causal) + linear head.
    shift=k: token i is classified from the hidden state at position i+k (k-token lookahead)."""
    def __init__(self, lm, d_model, n_labels, dropout=0.1, shift=0):
        super().__init__()
        self.lm = lm
        self.shift = shift
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids):
        lm = self.lm
        x = lm.token_embedding(input_ids)
        for layer in lm.layers:
            x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        out = self.head(self.drop(lm.norm(x)))
        k = self.shift
        if k > 0:   # position i now holds the prediction for token i (made from position i+k)
            out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

# ============================================================
# 6. Prediction (windows averaged) + test-file writer
# ============================================================
@torch.no_grad()
def predict_docs(ner, docs, batch=8):
    ner.eval(); out = {}
    for d in docs:
        n = len(d["ids"])
        probs = torch.zeros(n, NL, device=device)
        starts = make_windows(n)
        for i in range(0, len(starts), batch):
            chunk = starts[i:i + batch]
            seqs = [d["ids"][s:s + WINDOW] for s in chunk]
            L = max(len(q) for q in seqs)
            x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
            for j, q in enumerate(seqs): x[j, :len(q)] = torch.tensor(q)
            with amp(): lg = ner(x.to(device))
            p = F.softmax(lg.float(), dim=-1)
            k = ner.shift
            for j, (s, q) in enumerate(zip(chunk, seqs)):
                valid = len(q) if (s + len(q) >= n) else len(q) - k   # last k tokens of a window have no lookahead
                probs[s:s + valid] += p[j, :valid]
        out[d["id"]] = spans_out(d["text"], d["offsets"], probs.argmax(-1).tolist())
    return out

def write_test_predictions(ner, name):
    preds = predict_docs(ner, test_docs)
    path = f"{P2_DIR}/prediction_{name}.json"
    with open(path, "w", encoding="utf-8") as f: json.dump(preds, f, ensure_ascii=False)
    print(f"Saved {len(preds)} test docs / {sum(len(v) for v in preds.values())} entities -> {path}")
    return path

# ============================================================
# 7. Trainer (optionally with distillation from a teacher NER model)
# ============================================================
def train_ner(ner, name, teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
              backbone_lr=1e-4, head_lr=1e-3, batch_size=8, weight_decay=0.01,
              warmup_frac=0.06, debug=False):
    tw = train_windows[:batch_size * 10] if debug else train_windows
    vdocs = val_docs[:10] if debug else val_docs
    if debug: epochs = 1; name = name + "_debug"
    vgold = {d["id"]: d["gold_raw"] for d in vdocs}
    ner.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)

    groups = {}
    for n_, p in ner.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
         for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / batch_size); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(optimizer, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))

    n_train = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    print(f"[{name}] trainable params: {n_train/1e6:.2f}M | windows: {len(tw)} | steps/epoch: {spe} | "
          f"KD: {'alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None and alpha > 0 else 'off'}")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    best_f1, best_res, bad, history = -1.0, None, 0, []
    best_path = f"{P2_DIR}/{name}_best.pt"; t0 = time.time()

    if teacher is not None: assert teacher.shift == ner.shift, "teacher and student must use the same shift"
    for ep in range(epochs):
        ner.train(); idx = list(range(len(tw))); random.shuffle(idx)
        loss_sum = torch.tensor(0.0, device=device); nb = 0
        bar = tqdm(range(0, len(idx), batch_size), desc=f"{name} ep {ep+1}/{epochs}")
        for i in bar:
            X, Y = collate_windows([tw[j] for j in idx[i:i + batch_size]])
            X, Y = X.to(device), Y.to(device)
            if ner.shift > 0: Y[:, -ner.shift:] = -100        # the last k columns have no lookahead
            with amp(): lg = ner(X)
            lg = lg.float()
            loss = F.cross_entropy(lg.reshape(-1, NL), Y.reshape(-1), weight=CLASS_W, ignore_index=-100)
            if teacher is not None and alpha > 0:
                with torch.no_grad(), amp(): tl = teacher(X)
                m = Y != -100
                kd = F.kl_div(F.log_softmax(lg[m] / T, -1), F.softmax(tl.float()[m] / T, -1),
                              reduction="batchmean") * (T * T)
                loss = (1 - alpha) * loss + alpha * kd
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ner.parameters(), 1.0)
            optimizer.step(); sched.step()
            loss_sum += loss.detach(); nb += 1
            if nb % 20 == 0: bar.set_postfix(loss=f"{(loss_sum / nb).item():.4f}")

        res = strict_scores(vgold, predict_docs(ner, vdocs), LABELS)
        history.append({"epoch": ep + 1, "train_loss": (loss_sum / max(1, nb)).item(), "macro_f1": res["macro_f1"]})
        print(f"epoch {ep+1}: train loss {history[-1]['train_loss']:.4f} | val macro P/R/F1 = "
              f"{res['macro_p']:.4f}/{res['macro_r']:.4f}/{res['macro_f1']:.4f} | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best_f1:
            best_f1, best_res, bad = res["macro_f1"], res, 0
            torch.save({"model": ner.state_dict(), "epoch": ep + 1, "val_macro_f1": best_f1}, best_path)
        else:
            bad += 1
            if bad >= patience: print(f"Early stop (no improvement for {patience} epochs)."); break

    ner.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)["model"])
    mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print(f"✅ [{name}] best val macro-F1 {best_f1:.4f} | {(time.time()-t0)/60:.1f} min | peak mem {mem:.1f} GB | saved {best_path}")
    return {"res": best_res, "history": history, "best_path": best_path, "minutes": (time.time() - t0) / 60,
            "peak_gb": mem, "trainable_params": n_train}

print("\n✅ Phase-2 toolkit ready.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Labels (12): ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Offset test -> text[s:e]==entity: 8393 | text[s:e+1]==entity: 0
Gold end offset is EXCLUSIVE
Classes: 25 (O + B/I for each of 12 labels)
Tokenized 70 train docs + 35 test docs in 2s
Train docs: 60 (399,913 tokens) -> 530 windows of ≤1024 (~0.54M tokens/epoch)
Val docs: 10 | Test docs: 35 | docs longer than WINDOW: 70 | max tokens: 23710
Entity-token conflicts (overlapping gold): 0
Val entity counts: {'PREC': 212, 'STAT': 171, 'DATE': 157, 'AUTH': 150, 'COURT': 97, 'RESP': 89, 'CASENO': 75, 'APP': 66, 'JUDGE': 32, 'WIT': 23, 'R.COUNSEL': 22, 'A.COUNSEL': 20}
O tokens: 90.8% | class weights (min/max entity): 6.1/10.0

=== ORACLE (ceiling for any model with this tokenizer + labelling) ===
MACRO  P=0.9866  R=0.9866  F1=0.9866  | micro F1=0.9819
label            P       R      F1   gold   pred     tp
DATE         1.000   1.000   1.000   1575   1575   1575
STAT         0.

In [ ]:
import glob
for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "alternating_patterns",
             "val_dataset", "train_docs", "NL", "P2_DIR", "print_scores", "strict_scores"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

DEBUG_MODE = False   # True = smoke test (few seconds per block). Set False for the real runs.

SECOND_CKPT =  "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt"      # best checkpoint of the strided model
if not os.path.exists(SECOND_CKPT):
    print("Checkpoint not found. .pt files under the Phase-1 folder:")
    for p in sorted(glob.glob("/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/**/*.pt", recursive=True)):
        print("  ", p, f"{os.path.getsize(p)/1e6:.0f} MB")
    raise FileNotFoundError("Set SECOND_CKPT to the best checkpoint of your strided model (one of the paths above).")

SECOND_PATTERNS = alternating_patterns(10, [(256, 32), (256, 128)])

def build_second_ner(dropout=0.1):
    """Fresh NER model from the PRETRAINED strided checkpoint (patterns must be passed)."""
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=SECOND_PATTERNS)
    ck_ = torch.load(SECOND_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck_["model"] if "model" in ck_ else ck_)
    return NERModel(lm_, d_model=768, n_labels=NL, dropout=dropout).to(device)

# sanity check: Phase-1 val loss with the strided patterns (compare with what you remember for this model)
_m = build_second_ner(); _m.lm.eval(); ls = []
with torch.no_grad():
    for i in range(0, 32, 4):
        ids = torch.stack([val_dataset[j]["input_ids"] for j in range(i, i + 4)]).to(device).long()
        with amp(): ls.append(_m.lm(ids, labels=ids)["loss"].float().item())
print("Layer patterns:", _m.lm.describe_patterns()[:2], "...")
print(f"Second model: Phase-1 val loss ≈ {np.mean(ls):.4f}")
del _m; torch.cuda.empty_cache()

Layer patterns: ['window=256, stride=32', 'window=256, stride=128'] ...
Second model: Phase-1 val loss ≈ 2.3546


In [ ]:
random.seed(1); torch.manual_seed(1)
ner = build_second_ner()
out_full = train_ner(ner, "ner_second_full_ft", epochs=10, patience=3, backbone_lr=1e-4, head_lr=1e-3,
                     batch_size=8, debug=DEBUG_MODE)
print("\n=== Validation (strict) — best epoch ===")
print_scores(out_full["res"])
if not DEBUG_MODE: write_test_predictions(ner, "second_full_ft")
del ner; torch.cuda.empty_cache()

[ner_second_full_ft] trainable params: 87.53M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_full_ft ep 1/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 1.1092 | val macro P/R/F1 = 0.1735/0.3374/0.2221 | 0.1 min


ner_second_full_ft ep 2/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.1466 | val macro P/R/F1 = 0.1930/0.3430/0.2424 | 0.3 min


ner_second_full_ft ep 3/10:   0%|          | 0/67 [00:00<?, ?it/s]

... (11 outputs left out)

ner_second_full_ft ep 9/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 9: train loss 0.0024 | val macro P/R/F1 = 0.2957/0.3529/0.3111 | 1.3 min


ner_second_full_ft ep 10/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 10: train loss 0.0018 | val macro P/R/F1 = 0.2938/0.3607/0.3137 | 1.5 min
Early stop (no improvement for 3 epochs).
✅ [ner_second_full_ft] best val macro-F1 0.3201 | 1.5 min | peak mem 13.0 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_full_ft_best.pt

=== Validation (strict) — best epoch ===
MACRO  P=0.3021  R=0.3596  F1=0.3201  | micro F1=0.3920
label            P       R      F1   gold   pred     tp
PREC         0.171   0.241   0.200    212    298     51
STAT         0.480   0.567   0.520    171    202     97
DATE         0.914   0.949   0.931    157    163    149
AUTH         0.178   0.107   0.133    150     90     16
COURT        0.574   0.722   0.639     97    122     70
RESP         0.140   0.067   0.091     89     43      6
CASENO       0.260   0.427   0.323     75    123     32
APP          0.157   0.121   0.137     66     51      8
JUDGE        0.455   0.469   0.462     32     33     15
WIT          0.000   0.000   0.000     23     16      

In [ ]:
class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B (A x)"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

def inject_adapters(ner, cls, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in ner.parameters(): p.requires_grad_(False)
    for layer in ner.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    for p in ner.head.parameters(): p.requires_grad_(True)

@torch.no_grad()
def merge_diff(mod):
    W0 = mod.base.weight; x = torch.randn(4, W0.shape[1], device=W0.device)
    Wm = W0 + mod.scale * (mod.B @ mod.A) if isinstance(mod, LoRALinear) else W0 + (mod.scale * (mod.B @ mod.A)) @ W0
    return (mod(x) - F.linear(x, Wm)).abs().max().item()

LORA_RESULTS = []
for lr in [1e-3, 3e-3]:
    random.seed(1); torch.manual_seed(1)
    ner = build_second_ner()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    ner.eval()
    with torch.no_grad(): before = ner(ids)
    inject_adapters(ner, LoRALinear, r=8, alpha=16)
    with torch.no_grad(): after = ner(ids)
    d = (before - after).abs().max().item(); assert d < 1e-4, f"not identity at init ({d})"
    print(f"[lora] identity-at-init OK ({d:.1e})")
    out = train_ner(ner, f"ner_second_lora_lr{lr:g}", epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, debug=DEBUG_MODE)
    print(f"[lora] merge check: {merge_diff(ner.lm.layers[0].attn.q_proj):.1e}")
    if not DEBUG_MODE: write_test_predictions(ner, f"second_lora_lr{lr:g}")
    LORA_RESULTS.append((lr, out["trainable_params"], max(h["macro_f1"] for h in out["history"]),
                         out["res"]["macro_p"], out["res"]["macro_r"], out["minutes"], out["peak_gb"]))
    del ner; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in LORA_RESULTS: print(f"{'lora':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")

[lora] identity-at-init OK (0.0e+00)
[ner_second_lora_lr0.001] trainable params: 0.43M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_lora_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.3447 | val macro P/R/F1 = 0.0595/0.2062/0.0901 | 0.1 min


ner_second_lora_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.3775 | val macro P/R/F1 = 0.1384/0.2996/0.1847 | 0.3 min


ner_second_lora_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (103 outputs left out)

ner_second_lora_lr0.003 ep 29/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 29: train loss 0.0009 | val macro P/R/F1 = 0.2580/0.3370/0.2868 | 4.1 min


ner_second_lora_lr0.003 ep 30/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 30: train loss 0.0008 | val macro P/R/F1 = 0.2694/0.3347/0.2929 | 4.3 min
✅ [ner_second_lora_lr0.003] best val macro-F1 0.2929 | 4.3 min | peak mem 11.2 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_lora_lr0.003_best.pt
[lora] merge check: 2.9e-06
Saved 35 test docs / 5145 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_second_lora_lr0.003.json

method       lr  trainable  best F1       P       R   min    GB
lora      1e-03    428,825   0.2993  0.2723  0.3460   3.7  11.2
lora      3e-03    428,825   0.2929  0.2694  0.3347   4.3  11.2


In [ ]:
class LoRMAPlusLinear(nn.Module):
    """LoRMA+: W = ((alpha/r) B A + I) W0, B,A on the OUTPUT side. y = W0 x; out = y + (alpha/r) B (A y)"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_adapters(ner, cls, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in ner.parameters(): p.requires_grad_(False)
    for layer in ner.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    for p in ner.head.parameters(): p.requires_grad_(True)

@torch.no_grad()
def merge_diff(mod):
    W0 = mod.base.weight; x = torch.randn(4, W0.shape[1], device=W0.device)
    Wm = W0 + (mod.scale * (mod.B @ mod.A)) @ W0          # (I + s·BA) W0
    return (mod(x) - F.linear(x, Wm)).abs().max().item()

LORMA_RESULTS = []
for lr in [1e-3, 3e-3]:
    random.seed(1); torch.manual_seed(1)
    ner = build_second_ner()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    ner.eval()
    with torch.no_grad(): before = ner(ids)
    inject_adapters(ner, LoRMAPlusLinear, r=10, alpha=20)
    with torch.no_grad(): after = ner(ids)
    d = (before - after).abs().max().item(); assert d < 1e-4, f"not identity at init ({d})"
    print(f"[lorma] identity-at-init OK ({d:.1e})")
    out = train_ner(ner, f"ner_second_lorma_lr{lr:g}", epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, debug=DEBUG_MODE)
    print(f"[lorma] merge check: {merge_diff(ner.lm.layers[0].attn.q_proj):.1e}")
    if not DEBUG_MODE: write_test_predictions(ner, f"second_lorma_lr{lr:g}")
    LORMA_RESULTS.append((lr, out["trainable_params"], max(h["macro_f1"] for h in out["history"]),
                          out["res"]["macro_p"], out["res"]["macro_r"], out["minutes"], out["peak_gb"]))
    del ner; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in LORMA_RESULTS: print(f"{'lorma':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")

[lorma] identity-at-init OK (0.0e+00)
[ner_second_lorma_lr0.001] trainable params: 0.43M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_lorma_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.3048 | val macro P/R/F1 = 0.0551/0.1939/0.0841 | 0.1 min


ner_second_lorma_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.3717 | val macro P/R/F1 = 0.1509/0.3099/0.1986 | 0.3 min


ner_second_lorma_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (57 outputs left out)

ner_second_lorma_lr0.003 ep 13/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 13: train loss 0.0467 | val macro P/R/F1 = 0.1916/0.3286/0.2359 | 1.8 min


ner_second_lorma_lr0.003 ep 14/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 14: train loss 0.0384 | val macro P/R/F1 = 0.2080/0.3319/0.2476 | 1.9 min
Early stop (no improvement for 6 epochs).
✅ [ner_second_lorma_lr0.003] best val macro-F1 0.2679 | 2.0 min | peak mem 11.1 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_lorma_lr0.003_best.pt
[lorma] merge check: 1.1e-06
Saved 35 test docs / 6852 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_second_lorma_lr0.003.json

method       lr  trainable  best F1       P       R   min    GB
lorma     1e-03    428,825   0.2806  0.2489  0.3361   2.6  11.1
lorma     3e-03    428,825   0.2679  0.2162  0.3785   2.0  11.1


In [ ]:
PREFIX_LEN = 32

class PrefixNERModel(NERModel):
    """Frozen backbone + trainable per-layer K/V prefixes (+ NER head), fed through past_kv."""
    def __init__(self, lm, d_model, n_labels, init_ids, dropout=0.1, shift=0):
        super().__init__(lm, d_model, n_labels, dropout, shift)
        for p in self.parameters(): p.requires_grad_(False)
        lm.eval()
        with torch.no_grad():
            dev = next(lm.parameters()).device
            pkv = lm(torch.tensor([init_ids], device=dev))["past_key_values"]
        self.prefix_k = nn.ParameterList([nn.Parameter(k.detach().clone()) for k, _ in pkv])
        self.prefix_v = nn.ParameterList([nn.Parameter(v.detach().clone()) for _, v in pkv])
        for p in self.head.parameters(): p.requires_grad_(True)
    def hidden(self, input_ids):
        lm = self.lm; B = input_ids.shape[0]
        x = lm.token_embedding(input_ids)
        for i, layer in enumerate(lm.layers):
            pkv = (self.prefix_k[i].expand(B, -1, -1, -1), self.prefix_v[i].expand(B, -1, -1, -1))
            x, _ = layer(x, lm.cos, lm.sin, past_kv=pkv)
        return lm.norm(x)
    def forward(self, input_ids):
        out = self.head(self.drop(self.hidden(input_ids)))
        k = self.shift
        if k > 0: out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

@torch.no_grad()
def plain_hidden(lm, ids):
    x = lm.token_embedding(ids)
    for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
    return lm.norm(x)

PREFIX_RESULTS = []
for lr in [1e-3, 5e-3]:
    random.seed(1); torch.manual_seed(1)
    base = build_second_ner()
    init_ids = train_docs[0]["ids"][:PREFIX_LEN]
    ner = PrefixNERModel(base.lm, d_model=768, n_labels=NL, init_ids=init_ids).to(device)
    ner.eval()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    with torch.no_grad():
        h_prefix = ner.hidden(ids)
        h_full = plain_hidden(ner.lm, torch.cat([torch.tensor([init_ids], device=device), ids], dim=1))[:, PREFIX_LEN:]
    d = (h_prefix - h_full).abs().max().item(); assert d < 1e-3, f"prefix path mismatch ({d})"
    print(f"[prefix] init check OK ({d:.1e})")
    out = train_ner(ner, f"ner_second_prefix_lr{lr:g}", epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, weight_decay=0.0, debug=DEBUG_MODE)
    if not DEBUG_MODE: write_test_predictions(ner, f"second_prefix_lr{lr:g}")
    PREFIX_RESULTS.append((lr, out["trainable_params"], max(h["macro_f1"] for h in out["history"]),
                           out["res"]["macro_p"], out["res"]["macro_r"], out["minutes"], out["peak_gb"]))
    del ner, base; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in PREFIX_RESULTS: print(f"{'prefix':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")

[prefix] init check OK (2.0e-05)
[ner_second_prefix_lr0.001] trainable params: 0.18M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_prefix_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.6633 | val macro P/R/F1 = 0.0096/0.1004/0.0175 | 0.1 min


ner_second_prefix_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.7808 | val macro P/R/F1 = 0.0524/0.2363/0.0844 | 0.3 min


ner_second_prefix_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (111 outputs left out)

ner_second_prefix_lr0.005 ep 29/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 29: train loss 0.2132 | val macro P/R/F1 = 0.1252/0.3012/0.1728 | 4.0 min


ner_second_prefix_lr0.005 ep 30/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 30: train loss 0.2113 | val macro P/R/F1 = 0.1246/0.3012/0.1724 | 4.1 min
✅ [ner_second_prefix_lr0.005] best val macro-F1 0.1751 | 4.1 min | peak mem 11.0 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_prefix_lr0.005_best.pt
Saved 35 test docs / 11651 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_second_prefix_lr0.005.json

method       lr  trainable  best F1       P       R   min    GB
prefix    1e-03    183,065   0.1496  0.1044  0.2933   4.1  11.0
prefix    5e-03    183,065   0.1751  0.1269  0.3048   4.1  11.0


## Rhetorical Role Prediction

In [8]:
import os, json, math, time, random, pickle, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
for n in ["tokenizer", "device", "HF_TOKEN", "HF_DATASET_REPO"]: assert n in globals(), f"'{n}' missing, run the setup cell"

P3_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_3"; os.makedirs(P3_DIR + "/submission", exist_ok=True)
WINDOW, MAX_SENT_TOK = 1024, 128
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0
PAD_ID = tokenizer.token_to_id("[PAD]"); NL_IDS = tokenizer.encode("\n").ids
def amp(): return torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)

# ---------------- data ----------------
def _load(f): return json.load(open(hf_hub_download(HF_DATASET_REPO, f"Phase_3_RR/{f}", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"))
L2I = _load("Train/label_vocab.json"); LABELS = sorted(L2I, key=L2I.get); NC = len(LABELS)

def norm_docs(raw, domain):
    items = list(raw.items()) if isinstance(raw, dict) else raw
    docs = []
    for it in items:
        did, v = (it if isinstance(it, tuple) else (it["id"], it))
        text = v if isinstance(v, list) else v["text"]
        labels = v.get("labels") if isinstance(v, dict) else None
        d = dict(id=did, domain=domain, text=text, labels=labels)
        d["sent_ids"] = [tokenizer.encode(s.strip()).ids[:MAX_SENT_TOK] + NL_IDS for s in text]
        d["y"] = [L2I[l] for l in labels] if labels is not None else None
        if labels is not None: assert len(labels) == len(text)
        docs.append(d)
    return docs

train_docs = norm_docs(_load("Train/CL_train.json"), "CL") + norm_docs(_load("Train/IT_train.json"), "IT")
val_docs   = norm_docs(_load("Train/CL_dev.json"), "CL")   + norm_docs(_load("Train/IT_dev.json"), "IT")
test_docs  = norm_docs(_load("Test/CL_test.json"), "CL")   + norm_docs(_load("Test/IT_test.json"), "IT")
assert len({d["id"] for d in test_docs}) == len(test_docs), "test doc ids are not unique across domains"
for nm, ds in [("train", train_docs), ("val", val_docs), ("test", test_docs)]:
    print(f"{nm:5s} docs={len(ds)} sents={sum(len(d['text']) for d in ds)} "
          f"CL={sum(d['domain']=='CL' for d in ds)} IT={sum(d['domain']=='IT' for d in ds)}")

cnt = np.bincount([y for d in train_docs for y in d["y"]], minlength=NC).astype(float)
CLASS_W = torch.tensor(np.clip(np.sqrt(cnt.sum() / (NC * np.maximum(cnt, 1))), 0, CLASS_W_CLIP), dtype=torch.float32, device=device)
print("class weights:", {LABELS[i]: round(float(CLASS_W[i]), 2) for i in range(NC)})

# ---------------- context modes / windows ----------------
CTX = {"sentence": dict(max_sents=1, stride_frac=0.5),        # each sentence alone
       "local8":   dict(max_sents=8, stride_frac=0.5),        # up to 8 consecutive sentences
       "document": dict(max_sents=10**9, stride_frac=0.75)}   # as many sentences as fit in 1024 tokens
BW = {"sentence": 64, "local8": 32, "document": 8}            # windows per batch

def make_windows(sent_ids, max_sents, stride_frac, max_tokens=WINDOW):
    n, lens, wins, a = len(sent_ids), [len(x) for x in sent_ids], [], 0
    while True:
        b, tot = a, 0
        while b < n and tot + lens[b] <= max_tokens and (b - a) < max_sents: tot += lens[b]; b += 1
        wins.append((a, b))
        if b >= n: break
        c, acc, target = a, 0, max(1, int(stride_frac * tot))
        while c < b and acc < target: acc += lens[c]; c += 1
        a = max(c, a + 1)
    return wins

def all_windows(docs, mode): return [(di, a, b) for di, d in enumerate(docs) for a, b in make_windows(d["sent_ids"], **CTX[mode])]

def collate(wins, docs):
    seqs, spans, ys = [], [], []
    for di, a, b in wins:
        d = docs[di]; ids, sp = [], []
        for k in range(a, b): s = len(ids); ids += d["sent_ids"][k]; sp.append((s, len(ids)))
        seqs.append(ids); spans.append(sp)
        if d["y"] is not None: ys += d["y"][a:b]
    L = max(len(s) for s in seqs)
    x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    for j, s in enumerate(seqs): x[j, :len(s)] = torch.tensor(s)
    return x, spans, (torch.tensor(ys) if ys else None)

# ---------------- model: backbone + sentence pooling + head ----------------
def pool(H, spans):
    jw, st, en = [], [], []
    for j, sp in enumerate(spans):
        for s, e in sp: jw.append(j); st.append(s); en.append(e)
    jw, st, en = [torch.tensor(t, device=H.device) for t in (jw, st, en)]
    last = H[jw, en - 1]                                                   # delimiter token: has seen the whole sentence
    cs = torch.cat([torch.zeros_like(H[:, :1]).float(), H.float().cumsum(1)], dim=1)
    mean = ((cs[jw, en] - cs[jw, st]) / (en - st).unsqueeze(1)).to(H.dtype)  # mean over the sentence's tokens
    return torch.cat([last, mean], -1)

class RRModel(nn.Module):
    """Decoder backbone (causal) -> per-sentence features [last token ; mean] -> linear head."""
    def __init__(self, lm, d_model, n_labels=NC, dropout=0.1):
        super().__init__()
        self.lm, self.drop, self.head = lm, nn.Dropout(dropout), nn.Linear(2 * d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def hidden(self, ids):
        lm = self.lm; x = lm.token_embedding(ids)
        for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        return lm.norm(x)
    def forward(self, ids, spans): return self.head(self.drop(pool(self.hidden(ids), spans)))

# ---------------- metrics (own implementation) ----------------
def rr_scores(gold, pred):
    g, p = np.array(gold), np.array(pred); labs = sorted(set(g.tolist()) | set(p.tolist())); per = {}
    for l in labs:
        tp = int(((g == l) & (p == l)).sum()); fp = int(((g != l) & (p == l)).sum()); fn = int(((g == l) & (p != l)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[LABELS[l]] = dict(p=pr, r=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=int((g == l).sum()))
    m = lambda k: float(np.mean([v[k] for v in per.values()]))
    return dict(macro_f1=m("f1"), macro_p=m("p"), macro_r=m("r"), acc=float((g == p).mean()), per=per)

# ---------------- prediction ----------------
@torch.no_grad()
def predict_probs(model, docs, mode):
    model.eval(); bw = BW[mode]; out = {}
    for d in docs:
        n = len(d["sent_ids"]); P = torch.zeros(n, NC, device=device); C = torch.zeros(n, device=device)
        wins = [(0, a, b) for a, b in make_windows(d["sent_ids"], **CTX[mode])]
        for i in range(0, len(wins), bw):
            chunk = wins[i:i + bw]; x, spans, _ = collate(chunk, [d])
            with amp(): lg = model(x.to(device), spans)
            p = F.softmax(lg.float(), -1); k = 0
            for _, a, b in chunk: P[a:b] += p[k:k + b - a]; C[a:b] += 1; k += b - a
        out[d["id"]] = (P / C.unsqueeze(1)).cpu().numpy()
    return out

def evaluate(model, docs, mode):
    P = predict_probs(model, docs, mode)
    G = {dm: ([], []) for dm in ("CL", "IT", "ALL")}
    for d in docs:
        pr = P[d["id"]].argmax(1).tolist()
        for key in (d["domain"], "ALL"): G[key][0].extend(d["y"]); G[key][1].extend(pr)
    res = rr_scores(*G["ALL"]); res["by_domain"] = {dm: rr_scores(*G[dm]) for dm in ("CL", "IT") if G[dm][0]}
    return res, P

def print_rr(res):
    print(f"ALL  macro-F1 {res['macro_f1']:.4f} | P {res['macro_p']:.4f} | R {res['macro_r']:.4f} | acc {res['acc']:.4f}")
    for dm, r in res["by_domain"].items(): print(f"  {dm}  macro-F1 {r['macro_f1']:.4f} | P {r['macro_p']:.4f} | R {r['macro_r']:.4f} | acc {r['acc']:.4f}")

def write_test_predictions(model, name, mode):
    P = predict_probs(model, test_docs, mode)
    allp = {d["id"]: [LABELS[i] for i in P[d["id"]].argmax(1)] for d in test_docs}
    for d in test_docs: assert len(allp[d["id"]]) == len(d["text"])
    for suffix, sel in [("all", allp), ("CL", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                        ("IT", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
        json.dump(sel, open(f"{P3_DIR}/prediction_{name}_{suffix}.json", "w"), ensure_ascii=False)
    pickle.dump(P, open(f"{P3_DIR}/probs_test_{name}.pkl", "wb"))
    print(f"Saved prediction_{name}_{{all,CL,IT}}.json ({len(allp)} docs, {sum(map(len, allp.values()))} sentences)")

# ---------------- trainer (optional distillation from a teacher RRModel) ----------------
def train_rr(model, name, mode="document", teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
             backbone_lr=1e-4, head_lr=1e-3, bw=None, weight_decay=0.01, warmup_frac=0.06, debug=False, seed=1):
    random.seed(seed); torch.manual_seed(seed); bw = bw or BW[mode]
    tw = all_windows(train_docs, mode); vdocs = val_docs
    if debug:
        epochs, name = 1, name + "_debug"; tw = tw[:bw * 4]
        vdocs = [next(d for d in val_docs if d["domain"] == dm) for dm in ("CL", "IT")]
    model.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)
    groups = {}
    for n_, p in model.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
                             for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / bw); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{name}] mode={mode} windows={len(tw)} trainable={n_train:,} {'KD alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None else ''}")
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    best, best_state, bad, hist = -1.0, None, 0, []
    cw = CLASS_W if USE_CLASS_WEIGHTS else None
    for ep in range(epochs):
        model.train(); random.shuffle(tw); tl, nb = 0.0, 0
        for i in range(0, len(tw), bw):
            x, spans, y = collate(tw[i:i + bw], train_docs); x, y = x.to(device), y.to(device)
            with amp():
                lg = model(x, spans).float()
                loss = F.cross_entropy(lg, y, weight=cw)
                if teacher is not None:
                    with torch.no_grad(): tlg = teacher(x, spans).float()
                    kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tlg / T, -1), reduction="batchmean") * T * T
                    loss = (1 - alpha) * loss + alpha * kd
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); sched.step(); tl += loss.item(); nb += 1
        res, _ = evaluate(model, vdocs, mode)
        hist.append(dict(epoch=ep + 1, loss=tl / nb, macro_f1=res["macro_f1"], **{f"f1_{k}": v["macro_f1"] for k, v in res["by_domain"].items()}))
        print(f"  ep{ep+1:2d} loss {tl/nb:.4f} | val macro-F1 {res['macro_f1']:.4f} "
              + " ".join(f"{k} {v['macro_f1']:.3f}" for k, v in res["by_domain"].items()) + f" | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best:
            best, bad = res["macro_f1"], 0
            names = {n_ for n_, p in model.named_parameters() if p.requires_grad}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}
        else:
            bad += 1
            if bad >= patience: print("  early stop"); break
    model.load_state_dict(best_state, strict=False)
    path = f"{P3_DIR}/rr_{name}_best.pt"; torch.save(best_state, path)
    res, P = evaluate(model, vdocs, mode)
    print(f"[{name}] BEST on dev:"); print_rr(res)
    return dict(res=res, history=hist, best_path=path, minutes=(time.time() - t0) / 60,
                peak_gb=torch.cuda.max_memory_allocated() / 1e9, trainable_params=n_train, val_probs=P)

# ---------------- sanity checks (no model needed) ----------------
for mode in CTX:
    W = all_windows(train_docs, mode)
    cover = collections.defaultdict(set)
    for di, a, b in W: cover[di].update(range(a, b))
    assert all(len(cover[di]) == len(d["sent_ids"]) for di, d in enumerate(train_docs)), f"{mode}: some sentences not covered"
    mx = max(sum(len(train_docs[di]["sent_ids"][k]) for k in range(a, b)) for di, a, b in W)
    print(f"{mode:9s} windows={len(W):5d} | every sentence covered | longest window {mx} tokens (<= {WINDOW})")
    assert mx <= WINDOW
tt = sorted(sum(map(len, d["sent_ids"])) for d in test_docs)
print("test tokens/doc median", tt[len(tt) // 2], "max", tt[-1], "| val labels:", collections.Counter(LABELS[y] for d in val_docs for y in d["y"]).most_common(4), "...")
print("✅ Phase-3 toolkit ready")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

train docs=80 sents=16845 CL=40 IT=40
val   docs=10 sents=2142 CL=5 IT=5
test  docs=10 sents=2197 CL=5 IT=5
class weights: {'Fact': 0.56, 'Issue': 3.48, 'ArgumentPetitioner': 0.9, 'ArgumentRespondent': 0.98, 'Statute': 1.09, 'Dissent': 2.14, 'PrecedentReliedUpon': 0.82, 'PrecedentNotReliedUpon': 1.51, 'PrecedentOverruled': 10.0, 'RulingByLowerCourt': 1.22, 'RatioOfTheDecision': 0.55, 'RulingByPresentCourt': 2.05, 'None': 2.09}
sentence  windows=16845 | every sentence covered | longest window 129 tokens (<= 1024)
local8    windows= 3694 | every sentence covered | longest window 825 tokens (<= 1024)
document  windows=  892 | every sentence covered | longest window 1024 tokens (<= 1024)
test tokens/doc median 7457 max 35993 | val labels: [('Fact', 444), ('RatioOfTheDecision', 403), ('ArgumentPetitioner', 400), ('ArgumentRespondent', 272)] ...
✅ Phase-3 toolkit ready


In [ ]:
DEBUG_MODE = False   # True first, then False
SECOND_CKPT = f"{DRIVE_PROJECT_DIR}/alt_stride_32_128_checkpoints/best.pt"
assert os.path.exists(SECOND_CKPT), SECOND_CKPT
for n in ["CustomDecoderLM", "alternating_patterns"]: assert n in globals(), "run the strided model-definition cell first"
SECOND_PATTERNS = alternating_patterns(10, [(256, 32), (256, 128)])

def build_second_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=SECOND_PATTERNS)
    ck = torch.load(SECOND_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)       # pretrained strided Phase-1 weights
    return RRModel(lm_, 768, NC, dropout).to(device)

class LoRMAPlusLinear(nn.Module):
    """LoRMA+: W = ((alpha/r) B A + I) W0, acting on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_lorma(rr, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in rr.parameters(): p.requires_grad_(False)
    for layer in rr.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, LoRMAPlusLinear(base, r, alpha).to(base.weight.device))
    for p in rr.head.parameters(): p.requires_grad_(True)

@torch.no_grad()
def check_merge(mod):
    W0 = mod.base.weight; x = torch.randn(4, W0.shape[1], device=W0.device)
    return (mod(x) - F.linear(x, W0 + (mod.scale * (mod.B @ mod.A)) @ W0)).abs().max().item()

SEC = {}
# --- reference: full fine-tuning of the strided model ---
random.seed(1); torch.manual_seed(1)
m = build_second_rr()
SEC["full"] = train_rr(m, "second_full_document", mode="document", epochs=10, patience=3,
                       backbone_lr=1e-4, head_lr=1e-3, debug=DEBUG_MODE)
if not DEBUG_MODE: write_test_predictions(m, "second_full_document", "document")
del m; torch.cuda.empty_cache()

# --- LoRMA+ (r=10, alpha=20) on q/k/v/out in all layers ---
for lr in [1e-3, 3e-3]:
    random.seed(1); torch.manual_seed(1)
    m = build_second_rr()
    xb, sp, _ = collate(all_windows(val_docs, "sentence")[:4], val_docs); xb = xb.to(device)
    m.eval()
    with torch.no_grad(): before = m(xb, sp)
    inject_lorma(m, 10, 20)
    with torch.no_grad(): d = (before - m(xb, sp)).abs().max().item()
    assert d < 1e-3, f"not identity at init ({d})"; print(f"[lorma+] identity at init OK ({d:.1e})")
    SEC[f"lorma{lr:g}"] = train_rr(m, f"second_lorma_lr{lr:g}", mode="document", epochs=20, patience=5,
                                   backbone_lr=lr, head_lr=1e-3, debug=DEBUG_MODE)
    print("merge check (layer-0 q_proj):", f"{check_merge(m.lm.layers[0].attn.q_proj):.1e}")
    if not DEBUG_MODE: write_test_predictions(m, f"second_lorma_lr{lr:g}", "document")
    del m; torch.cuda.empty_cache()

print(f"\n{'run':14s} {'F1':>7s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'trainable':>11s} {'min':>5s} {'GB':>5s}")
for k, o in SEC.items():
    r = o["res"]; bd = r["by_domain"]
    print(f"{k:14s} {r['macro_f1']:7.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} {r['macro_p']:7.4f} {r['macro_r']:7.4f} {o['trainable_params']:11,d} {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

[second_full_document] mode=document windows=892 trainable=87,526,669 
  ep 1 loss 1.9995 | val macro-F1 0.3656 CL 0.325 IT 0.370 | 0.3 min
  ep 2 loss 0.8946 | val macro-F1 0.4770 CL 0.469 IT 0.484 | 0.5 min
  ep 3 loss 0.3851 | val macro-F1 0.4457 CL 0.381 IT 0.498 | 0.8 min
  ep 4 loss 0.1840 | val macro-F1 0.4799 CL 0.399 IT 0.528 | 1.0 min
  ep 5 loss 0.0969 | val macro-F1 0.4955 CL 0.452 IT 0.515 | 1.3 min
  ep 6 loss 0.0423 | val macro-F1 0.4617 CL 0.417 IT 0.489 | 1.6 min
  ep 7 loss 0.0251 | val macro-F1 0.4914 CL 0.423 IT 0.524 | 1.8 min
  ep 8 loss 0.0132 | val macro-F1 0.5091 CL 0.449 IT 0.529 | 2.1 min
  ep 9 loss 0.0083 | val macro-F1 0.5001 CL 0.456 IT 0.515 | 2.3 min
  ep10 loss 0.0027 | val macro-F1 0.5112 CL 0.446 IT 0.526 | 2.6 min
[second_full_document] BEST on dev:
ALL  macro-F1 0.5112 | P 0.5610 | R 0.4911 | acc 0.5593
  CL  macro-F1 0.4456 | P 0.4846 | R 0.4916 | acc 0.5423
  IT  macro-F1 0.5257 | P 0.6212 | R 0.4877 | acc 0.5912
Saved prediction_second_full_docu

## Court Judgement Prediction

In [ ]:
import json, os, numpy as np
from huggingface_hub import snapshot_download

P4_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_4"
os.makedirs(P4_DIR, exist_ok=True)

# Downloads only the Phase-4 folder from the course dataset repo (same repo as Phase 3).
# If the folder name differs, change allow_patterns.
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_4_CJPE/*", "*/Phase_4_CJPE/*"],
                          token=HF_TOKEN)
print("downloaded to:", local)
for r, _, fs in os.walk(local):
    for f in fs:
        if f.endswith(".json"): print(os.path.join(r, f))

def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_4" in r: return os.path.join(r, name)

def load(name):
    d = json.load(open(find(name)))
    print(name, type(d).__name__, len(d))
    return d

tr, dv, te = load("train.json"), load("dev.json"), load("test.json")

def peek(d):
    x = d[0] if isinstance(d, list) else {k: (v[0] if isinstance(v, list) else v) for k, v in d.items()}
    print({k: (str(v)[:80] + "…" if len(str(v)) > 80 else v) for k, v in x.items()})
for n, d in [("train", tr), ("dev", dv), ("test", te)]:
    print(n); peek(d)

# label balance + token lengths (uses your Phase-1 tokenizer from the setup cell)
def rows(d):
    return d if isinstance(d, list) else [dict(zip(d.keys(), v)) for v in zip(*d.values())]
for n, d in [("train", tr), ("dev", dv)]:
    R = rows(d)
    labs = np.array([r["label"] for r in R])
    lens = np.array([len(tokenizer.encode(r["text"]).ids) for r in R[:300]])
    print(n, "label counts:", np.bincount(labs), "| tokens (first 300 docs) median/p90/max:",
          int(np.median(lens)), int(np.percentile(lens, 90)), int(lens.max()))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_3_RR/Test/IT_test.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_3_RR/Test/CL_test.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a

In [10]:
   import pickle, os
   P4_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_4"
   C = pickle.load(open(f"{P4_DIR}/cjpe_tok_cache.pkl", "rb"))
   print({k: len(v["ids"]) for k, v in C.items()}, "✅ Phase-4 data ready")



{'train': 37387, 'dev': 3505, 'test': 1517} ✅ Phase-4 data ready


In [ ]:
import pickle, time, numpy as np, os

CACHE = f"{P4_DIR}/cjpe_tok_cache.pkl"
HEAD_N = TAIL_N = 4096

def rows_of(d): return d
def tok_split(texts):
    out = []
    for i in range(0, len(texts), 500):
        batch = texts[i:i+500]
        # only the first/last 40k chars can reach the first/last 4096 tokens
        h = [t[:40000] for t in batch]
        tl = [t[-40000:] if len(t) > 80000 else None for t in batch]
        eh = tokenizer.encode_batch(h)
        et = tokenizer.encode_batch([x for x in tl if x is not None]) if any(x is not None for x in tl) else []
        k = 0
        for j, t in enumerate(batch):
            hi = eh[j].ids
            if tl[j] is None:                      # short enough: whole doc encoded
                ti, total = hi, len(hi)
            else:
                ti, total = et[k].ids, -1; k += 1  # total unknown (very long)
            out.append((np.array(hi[:HEAD_N], dtype=np.uint16),
                        np.array(ti[-TAIL_N:], dtype=np.uint16),
                        total if total >= 0 else 10**6))
    return out

if os.path.exists(CACHE):
    C = pickle.load(open(CACHE, "rb")); print("loaded cache")
else:
    t0 = time.time(); C = {}
    for name, d in [("train", tr), ("dev", dv), ("test", te)]:
        texts = [r["text"] for r in d]
        toks = tok_split(texts)
        C[name] = dict(ids=[r["id"] for r in d],
                       y=(np.array([r["label"] for r in d]) if name != "test" else None),
                       head=[a for a, _, _ in toks], tail=[b for _, b, _ in toks],
                       n=np.array([n for _, _, n in toks]))
        print(name, len(d), f"{(time.time()-t0)/60:.1f} min")
    pickle.dump(C, open(CACHE, "wb"))

# sanity: test 'label' field must be ignored; just check if it is a constant placeholder
print("test label field values:", np.unique([r.get("label") for r in te], return_counts=True))
print("train docs ≤4096 tok:", (C["train"]["n"] <= 4096).mean().round(3),
      "| ≤8192:", (C["train"]["n"] <= 8192).mean().round(3))
print("✅ cache ready:", {k: len(v["ids"]) for k, v in C.items()})

In [13]:
def build_gqa_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)      # strict: pretrained Phase-1 weights
    return RRModel(lm_, 768, NC, dropout).to(device)


def cj_from(rr):
    """Phase-3 RRModel -> CJPE model: keep the Phase-1 LM, discard the old head, fresh 2-class head."""
    d = rr.lm.token_embedding.weight.shape[1]          # model width, independent of the old head
    m = CJModel(rr.lm, d, 2, 0.1).to(device)
    m.head = nn.Linear(2 * d, 2).to(device)
    nn.init.normal_(m.head.weight, std=0.02); nn.init.zeros_(m.head.bias)
    return m

In [14]:
import json, time, math, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

CH = 1024
MODES = {"first1": 1, "last1": 1, "first4": 4, "last4": 4, "ht4": 4}   # value = max chunks per doc

# ---------- document -> chunks (from the cached head/tail tokens) ----------
def get_chunks(split, i, mode):
    d = C[split]; n = int(d["n"][i]); head, tail = d["head"][i], d["tail"][i]
    if   mode == "first1": x = head[:CH]
    elif mode == "last1":  x = tail[-CH:]
    elif mode == "first4": x = head[:4*CH]
    elif mode == "last4":  x = tail[-4*CH:]
    elif mode == "ht4":    x = head if n <= 4*CH else np.concatenate([head[:2*CH], tail[-2*CH:]])
    L = len(x)
    if mode.startswith("last"):                      # align chunks to the END of the judgment
        ends = list(range(L, 0, -CH))[::-1]
        return [x[max(0, e-CH):e] for e in ends]
    return [x[s:s+CH] for s in range(0, L, CH)]

def collate_cj(split, idxs, mode):
    seqs, owner = [], []
    for j, i in enumerate(idxs):
        cs = get_chunks(split, i, mode)
        if not cs or len(cs[0]) == 0: cs = [np.zeros(1, np.uint16)]
        for c in cs: seqs.append(c); owner.append(j)
    T = max(len(s) for s in seqs)
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.from_numpy(s.astype(np.int64))
    lens = torch.tensor([len(s) for s in seqs])
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(idxs)

# ---------- model: Phase-1 LM -> chunk features -> mean over chunks -> 2-class head ----------
class CJModel(RRModel):
    def forward(self, ids, lens, owner, n_docs):
        H = self.hidden(ids)                                            # (chunks, T, d)
        B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        mean = (H.float() * mask).sum(1) / lens[:, None]
        last = H[torch.arange(B, device=H.device), lens - 1].float()
        f = torch.cat([last, mean], -1)                                 # (chunks, 2d)
        D = torch.zeros(n_docs, f.size(1), device=f.device).index_add_(0, owner, f)
        D = D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]
        return self.head(self.drop(D))

def cj_from(rr):
    """Turn a Phase-3 RRModel (13-label head) into a CJPE model: same LM, fresh 2-class head."""
    d = rr.head.in_features // 2
    return CJModel(rr.lm, d, 2, 0.1).to(device)

# ---------- inference / metrics ----------
@torch.no_grad()
def cj_logits(model, split, mode, idxs=None, chunk_batch=32):
    model.eval()
    idxs = np.arange(len(C[split]["ids"])) if idxs is None else np.asarray(idxs)
    bs = max(1, chunk_batch // MODES[mode]); out = np.zeros((len(idxs), 2), np.float32)
    for s in range(0, len(idxs), bs):
        b = idxs[s:s+bs]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lg = model(*collate_cj(split, b, mode))
        out[s:s+len(b)] = lg.float().cpu().numpy()
    return out

def cj_scores(y, lg, thr=0.5):
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pr = (p1 > thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pr, average="macro", zero_division=0)
    return dict(f1=Fm, p=P, r=R, cm=confusion_matrix(y, pr, labels=[0, 1]))

def best_thr(y, lg):
    best = (0.5, -1.0)
    for t in np.arange(0.2, 0.81, 0.02):
        f = cj_scores(y, lg, t)["f1"]
        if f > best[1]: best = (float(t), f)
    return best

# ---------- training (CE, or KD from cached teacher logits) ----------
def train_cj(model, name, mode, epochs=2, lr=5e-5, head_lr=1e-3, wd=0.01, bs_chunks=16,
             n_train=None, kd=None, alpha=0.5, T=2.0, evals_per_epoch=2, patience=3,
             class_weight=True, debug=False, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    ytr = C["train"]["y"]; ydv = C["dev"]["y"]
    dev_idx = np.arange(200) if debug else np.arange(len(ydv))
    tr_idx = rng.permutation(len(ytr))[: (128 if debug else (n_train or len(ytr)))]
    if debug: epochs, evals_per_epoch = 1, 1
    cw = None
    if class_weight:
        cnt = np.bincount(ytr, minlength=2); cw = torch.tensor(len(ytr) / (2 * cnt), dtype=torch.float, device=device)
    kd_t = torch.tensor(kd, dtype=torch.float) if kd is not None else None

    head_p = [p for p in model.head.parameters()]
    hid = {id(p) for p in head_p}
    body_p = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body_p, "lr": lr}, {"params": head_p, "lr": head_lr}], weight_decay=wd)
    bs = max(1, bs_chunks // MODES[mode]); spe = math.ceil(len(tr_idx) / bs); total = spe * epochs
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    every = max(1, spe // evals_per_epoch)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{name}] mode={mode} train_docs={len(tr_idx)} steps={total} trainable={n_tr/1e6:.2f}M / {n_all/1e6:.1f}M")

    path = f"{P4_DIR}/cj_{name}_best.pt"; best = (-1.0, None); bad = 0; step = 0; t0 = time.time()
    torch.cuda.reset_peak_memory_stats(); hist = []; done = False
    for ep in range(epochs):
        perm = rng.permutation(tr_idx); model.train()
        for s in range(0, len(perm), bs):
            b = perm[s:s+bs]
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(*collate_cj("train", b, mode)).float()
            y = torch.tensor(ytr[b], device=device)
            loss = F.cross_entropy(lg, y, weight=cw)
            if kd_t is not None:
                t_lg = kd_t[b].to(device)
                kl = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(t_lg / T, -1), reduction="batchmean") * T * T
                loss = (1 - alpha) * loss + alpha * kl
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); step += 1
            if step == 20: print(f"  ~{(time.time()-t0)/20*total/60:.0f} min training total (excl. dev evals)")
            if step % 100 == 0: print(f"  step {step}/{total} loss {loss.item():.4f}")
            if step % every == 0 or step == total:
                lgd = cj_logits(model, "dev", mode, dev_idx); sc = cj_scores(ydv[dev_idx], lgd)
                hist.append((step, sc["f1"])); model.train()
                print(f"  eval@{step}: macro-F1 {sc['f1']:.4f}  P {sc['p']:.4f}  R {sc['r']:.4f}  ({(time.time()-t0)/60:.1f} min)")
                if sc["f1"] > best[0]:
                    best = (sc["f1"], step); bad = 0
                    torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, path)
                else:
                    bad += 1
                    if bad >= patience: done = True; break
        if done: break
    if os.path.exists(path):
        model.load_state_dict(torch.load(path), strict=False)
    lgd = cj_logits(model, "dev", mode, dev_idx); y = ydv[dev_idx]
    sc = cj_scores(y, lgd); thr, f_thr = best_thr(y, lgd)
    res = dict(name=name, mode=mode, f1=sc["f1"], p=sc["p"], r=sc["r"], cm=sc["cm"].tolist(), thr=thr, f1_thr=f_thr,
               minutes=(time.time()-t0)/60, peak_gb=torch.cuda.max_memory_allocated()/1e9,
               trainable=n_tr, total=n_all, best_step=best[1], dev_logits=lgd)
    print(f"✅ [{name}] dev macro-F1 {sc['f1']:.4f} (P {sc['p']:.4f} R {sc['r']:.4f}) | tuned thr {thr:.2f} → {f_thr:.4f} "
          f"| {res['minutes']:.1f} min, peak {res['peak_gb']:.1f} GB\n   confusion [[TN,FP],[FN,TP]] = {res['cm']}")
    return res

# ---------- outputs ----------
def write_pred(model, name, mode, thr=0.5):
    lg = cj_logits(model, "test", mode)
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pred = (p1 > thr).astype(int)
    out = [{"id": i, "pred": int(p)} for i, p in zip(C["test"]["ids"], pred)]
    assert len(out) == len(C["test"]["ids"]) == 1517
    f = f"{P4_DIR}/prediction_{name}.json"; json.dump(out, open(f, "w"))
    np.save(f"{P4_DIR}/test_logits_{name}.npy", lg)
    print(f"wrote {f} | preds 0/1 = {np.bincount(pred, minlength=2).tolist()}  (rename to prediction.json when uploading)")

def save_kd_logits(model, name, mode):
    lg = cj_logits(model, "train", mode); np.save(f"{P4_DIR}/kd_logits_{name}.npy", lg); return lg

print("✅ Phase-4 toolkit ready")

✅ Phase-4 toolkit ready


In [ ]:
SECOND_CKPT = f"{DRIVE_PROJECT_DIR}/alt_stride_32_128_checkpoints/best.pt"
assert os.path.exists(SECOND_CKPT), SECOND_CKPT
for n in ["CustomDecoderLM", "alternating_patterns", "cj_from", "train_cj"]:
    assert n in globals(), f"missing {n}: run the earlier cells first"
SECOND_PATTERNS = alternating_patterns(10, [(256, 32), (256, 128)])

def build_second_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=SECOND_PATTERNS)
    ck = torch.load(SECOND_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)       # pretrained strided Phase-1 weights
    return RRModel(lm_, 768, 13, dropout).to(device)

class LoRMAPlusLinear(nn.Module):
    """LoRMA+: W = ((alpha/r) B A + I) W0, acting on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_lorma(m, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in m.parameters(): p.requires_grad_(False)
    for layer in m.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, LoRMAPlusLinear(base, r, alpha).to(base.weight.device))
    for p in m.head.parameters(): p.requires_grad_(True)

@torch.no_grad()
def check_merge(mod):
    W0 = mod.base.weight; x = torch.randn(4, W0.shape[1], device=W0.device)
    return (mod(x) - F.linear(x, W0 + (mod.scale * (mod.B @ mod.A)) @ W0)).abs().max().item()

DEBUG   = False       # True first (~1-2 min), then False
MODE    = "last4"
N_TRAIN = None          # None = all 37k docs; e.g. 20000 to roughly halve the time
CJ = globals().get("CJ", {})

random.seed(1); torch.manual_seed(1)
m = cj_from(build_second_rr())
batch = collate_cj("dev", [0, 1], "last1"); m.eval()
with torch.no_grad(): before = m(*batch)
inject_lorma(m, 10, 20)
with torch.no_grad(): d = (before - m(*batch)).abs().max().item()
assert d < 1e-3, f"LoRMA+ is not identity at init ({d})"; print(f"[lorma+] identity at init OK ({d:.1e})")

NAME = f"second_lorma_{MODE}"
CJ[NAME] = train_cj(m, NAME, MODE, epochs=2, lr=1e-3, head_lr=1e-3, n_train=N_TRAIN, debug=DEBUG)
print("merge check (layer-0 q_proj):", f"{check_merge(m.lm.layers[0].attn.q_proj):.1e}")
if not DEBUG: write_pred(m, NAME, MODE)

[lorma+] identity at init OK (0.0e+00)
[second_lorma_last4] mode=last4 train_docs=37387 steps=18694 trainable=0.41M / 87.9M
  ~48 min training total (excl. dev evals)
  step 100/18694 loss 0.9239
  step 200/18694 loss 0.5449
  step 300/18694 loss 0.3481
  step 400/18694 loss 0.5743
  step 500/18694 loss 1.1290
  step 600/18694 loss 1.1017
  step 700/18694 loss 0.5649
  step 800/18694 loss 0.5585
  step 900/18694 loss 0.9718
  step 1000/18694 loss 0.4775
  step 1100/18694 loss 2.1971
  step 1200/18694 loss 0.4327
  step 1300/18694 loss 0.8576
  step 1400/18694 loss 0.5635
  step 1500/18694 loss 0.8952
  step 1600/18694 loss 0.8353
  step 1700/18694 loss 1.1087
  step 1800/18694 loss 0.8397
  step 1900/18694 loss 3.4518
  step 2000/18694 loss 1.0674
  step 2100/18694 loss 1.9202
  step 2200/18694 loss 1.3161
  step 2300/18694 loss 2.3367
  step 2400/18694 loss 0.7310
  step 2500/18694 loss 0.8242
  step 2600/18694 loss 1.0028
  step 2700/18694 loss 0.7466
  step 2800/18694 loss 1.1051
  

In [ ]:
   for k, r in CJ.items():
       if len(r["dev_logits"]) == len(C["dev"]["y"]): np.save(f"{P4_DIR}/dev_logits_{k}.npy", r["dev_logits"])

In [ ]:
SECOND_CKPT = f"{DRIVE_PROJECT_DIR}/alt_stride_32_128_checkpoints/best.pt"
assert os.path.exists(SECOND_CKPT), SECOND_CKPT
for n in ["CustomDecoderLM", "alternating_patterns", "cj_from", "train_cj", "write_pred", "C", "P4_DIR"]:
    assert n in globals(), f"missing {n}: run the earlier cells first"
assert "ht4" in MODES and "gqa_full" not in str(C.keys()), "this looks like the Phase-5 toolkit; restart the runtime first"
assert C["train"]["y"] is not None, "wrong cache: this must be the Phase-4 (CJPE) cache"
SECOND_PATTERNS = alternating_patterns(10, [(256, 32), (256, 128)])

def build_second_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=SECOND_PATTERNS)
    ck = torch.load(SECOND_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)       # pretrained strided Phase-1 weights
    return RRModel(lm_, 768, 13, dropout).to(device)

DEBUG   = False       # True first (~2 min), then False
MODE    = "last4"
CJ = globals().get("CJ", {})

random.seed(1); torch.manual_seed(1)
m = cj_from(build_second_rr())
NAME = f"second_full_{MODE}"
CJ[NAME] = train_cj(m, NAME, MODE, epochs=2, lr=5e-5, head_lr=1e-3, debug=DEBUG)
if not DEBUG: write_pred(m, NAME, MODE)

[second_full_last4] mode=last4 train_docs=37387 steps=18694 trainable=87.51M / 87.5M
  ~51 min training total (excl. dev evals)
  step 100/18694 loss 0.6777
  step 200/18694 loss 0.1096
  step 300/18694 loss 0.4050
  step 400/18694 loss 0.3901
  step 500/18694 loss 1.8374
  step 600/18694 loss 0.8716
  step 700/18694 loss 2.1099
  step 800/18694 loss 0.4010
  step 900/18694 loss 0.4469
  step 1000/18694 loss 1.1943
  step 1100/18694 loss 1.4758
  step 1200/18694 loss 0.1691
  step 1300/18694 loss 1.0172
  step 1400/18694 loss 0.8811
  step 1500/18694 loss 1.0941
  step 1600/18694 loss 0.3639
  step 1700/18694 loss 0.8603
  step 1800/18694 loss 1.5608
  step 1900/18694 loss 2.0903
  step 2000/18694 loss 0.9643
  step 2100/18694 loss 0.7814
  step 2200/18694 loss 1.1121
  step 2300/18694 loss 1.5388
  step 2400/18694 loss 0.5799
  step 2500/18694 loss 0.0376
  step 2600/18694 loss 0.6819
  step 2700/18694 loss 2.3585
  step 2800/18694 loss 0.2361
  step 2900/18694 loss 0.4216
  step 3000

## Legal Statute Identification

In [15]:
import json, os, pickle, time, numpy as np
from collections import Counter
from huggingface_hub import snapshot_download

P5_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_5"
os.makedirs(P5_DIR, exist_ok=True)

local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_5_LSI/*", "*/Phase_5_LSI/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_5" in r: return os.path.join(r, name)
tr, dv, te = [json.load(open(find(n))) for n in ("train.json", "dev.json", "test.json")]
print("sizes:", len(tr), len(dv), len(te))
print("train keys:", list(tr[0].keys()), "| test keys:", list(te[0].keys()))
print("example labels:", tr[0]["labels"][:10], "| text start:", tr[0]["text"][:100].replace("\n", " "))

# ---- label statistics ----
ctr, ctd = Counter(l for r in tr for l in r["labels"]), Counter(l for r in dv for l in r["labels"])
allids = sorted(set(ctr) | set(ctd))
print(f"\nstatute ids: train {len(ctr)} | dev {len(ctd)} | union {len(allids)} | min id {min(allids)} max id {max(allids)}")
print("dev ids unseen in train:", len(set(ctd) - set(ctr)), "| train ids absent in dev:", len(set(ctr) - set(ctd)))
fr = np.array(sorted(ctr.values(), reverse=True))
print("train positives per statute: max", fr[0], "median", int(np.median(fr)), "min", fr[-1])
print("statutes with <=5 train positives:", int((fr <= 5).sum()), "| <=20:", int((fr <= 20).sum()), "| >=500:", int((fr >= 500).sum()))
nl = np.array([len(r["labels"]) for r in tr])
print("labels per train doc: mean %.2f median %d max %d | docs with 1 label: %.1f%%" % (nl.mean(), np.median(nl), nl.max(), 100 * (nl == 1).mean()))
print("duplicate ids inside a doc:", sum(len(r["labels"]) != len(set(r["labels"])) for r in tr))
print("top 10 statutes (id, train count):", ctr.most_common(10))

# ---- tokenise once: keep first/last 4096 tokens (same scheme as Phase 4) ----
def tok_split(texts, N=4096):
    out = []
    for i in range(0, len(texts), 500):
        b = texts[i:i + 500]
        eh = tokenizer.encode_batch([t[:40000] for t in b])
        long_ = [j for j, t in enumerate(b) if len(t) > 80000]
        et = dict(zip(long_, tokenizer.encode_batch([b[j][-40000:] for j in long_]))) if long_ else {}
        for j in range(len(b)):
            hi = eh[j].ids; ti, total = (et[j].ids, 10**6) if j in et else (hi, len(hi))
            out.append((np.array(hi[:N], dtype=np.uint16), np.array(ti[-N:], dtype=np.uint16), total))
    return out

CACHE = f"{P5_DIR}/lsi_tok_cache.pkl"
if os.path.exists(CACHE):
    C = pickle.load(open(CACHE, "rb")); print("loaded cache")
else:
    t0 = time.time(); C = {}
    for name, d in [("train", tr), ("dev", dv), ("test", te)]:
        tk = tok_split([r["text"] for r in d])
        C[name] = dict(ids=[r["id"] for r in d], labels=[r.get("labels") for r in d],
                       head=[a for a, _, _ in tk], tail=[b for _, b, _ in tk], n=np.array([n for _, _, n in tk]))
        print(name, f"{(time.time() - t0) / 60:.1f} min")
    pickle.dump(C, open(CACHE, "wb"))
n = C["train"]["n"]
print("tokens per train doc: median", int(np.median(n)), "| p90", int(np.percentile(n, 90)),
      "| <=4096: %.1f%%" % (100 * (n <= 4096).mean()), "| <=8192: %.1f%%" % (100 * (n <= 8192).mean()))
print("✅ Phase-5 cache ready")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

sizes: 42750 10181 13019
train keys: ['id', 'text', 'labels'] | test keys: ['id', 'text']
example labels: [69, 9, 3] | text start: (a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this p

statute ids: train 100 | dev 100 | union 100 | min id 0 max id 99
dev ids unseen in train: 0 | train ids absent in dev: 0
train positives per statute: max 11731 median 679 min 272
statutes with <=5 train positives: 0 | <=20: 0 | >=500: 70
labels per train doc: mean 3.71 median 3 max 31 | docs with 1 label: 21.4%
duplicate ids inside a doc: 3
top 10 statutes (id, train count): [(5, 11731), (34, 10380), (97, 7881), (43, 7728), (39, 7121), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835)]
loaded cache
tokens per train doc: median 663 | p90 4286 | <=4096: 89.2% | <=8192: 97.3%
✅ Phase-5 cache ready


In [16]:
import json, time, math, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
assert "C" in globals() and "P5_DIR" in globals() and "RRModel" in globals(), "run the cache cell and the Phase-3 toolkit first"

N_LAB, CH = 100, 1024
MODES = {"first1": 1, "last1": 1, "first4": 4, "last4": 4, "ht4": 4}
GRID = np.arange(0.05, 0.80, 0.025)

def multi_hot(lists):
    Y = np.zeros((len(lists), N_LAB), np.float32)
    for i, l in enumerate(lists): Y[i, list(set(l))] = 1          # also removes duplicate ids
    return Y
YTR, YDV = multi_hot(C["train"]["labels"]), multi_hot(C["dev"]["labels"])

# ---------- document -> chunks (views over the cached head/tail tokens) ----------
def get_chunks(split, i, mode):
    d = C[split]; n = int(d["n"][i]); head, tail = d["head"][i], d["tail"][i]
    if   mode == "first1": x = head[:CH]
    elif mode == "last1":  x = tail[-CH:]
    elif mode == "first4": x = head[:4*CH]
    elif mode == "last4":  x = tail[-4*CH:]
    elif mode == "ht4":    x = head if n <= 4*CH else np.concatenate([head[:2*CH], tail[-2*CH:]])
    L = len(x)
    if mode.startswith("last"):
        ends = list(range(L, 0, -CH))[::-1]; return [x[max(0, e-CH):e] for e in ends]
    return [x[s:s+CH] for s in range(0, L, CH)]

_CNT = {}
def chunk_counts(split, mode):
    if (split, mode) not in _CNT:
        _CNT[(split, mode)] = np.array([max(1, len(get_chunks(split, i, mode))) for i in range(len(C[split]["ids"]))])
    return _CNT[(split, mode)]

def make_batches(split, idxs, mode, max_chunks=16, rng=None, mega=64):
    """Batches of <= max_chunks chunks; documents of similar length are grouped to avoid padding waste."""
    cnt = chunk_counts(split, mode); n = np.minimum(C[split]["n"], 4*CH); idxs = np.asarray(idxs)
    if rng is not None:
        idxs = idxs[rng.permutation(len(idxs))]
        groups = [idxs[s:s+max_chunks*mega] for s in range(0, len(idxs), max_chunks*mega)]
        groups = [g[np.argsort(n[g], kind="stable")] for g in groups]
    else:
        groups = [idxs[np.argsort(n[idxs], kind="stable")]]
    out = []
    for g in groups:
        cur, c = [], 0
        for i in g:
            if cur and c + cnt[i] > max_chunks: out.append(np.array(cur)); cur, c = [], 0
            cur.append(i); c += cnt[i]
        if cur: out.append(np.array(cur))
    if rng is not None: out = [out[j] for j in rng.permutation(len(out))]
    return out

def collate_ls(split, idxs, mode):
    seqs, owner = [], []
    for j, i in enumerate(idxs):
        cs = get_chunks(split, i, mode)
        if not cs or len(cs[0]) == 0: cs = [np.zeros(1, np.uint16)]
        for c in cs: seqs.append(c); owner.append(j)
    T = max(len(s) for s in seqs)
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.from_numpy(s.astype(np.int64))
    lens = torch.tensor([len(s) for s in seqs])
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(idxs)

# ---------- model: Phase-1 LM -> chunk features -> per-chunk statute logits -> document logits ----------
class LSIModel(RRModel):
    agg = "mean"                                   # "mean" or "max" over chunks
    def forward(self, ids, lens, owner, n_docs):
        H = self.hidden(ids); B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        mean = (H.float() * mask).sum(1) / lens[:, None]
        last = H[torch.arange(B, device=H.device), lens - 1].float()
        lg = self.head(self.drop(torch.cat([last, mean], -1))).float()          # (chunks, N_LAB)
        if self.agg == "max":
            return torch.full((n_docs, lg.size(1)), -1e4, device=lg.device).scatter_reduce(
                0, owner[:, None].expand_as(lg), lg, "amax", include_self=False)
        D = torch.zeros(n_docs, lg.size(1), device=lg.device).index_add_(0, owner, lg)
        return D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]

def ls_from(rr):
    """Phase-3 RRModel -> LSI model: same LM, fresh 100-way head (bias = log prior odds of each statute)."""
    d = rr.lm.token_embedding.weight.shape[1]
    m = LSIModel(rr.lm, d, N_LAB, 0.1).to(device)
    m.head = nn.Linear(2 * d, N_LAB).to(device)
    nn.init.normal_(m.head.weight, std=0.02)
    p = YTR.mean(0).clip(1e-4, 1 - 1e-4)
    with torch.no_grad(): m.head.bias.copy_(torch.tensor(np.log(p / (1 - p)), dtype=torch.float))
    return m

# ---------- inference, decoding, metrics ----------
@torch.no_grad()
def ls_logits(model, split, mode, idxs=None, max_chunks=32):
    model.eval(); N = len(C[split]["ids"]); idxs = np.arange(N) if idxs is None else np.asarray(idxs)
    out = np.zeros((N, N_LAB), np.float32)
    for b in make_batches(split, idxs, mode, max_chunks, rng=None):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out[b] = model(*collate_ls(split, b, mode)).float().cpu().numpy()
    return out[idxs]

def sigmoid(z): return 1 / (1 + np.exp(-z))

def decode(P, thr, force_one=True):
    pred = P >= thr
    if force_one:
        none = ~pred.any(1); pred[none, P[none].argmax(1)] = True      # every document has at least one statute
    return pred

def ls_scores(Y, pred):
    Yb = Y.astype(bool); tp = (pred & Yb).sum(0); fp = (pred & ~Yb).sum(0); fn = (~pred & Yb).sum(0)
    f = np.where(2*tp + fp + fn > 0, 2*tp / np.maximum(2*tp + fp + fn, 1), 0.0)
    p = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0); r = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    return dict(macro=float(f.mean()), p=float(p.mean()), r=float(r.mean()),
                micro=float(2*tp.sum() / max(2*tp.sum() + fp.sum() + fn.sum(), 1)), subset=float((pred == Yb).all(1).mean()), per_label=f)

def best_global(Y, P):
    best = (0.5, -1.0)
    for t in GRID:
        f = ls_scores(Y, decode(P, t))["macro"]
        if f > best[1]: best = (float(t), f)
    return best

def best_per_label(Y, P):
    Yb = Y.astype(bool); thr = np.full(N_LAB, 0.5)
    for j in range(N_LAB):
        pj = P[:, j][:, None] >= GRID[None, :]; yj = Yb[:, j][:, None]
        tp = (pj & yj).sum(0); fp = (pj & ~yj).sum(0); fn = (~pj & yj).sum(0)
        thr[j] = GRID[(2*tp / np.maximum(2*tp + fp + fn, 1)).argmax()]
    return thr

def honest_perlabel(Y, P):
    """Per-label thresholds tuned on one half of dev and scored on the other half (and vice versa)."""
    idx = np.arange(len(Y)); A, B = idx[::2], idx[1::2]; out = []
    for a, b in [(A, B), (B, A)]:
        out.append(ls_scores(Y[b], decode(P[b], best_per_label(Y[a], P[a])))["macro"])
    return float(np.mean(out))

# ---------- training (BCE / weighted BCE, optional KD from cached teacher logits) ----------
def train_ls(model, name, mode, epochs=2, lr=5e-5, head_lr=1e-3, wd=0.01, max_chunks=16, n_train=None,
             loss_type="wbce", kd=None, alpha=0.5, T=2.0, evals_per_epoch=2, patience=3, debug=False, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed); ntr = len(YTR)
    dev_idx = np.arange(300) if debug else np.arange(len(YDV)); Ydev = YDV[dev_idx]
    tr_idx = rng.permutation(ntr)[: (160 if debug else (n_train or ntr))]
    if debug: epochs, evals_per_epoch = 1, 1
    pw = None
    if loss_type == "wbce":
        pos = YTR.sum(0); pw = torch.tensor(np.clip(np.sqrt((ntr - pos) / pos), 1, 10), dtype=torch.float, device=device)
    ytr_t = torch.tensor(YTR); kd_t = torch.tensor(kd, dtype=torch.float) if kd is not None else None
    head_p = list(model.head.parameters()); hid = {id(p) for p in head_p}
    body_p = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body_p, "lr": lr}, {"params": head_p, "lr": head_lr}], weight_decay=wd)
    spe = len(make_batches("train", tr_idx, mode, max_chunks, np.random.RandomState(seed))); total = spe * epochs
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    every = max(1, spe // evals_per_epoch)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{name}] mode={mode} loss={loss_type} train_docs={len(tr_idx)} steps={total} trainable={n_tr/1e6:.2f}M / {n_all/1e6:.1f}M")

    path = f"{P5_DIR}/ls_{name}_best.pt"; best = (-1.0, None); bad = 0; step = 0; t0 = time.time(); done = False
    torch.cuda.reset_peak_memory_stats()
    for ep in range(epochs):
        model.train()
        for b in make_batches("train", tr_idx, mode, max_chunks, rng):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(*collate_ls("train", b, mode)).float()
            y = ytr_t[b].to(device)
            loss = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
            if kd_t is not None:
                t_lg = kd_t[b].to(device)
                kdl = F.binary_cross_entropy_with_logits(lg / T, torch.sigmoid(t_lg / T)) * T * T
                loss = (1 - alpha) * loss + alpha * kdl
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); step += 1
            if step == 20: print(f"  ~{(time.time()-t0)/20*total/60:.0f} min training total (excl. dev evals)")
            if step % 200 == 0: print(f"  step {step}/{total} loss {loss.item():.4f}")
            if step % every == 0 or step == total:
                P = sigmoid(ls_logits(model, "dev", mode, dev_idx)); thr, f = best_global(Ydev, P); model.train()
                s5 = ls_scores(Ydev, decode(P, 0.5))
                print(f"  eval@{step}: macro-F1 @0.5 {s5['macro']:.4f} | @best global thr {thr:.3f}: {f:.4f}  ({(time.time()-t0)/60:.1f} min)")
                if f > best[0]:
                    best = (f, step); bad = 0
                    torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, path)
                else:
                    bad += 1
                    if bad >= patience: done = True; break
        if done: break
    if os.path.exists(path): model.load_state_dict(torch.load(path), strict=False)
    P = sigmoid(ls_logits(model, "dev", mode, dev_idx)); thr, f = best_global(Ydev, P)
    sg = ls_scores(Ydev, decode(P, thr)); tl = best_per_label(Ydev, P); sl = ls_scores(Ydev, decode(P, tl))
    hon = honest_perlabel(Ydev, P)
    res = dict(name=name, mode=mode, thr_global=thr, thr_label=tl, f1_at_05=ls_scores(Ydev, decode(P, 0.5))["macro"],
               macro_global=sg["macro"], p=sg["p"], r=sg["r"], micro=sg["micro"], subset=sg["subset"],
               macro_perlabel_optimistic=sl["macro"], macro_perlabel_honest=hon, per_label=sg["per_label"],
               minutes=(time.time()-t0)/60, peak_gb=torch.cuda.max_memory_allocated()/1e9, trainable=n_tr, total=n_all, dev_probs=P)
    print(f"✅ [{name}] dev macro-F1: @0.5 {res['f1_at_05']:.4f} | global thr {thr:.3f} → {sg['macro']:.4f} "
          f"(P {sg['p']:.4f} R {sg['r']:.4f}, micro {sg['micro']:.4f}, subset acc {sg['subset']:.4f})\n"
          f"   per-label thresholds: {sl['macro']:.4f} on the tuning data, {hon:.4f} on held-out dev halves "
          f"| {res['minutes']:.1f} min, peak {res['peak_gb']:.1f} GB")
    return res

# ---------- outputs ----------
def write_pred_ls(model, name, mode, res):
    L = ls_logits(model, "test", mode); P = sigmoid(L)
    np.save(f"{P5_DIR}/test_probs_{name}.npy", P); np.save(f"{P5_DIR}/dev_probs_{name}.npy", res["dev_probs"])
    for tag, thr in [("global", res["thr_global"]), ("perlabel", res["thr_label"])]:
        pred = decode(P, thr); ids = C["test"]["ids"]
        out = [{"id": i, "pred": [int(k) for k in np.flatnonzero(pr)]} for i, pr in zip(ids, pred)]
        assert len(out) == len(ids) == 13019 and all(0 < len(o["pred"]) == len(set(o["pred"])) and max(o["pred"]) < N_LAB for o in out)
        f = f"{P5_DIR}/prediction_{name}_{tag}.json"; json.dump(out, open(f, "w"))
        print(f"wrote {f} | mean statutes/doc {pred.sum(1).mean():.2f}")

def save_kd_ls(model, name, mode):
    L = ls_logits(model, "train", mode); np.save(f"{P5_DIR}/kd_logits_{name}.npy", L); return L

print("✅ Phase-5 toolkit ready")

✅ Phase-5 toolkit ready


In [ ]:
SECOND_CKPT = f"{DRIVE_PROJECT_DIR}/alt_stride_32_128_checkpoints/best.pt"
assert os.path.exists(SECOND_CKPT), SECOND_CKPT
for n in ["CustomDecoderLM", "alternating_patterns", "ls_from", "train_ls", "collate_ls", "C", "P5_DIR"]:
    assert n in globals(), f"missing {n}: run the earlier cells first"
SECOND_PATTERNS = alternating_patterns(10, [(256, 32), (256, 128)])

def build_second_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=SECOND_PATTERNS)
    ck = torch.load(SECOND_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)       # pretrained strided Phase-1 weights
    return RRModel(lm_, 768, 13, dropout).to(device)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0, acting on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_lorma(m, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in m.parameters(): p.requires_grad_(False)
    for layer in m.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, LoRMAPlusLinear(base, r, alpha).to(base.weight.device))
    for p in m.head.parameters(): p.requires_grad_(True)

DEBUG   = False          # True first (~1-2 min); then False
MODE    = "ht4"
N_TRAIN = None
LS = globals().get("LS", {})

random.seed(1); torch.manual_seed(1)
m = ls_from(build_second_rr())
batch = collate_ls("dev", [0, 1], "last1"); m.eval()
with torch.no_grad(): before = m(*batch)
inject_lorma(m, 16, 32)
with torch.no_grad(): d = (before - m(*batch)).abs().max().item()
assert d < 1e-3, f"LoRMA+ is not identity at init ({d})"; print(f"[lorma+] identity at init OK ({d:.1e})")

NAME = f"second_lorma_{MODE}"
LS[NAME] = train_ls(m, NAME, MODE, epochs=2, lr=1e-3, head_lr=1e-3, n_train=N_TRAIN, loss_type="wbce", debug=DEBUG)
if not DEBUG: write_pred_ls(m, NAME, MODE, LS[NAME])

[lorma+] identity at init OK (0.0e+00)
[second_lorma_ht4] mode=ht4 loss=wbce train_docs=42750 steps=9878 trainable=0.81M / 88.3M
  ~28 min training total (excl. dev evals)
  step 200/9878 loss 0.4966
  step 400/9878 loss 0.3664
  step 600/9878 loss 0.4870
  step 800/9878 loss 0.4915
  step 1000/9878 loss 0.1755
  step 1200/9878 loss 0.6833
  step 1400/9878 loss 0.5641
  step 1600/9878 loss 0.5320
  step 1800/9878 loss 0.4957
  step 2000/9878 loss 0.3361
  step 2200/9878 loss 0.4762
  step 2400/9878 loss 0.5089
  eval@2469: macro-F1 @0.5 0.0787 | @best global thr 0.225: 0.1048  (8.7 min)
  step 2600/9878 loss 0.5075
  step 2800/9878 loss 0.5082
  step 3000/9878 loss 0.2442
  step 3200/9878 loss 0.5620
  step 3400/9878 loss 0.8194
  step 3600/9878 loss 0.6880
  step 3800/9878 loss 0.3621
  step 4000/9878 loss 0.6557
  step 4200/9878 loss 0.5457
  step 4400/9878 loss 0.5144
  step 4600/9878 loss 0.4136
  step 4800/9878 loss 0.2083
  eval@4938: macro-F1 @0.5 0.0360 | @best global thr 0.125

## Abstractive Summarization


In [17]:
import os, json, pickle, numpy as np
from huggingface_hub import snapshot_download
P6_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_6"; os.makedirs(P6_DIR, exist_ok=True)
CACHE6 = f"{P6_DIR}/sum_tok_cache.pkl"
EOS6 = tokenizer.token_to_id("<|endoftext|>")

def doc_view(rec, n_head, n_tail=0):
    """Token view of a document: first n_head tokens, plus last n_tail tokens if the doc is longer."""
    n, h, t = rec["n"], rec["head"], rec["tail"]
    full = h if n <= 4096 else (np.concatenate([h, t[-(n - 4096):]]) if n <= 6144 else None)
    if full is not None:
        return full if n <= n_head + n_tail else np.concatenate([full[:n_head], full[n - n_tail:]] if n_tail else [full[:n_head]])
    return np.concatenate([h[:n_head], t[-n_tail:]] if n_tail else [h[:n_head]])

if os.path.exists(CACHE6):
    D6 = pickle.load(open(CACHE6, "rb")); print("loaded cache from Drive")
else:
    local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                              allow_patterns=["Phase_6_Summ/*", "*/Phase_6_Summ/*"], token=HF_TOKEN)
    def find(name):
        for r, _, fs in os.walk(local):
            if name in fs and "Phase_6" in r: return os.path.join(r, name)
    tr = [r for r in json.load(open(find("train.json"))) if r["summary"].strip()]
    te = json.load(open(find("test.json")))
    def enc(texts, bs=128):
        out = []
        for i in range(0, len(texts), bs):
            out += [e.ids for e in tokenizer.encode_batch([t[:400000] for t in texts[i:i + bs]])]
        return out
    def pack(ids): return dict(n=len(ids), head=np.array(ids[:4096], np.uint16), tail=np.array(ids[-2048:], np.uint16))
    tr_docs, tr_sums = enc([r["document"] for r in tr]), enc([r["summary"] for r in tr])
    te_docs = enc([r["document"] for r in te])
    recs = [dict(id=r["id"], **pack(d), s_len=len(s), summ=np.array(s[:1024], np.uint16)) for r, d, s in zip(tr, tr_docs, tr_sums)]
    perm = np.random.RandomState(0).permutation(len(recs))
    D6 = dict(val=[recs[i] for i in perm[:300]], train=[recs[i] for i in perm[300:]],
              test=[dict(id=r["id"], **pack(d)) for r, d in zip(te, te_docs)])
    pickle.dump(D6, open(CACHE6, "wb")); print("saved cache to Drive")

print("train:", len(D6["train"]), "| val:", len(D6["val"]), "| test:", len(D6["test"]), "| EOS id:", EOS6)
r = D6["train"][0]
print("view check: head-only 1536 ->", len(doc_view(r, 1536)), "| head 1024 + tail 512 ->", len(doc_view(r, 1024, 512)), "| doc length", r["n"])

loaded cache from Drive
train: 6728 | val: 300 | test: 100 | EOS id: 0
view check: head-only 1536 -> 1536 | head 1024 + tail 512 -> 1536 | doc length 3103


In [18]:
import math, os, torch, torch.nn as nn, torch.nn.functional as F
KIND = "strided"          # <-- set per notebook: gqa | strided | mla | encdec | student
assert "RRModel" in globals() and "tokenizer" in globals() and "device" in globals(), "run setup + Phase-3 toolkit first"
P1 = DRIVE_PROJECT_DIR                                    # Phase-1 folder (from the setup cell)
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"

def _load(lm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("model_state_dict", ck)) if isinstance(ck, dict) else ck
    lm.load_state_dict({k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()})
    return lm

def build_lm(kind):
    V = tokenizer.get_vocab_size()
    if kind == "gqa":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048, tie_weights=True),
                     f"{P1}/gqa_checkpoints/gqa_best.pt"), 768
    if kind == "strided":
        pat = alternating_patterns(10, [(256, 32), (256, 128)])
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True, layer_patterns=pat), f"{P1}/alt_stride_32_128_checkpoints/best.pt"), 768
    if kind == "mla":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, max_seq_len=4096, qk_rope_head_dim=64,
                                     tie_weights=True), f"{P1}/DeepSeek/deepseek_mla_85m_pretrained.pt"), 768
    if kind == "encdec":
        return _load(EncoderDecoderLM(vocab_size=V, d_model=640, n_enc_layer=6, n_dec_layer=6, n_head=10, n_kv_head=5,
                                      max_src_len=1024, max_tgt_len=320, bos_id=tokenizer.token_to_id("<|endoftext|>"),
                                      tie_weights=True), f"{P1}/encdec_6x6_checkpoints/best.pt"), 640
    if kind == "student":
        return _load(CustomDecoderLM(vocab_size=V, d_model=576, n_layer=7, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True), STAGE1_CKPT), 576
    raise ValueError(kind)

class Backbone(RRModel):
    """Phase-1 LM + a features() method. Task heads are attached by each phase (model.head = ...)."""
    def __init__(self, lm, d, kind):
        super().__init__(lm, d, 2, 0.1); del self.head
        self.kind, self.d = kind, d
    def features(self, ids, lens):                       # ids (chunks, T) right-padded, lens (chunks,) -> (chunks, 2d)
        if self.kind == "encdec":                        # encoder takes no mask: one chunk at a time
            out = []
            for j in range(ids.shape[0]):
                n = int(lens[j]); H = self.lm.encode(ids[j:j + 1, :n])[0].float()
                out.append(torch.cat([H[n - 1], H.mean(0)], -1))
            return torch.stack(out)
        H = self.hidden(ids); B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        return torch.cat([H[torch.arange(B, device=H.device), lens - 1].float(), (H.float() * mask).sum(1) / lens[:, None]], -1)

def make_backbone(kind=None):
    lm, d = build_lm(kind or KIND); return Backbone(lm, d, kind or KIND).to(device)

# ---------- adapters (own implementations) ----------
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0 on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

TARGETS = {k: ("q_proj", "k_proj", "v_proj", "out_proj") for k in ["gqa", "strided", "encdec", "student"]}
TARGETS["mla"] = ("q_up_proj", "kv_down_proj", "kv_up_proj", "out_proj")
def layers_of(bb): return bb.lm.enc_layers if bb.kind == "encdec" else bb.lm.layers

def inject(bb, method, r, alpha):
    """Freeze the Phase-1 LM and add adapters. Heads attached outside bb.lm stay trainable."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    cls = {"lora": LoRALinear, "lorma": LoRMAPlusLinear}[method]
    for layer in layers_of(bb):
        for nm in TARGETS[bb.kind]:
            base = getattr(layer.attn, nm); setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    return bb

def unfreeze_top(bb, k):
    """Partial (layer-wise) fine-tuning: only the top k layers (+ final norm for decoders) train."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    for layer in list(layers_of(bb))[-k:]:
        for p in layer.parameters(): p.requires_grad_(True)
    if bb.kind != "encdec":
        for p in bb.lm.norm.parameters(): p.requires_grad_(True)
    return bb

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad), sum(p.numel() for p in m.parameters())

# ---------- distillation losses ----------
def kd_multiclass(s, t, T=2.0): return F.kl_div(F.log_softmax(s / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
def kd_multilabel(s, t, T=2.0): return F.binary_cross_entropy_with_logits(s / T, torch.sigmoid(t / T)) * T * T

# ---------- self-test: shapes + adapters are the identity at init ----------
CORE_SELFTEST = True
if CORE_SELFTEST:
    ids = torch.randint(0, tokenizer.get_vocab_size(), (2, 64), device=device); lens = torch.tensor([50, 64], device=device)
    for method in ["lorma", "lora"]:
        bb = make_backbone().eval()
        with torch.no_grad(): f0 = bb.features(ids, lens)
        inject(bb, method, 8, 16)
        with torch.no_grad(): f1 = bb.features(ids, lens)
        assert f0.shape == (2, 2 * bb.d) and (f0 - f1).abs().max().item() < 1e-3, f"{method}: not identity at init"
        tr, tot = n_params(bb); print(f"[{KIND}] {method}: identity OK | trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M (adapters only, no head)")
        del bb; torch.cuda.empty_cache()
    bb = unfreeze_top(make_backbone(), 4); tr, tot = n_params(bb); print(f"[{KIND}] partial top-4: trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M"); del bb
    print(f"✅ core ready for KIND={KIND}")

[strided] lorma: identity OK | trainable 0.33M of 87.8M (adapters only, no head)
[strided] lora: identity OK | trainable 0.41M of 87.9M (adapters only, no head)
[strided] partial top-4: trainable 25.17M of 87.5M
✅ core ready for KIND=strided


In [19]:
import inspect, torch
bb = make_backbone(); lm = bb.lm; cls = type(lm)
print("class:", cls.__name__, "| KIND:", KIND)
print("public methods:", [m for m in dir(cls) if not m.startswith("_") and callable(getattr(cls, m))])
for nm in ["forward", "generate", "encode", "decode"]:
    f = getattr(cls, nm, None)
    if f is None: continue
    print(f"\n{nm}{inspect.signature(f)}\n  names used: {f.__code__.co_names}")
ids = torch.randint(0, 100, (1, 16), device=device)
with torch.no_grad():
    try:
        o = lm(ids); parts = o if isinstance(o, tuple) else (o,)
        print("\nlm(ids) ->", type(o).__name__, [getattr(t, "shape", type(t).__name__) for t in parts])
        if isinstance(o, tuple) and len(o) > 1:
            print("2nd output:", type(o[1]).__name__, "len", len(o[1]) if hasattr(o[1], "__len__") else "-")
            o2 = lm(ids[:, -1:], past_key_values=o[1])
            print("cached step ->", [getattr(t, "shape", type(t).__name__) for t in (o2 if isinstance(o2, tuple) else (o2,))])
    except Exception as e:
        print("forward probe failed:", repr(e)[:400])

class: CustomDecoderLM | KIND: strided
public methods: ['add_module', 'apply', 'bfloat16', 'buffers', 'children', 'compile', 'cpu', 'cuda', 'describe_patterns', 'double', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'ipu', 'load_state_dict', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'set_patterns', 'set_submodule', 'share_memory', 'state_dict', 'to', 'to_empty', 'train', 'type', 'xpu', 'zero_grad']

forward(self, input_ids, labels=None, past_key_values=None)
  names used: ('sha

In [ ]:
import torch, torch.nn.functional as F
bb = make_backbone().eval(); lm = bb.lm
V = tokenizer.get_vocab_size(); ids = torch.randint(0, V, (1, 40), device=device)
pick_lg = lambda o: next(v for v in o.values() if torch.is_tensor(v) and v.dim() == 3)
with torch.no_grad():
    o = lm(ids); print("keys:", {k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k, v in o.items()})
    full = pick_lg(o)
    pk_key = next((k for k, v in o.items() if "past" in k or "cache" in k or isinstance(v, (tuple, list))), None)
    print("cache key:", pk_key)
    o1 = lm(ids[:, :39]); o2 = lm(ids[:, 39:40], past_key_values=o1[pk_key])
    print("cached step vs full forward, max logit diff: %.2e" % (pick_lg(o2)[:, -1] - full[:, 39]).abs().max().item())
    lab = ids.clone(); lab[:, :20] = -100
    L = lm(ids, labels=lab); loss = next(v for k, v in L.items() if "loss" in k)
    lg = pick_lg(L)
    man = F.cross_entropy(lg[:, :-1].reshape(-1, V).float(), lab[:, 1:].reshape(-1), ignore_index=-100)
    print("model loss %.4f | manual shifted CE %.4f" % (loss.item(), man.item()))

keys: {'loss': 'NoneType', 'logits': (1, 40, 32000), 'past_key_values': 'tuple'}
cache key: past_key_values
cached step vs full forward, max logit diff: 7.63e-06
model loss 12.3433 | manual shifted CE 12.3433


In [20]:
import os, time, json, numpy as np, torch
try: from rouge_score import rouge_scorer
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge_score"]); from rouge_score import rouge_scorer
assert KIND != "encdec", "decoder cell: enc-dec gets its own"
MAXLEN = 4096 if KIND == "mla" else 2048
CFG = dict(doc_head=2048, doc_tail=1024, sum_max=960) if KIND == "mla" else dict(doc_head=1024, doc_tail=480, sum_max=512)
BS, ACC = (4, 2) if KIND == "mla" else (8, 1)
SEP = tokenizer.encode("\n\nSUMMARY:\n").ids
assert len(SEP) + CFG["doc_head"] + CFG["doc_tail"] + CFG["sum_max"] + 1 <= MAXLEN
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
R6 = globals().get("R6", {})

def make_prompt(r): return doc_view(r, CFG["doc_head"], CFG["doc_tail"]).astype(np.int64).tolist() + SEP
def make_ex(r):
    s = r["summ"][:CFG["sum_max"]].astype(np.int64).tolist()
    if r["s_len"] <= CFG["sum_max"]: s.append(EOS6)          # teach stopping only when the whole summary fit
    return make_prompt(r), s
def batchify(exs):
    T = max(len(p) + len(s) for p, s in exs)
    ids = torch.zeros(len(exs), T, dtype=torch.long); lab = torch.full((len(exs), T), -100, dtype=torch.long)
    for i, (p, s) in enumerate(exs):
        x = p + s; ids[i, :len(x)] = torch.tensor(x); lab[i, len(p):len(x)] = torch.tensor(s)
    return ids.to(device), lab.to(device)

@torch.no_grad()
def val_loss(bb, n=None):
    bb.eval(); vs = D6["val"][:n] if n else D6["val"]; out = []
    for i in range(0, len(vs), BS):
        ids, lab = batchify([make_ex(r) for r in vs[i:i + BS]])
        with torch.autocast("cuda", dtype=torch.bfloat16): out.append(bb.lm(ids, labels=lab)["loss"].item())
    bb.train(); return float(np.mean(out))

@torch.no_grad()
def generate(bb, prompt, max_new, nrep=4):
    lm = bb.lm.eval(); out, seen = [], {}
    with torch.autocast("cuda", dtype=torch.bfloat16):
        o = lm(torch.tensor([prompt], device=device)); pk = o["past_key_values"]; lg = o["logits"][0, -1].float()
        for _ in range(max_new):
            key = tuple(out[-(nrep - 1):]) if nrep and len(out) >= nrep - 1 else None
            if key is not None and key in seen: lg[list(seen[key])] = -float("inf")
            t = int(lg.argmax())
            if t == EOS6: break
            if key is not None: seen.setdefault(key, set()).add(t)
            out.append(t)
            o = lm(torch.tensor([[t]], device=device), past_key_values=pk); pk = o["past_key_values"]; lg = o["logits"][0, -1].float()
    return out

def rouge_eval(bb, n=40, max_new=None, show=0):
    r1, rl, wl = [], [], []
    for j, r in enumerate(D6["val"][:n]):
        hyp = tokenizer.decode(generate(bb, make_prompt(r), max_new or CFG["sum_max"]))
        ref = tokenizer.decode(r["summ"].astype(np.int64).tolist())          # reference truncated at 1024 tokens (proxy)
        sc = scorer.score(ref, hyp); r1.append(sc["rouge1"].fmeasure); rl.append(sc["rougeL"].fmeasure); wl.append(len(hyp.split()))
        if j < show: print(f"\n--- val doc {j} ---\nHYP: {hyp[:600]}\nREF: {ref[:600]}")
    return dict(rouge1=float(np.mean(r1)), rougeL=float(np.mean(rl)), words=float(np.mean(wl)))

def train_sum(bb, name, lr, epochs=3, wd=0.01, patience=3, debug=False):
    params = [p for p in bb.parameters() if p.requires_grad]; assert params, "nothing trainable"
    tr = D6["train"][:64] if debug else D6["train"]; epochs = 1 if debug else epochs
    steps = epochs * (len(tr) // (BS * ACC)); warm = max(1, int(0.06 * steps)); ev = max(1, steps // (2 * epochs))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    path = f"{P6_DIR}/sm_{name}_best.pt"; best, bad, step, run, t0, stop = 1e9, 0, 0, [], time.time(), False
    ntr, ntot = n_params(bb); print(f"[{name}] trainable {ntr/1e6:.2f}M / {ntot/1e6:.1f}M | steps {steps} | eval every {ev}")
    torch.cuda.reset_peak_memory_stats(); bb.train()
    for ep in range(epochs):
        order = np.random.RandomState(ep).permutation(len(tr)); k = 0
        while k + BS * ACC <= len(tr) and step < steps and not stop:
            for _ in range(ACC):
                ids, lab = batchify([make_ex(tr[i]) for i in order[k:k + BS]]); k += BS
                with torch.autocast("cuda", dtype=torch.bfloat16): loss = bb.lm(ids, labels=lab)["loss"] / ACC
                loss.backward(); run.append(loss.item() * ACC)
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % ev == 0 or step == steps:
                vl = val_loss(bb, 16 if debug else None)
                print(f"step {step}/{steps} | train {np.mean(run):.3f} | val {vl:.3f} | {(time.time()-t0)/60:.1f} min"); run = []
                if vl < best:
                    best, bad = vl, 0; torch.save({n: p.detach().cpu() for n, p in bb.named_parameters() if p.requires_grad}, path)
                else:
                    bad += 1; stop = bad >= patience
    bb.load_state_dict(torch.load(path), strict=False)
    m = rouge_eval(bb, n=6 if debug else 40, show=1 if debug else 0)
    R6[name] = dict(val_loss=best, **m, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best val loss {best:.3f} | ROUGE-1 {m['rouge1']:.4f} | ROUGE-L {m['rougeL']:.4f} | avg words {m['words']:.0f} | peak {R6[name]['peak_gb']:.1f} GB")
    return bb

def write_pred_sum(bb, name, max_new=None):
    preds = []; t0 = time.time()
    for i, r in enumerate(D6["test"]):
        h = tokenizer.decode(generate(bb, make_prompt(r), max_new or CFG["sum_max"])).strip()
        preds.append(dict(id=r["id"], summary=h or tokenizer.decode(doc_view(r, 150).astype(np.int64).tolist())))
        if (i + 1) % 25 == 0: print(f"{i+1}/100 | {(time.time()-t0)/60:.1f} min")
    assert len(preds) == 100 and len({p["id"] for p in preds}) == 100
    p = f"{P6_DIR}/prediction_{name}.json"; json.dump(preds, open(p, "w")); print("✅ wrote", p, "-> rename to prediction.json, zip, upload")
print("✅ summarisation toolkit ready | KIND =", KIND, "| context", MAXLEN, "|", CFG, "| BS", BS, "x ACC", ACC)

✅ summarisation toolkit ready | KIND = strided | context 2048 | {'doc_head': 1024, 'doc_tail': 480, 'sum_max': 512} | BS 8 x ACC 1


In [ ]:
bb = make_backbone(); inject(bb, "lora", 64, 128)
train_sum(bb, "dbg_strided", lr=5e-4, debug=False)
del bb; torch.cuda.empty_cache()

[dbg_strided] trainable 3.28M / 90.8M | steps 2523 | eval every 420
step 420/2523 | train 2.532 | val 2.471 | 2.7 min
step 840/2523 | train 2.488 | val 2.451 | 5.5 min
step 1260/2523 | train 2.420 | val 2.451 | 8.2 min
step 1680/2523 | train 2.423 | val 2.443 | 11.0 min
step 2100/2523 | train 2.343 | val 2.449 | 13.7 min
step 2520/2523 | train 2.355 | val 2.446 | 16.5 min
step 2523/2523 | train 2.281 | val 2.447 | 16.6 min

✅ [dbg_strided] best val loss 2.443 | ROUGE-1 0.3827 | ROUGE-L 0.2124 | avg words 359 | peak 35.6 GB


In [ ]:
bb = make_backbone(); inject(bb, "lora", 64, 128); bb = train_sum(bb, "strided_lora", lr=5e-4, epochs=2, debug=False)
write_pred_sum(bb, "strided_lora"); del bb; torch.cuda.empty_cache()

[strided_lora] trainable 3.28M / 90.8M | steps 1682 | eval every 420
step 420/1682 | train 2.527 | val 2.468 | 2.7 min
step 840/1682 | train 2.486 | val 2.448 | 5.5 min
step 1260/1682 | train 2.416 | val 2.446 | 8.2 min
step 1680/1682 | train 2.416 | val 2.440 | 11.0 min
step 1682/1682 | train 2.508 | val 2.440 | 11.1 min

✅ [strided_lora] best val loss 2.440 | ROUGE-1 0.4006 | ROUGE-L 0.2197 | avg words 331 | peak 35.6 GB
25/100 | 3.7 min
50/100 | 7.3 min
75/100 | 11.3 min
100/100 | 15.2 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_strided_lora.json -> rename to prediction.json, zip, upload


In [ ]:
bb = make_backbone(); inject(bb, "lorma", 64, 128); bb = train_sum(bb, "strided_lorma", lr=5e-4, epochs=2, debug=False)
write_pred_sum(bb, "strided_lorma"); del bb; torch.cuda.empty_cache()

[strided_lorma] trainable 2.62M / 90.1M | steps 1682 | eval every 420
step 420/1682 | train 2.531 | val 2.475 | 2.7 min
step 840/1682 | train 2.493 | val 2.456 | 5.4 min
step 1260/1682 | train 2.440 | val 2.452 | 8.1 min
step 1680/1682 | train 2.441 | val 2.447 | 10.8 min
step 1682/1682 | train 2.534 | val 2.447 | 10.9 min

✅ [strided_lorma] best val loss 2.447 | ROUGE-1 0.3829 | ROUGE-L 0.2124 | avg words 327 | peak 35.3 GB
25/100 | 3.8 min
50/100 | 7.3 min
75/100 | 10.9 min
100/100 | 14.5 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_strided_lorma.json -> rename to prediction.json, zip, upload


## Prior Case Retrieval

In [21]:
import json, os
from huggingface_hub import snapshot_download
P7_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_7"; os.makedirs(P7_DIR, exist_ok=True)
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_7_PCR/*", "*/Phase_7_PCR/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_7" in r: return os.path.join(r, name)
files = {n: find(n + ".json") for n in ["train", "train_candidates", "dev", "dev_candidates", "test", "test_candidates"]}
if not any(files.values()): raise SystemExit("Phase 7 files are not in the repo yet")
def short(x, n=100):
    if isinstance(x, str): return x[:n] + (f"... [{len(x)} chars]" if len(x) > n else "")
    if isinstance(x, list): return f"list[{len(x)}] " + str([short(i, 50) for i in x[:2]])
    if isinstance(x, dict): return {k: short(v, 50) for k, v in list(x.items())[:4]}
    return x
D = {}
for k, p in files.items():
    if not p: print(f"\n== {k}: NOT FOUND"); continue
    D[k] = json.load(open(p)); d = D[k]
    print(f"\n== {k}: {os.path.getsize(p)/1e6:.1f} MB | {type(d).__name__} with {len(d)} items")
    ex = d[0] if isinstance(d, list) else next(iter(d.items()))
    print("   first item:", {kk: short(v) for kk, v in ex.items()} if isinstance(ex, dict) else short(ex))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


== train: 33.2 MB | dict with 827 items
   first item: ('0000000599', {'text': ['PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 1545 1964 SCR (3) 573 CITATOR INFO : R 1970 SC 765 (9) R 1971 SC 342 (6) RF 1974 SC1185 (16) F 1984 SC 305 (9,21) R 1991 SC1557 (20,22) ACT: Election Dispute-Joinder of -parties-Joinder of candi- date who did not contest-If invalidates eletion petition- "Copy", meaning of-Defects in verification and affidavit- Maintainability of petition- Representation of the People Act , 1951 , ss. 81 , 82 , 83 , 90 . HEADNOTE: The validity of the election of the appellant to <ORG> at the third general elections held in the month of February, 1962, was challenged by two of the electors of the constituency from which the appellant was elected, by filing election petitions for s

In [22]:
import numpy as np, random
assert "D" in globals(), "re-run the Phase 7 inspection cell first"
print("split sizes:", {k: len(v) for k, v in D.items()})
for sp in ["train", "dev"]:
    q, c = D[sp], D[sp + "_candidates"]; pool = set(c)
    g = np.array([len(r["relevant_candidates"]) for r in q.values()]); allg = [x for r in q.values() for x in r["relevant_candidates"]]
    print(f"\n[{sp}] gold per query: mean {g.mean():.2f} | median {np.median(g):.0f} | min {g.min()} | p90 {np.percentile(g, 90):.0f} | max {g.max()}")
    print(f"  gold ids found in own pool: {100*np.mean([x in pool for x in allg]):.1f}% | pool docs gold for >=1 query: {100*len(set(allg) & pool)/len(pool):.1f}% | gold links per gold doc: {len(allg)/max(1, len(set(allg))):.2f}")
    print(f"  query ids that also appear in their own pool: {sum(k in pool for k in q)} of {len(q)}")
    hits = 7 * g / len(pool); P, R = hits.sum() / (7 * len(g)), hits.sum() / g.sum()
    print(f"  random-ranking Micro-F1@7 ≈ {2*P*R/(P+R):.4f} (P {P:.4f}, R {R:.4f})")
Pl = {sp: set(D[sp + "_candidates"]) for sp in ["train", "dev", "test"]}
print("\npool overlap train&dev:", len(Pl["train"] & Pl["dev"]), "| train&test:", len(Pl["train"] & Pl["test"]), "| dev&test:", len(Pl["dev"] & Pl["test"]))
print("query overlap train&dev:", len(set(D["train"]) & set(D["dev"])), "| train&test:", len(set(D["train"]) & set(D["test"])), "| dev&test:", len(set(D["dev"]) & set(D["test"])))
rng = random.Random(0); tot = 0
for name, d in [("train query", D["train"]), ("train cand", D["train_candidates"]), ("dev query", D["dev"]), ("dev cand", D["dev_candidates"]), ("test query", D["test"]), ("test cand", D["test_candidates"])]:
    recs = rng.sample(list(d.values()), min(150, len(d)))
    t = np.array([len(e.ids) for e in tokenizer.encode_batch([" ".join(r["text"])[:400000] for r in recs])]); tot += t.mean() * len(d)
    hdr = np.mean([r["text"][0].lstrip().startswith("PETITIONER") for r in d.values()])
    print(f"{name:12s} tokens p10/50/90/max: {int(np.percentile(t,10))}/{int(np.median(t))}/{int(np.percentile(t,90))}/{t.max()} | sentences (median) {int(np.median([len(r['text']) for r in recs]))} | start with 'PETITIONER': {100*hdr:.0f}%")
print(f"\nestimated total tokens across all six files: {tot/1e6:.0f} M")

split sizes: {'train': 827, 'train_candidates': 4320, 'dev': 118, 'dev_candidates': 1023, 'test': 237, 'test_candidates': 1727}

[train] gold per query: mean 6.42 | median 4 | min 0 | p90 14 | max 52
  gold ids found in own pool: 100.0% | pool docs gold for >=1 query: 95.1% | gold links per gold doc: 1.29
  query ids that also appear in their own pool: 827 of 827
  random-ranking Micro-F1@7 ≈ 0.0016 (P 0.0015, R 0.0016)

[dev] gold per query: mean 8.25 | median 5 | min 1 | p90 17 | max 92
  gold ids found in own pool: 100.0% | pool docs gold for >=1 query: 89.9% | gold links per gold doc: 1.06
  query ids that also appear in their own pool: 118 of 118
  random-ranking Micro-F1@7 ≈ 0.0074 (P 0.0081, R 0.0068)

pool overlap train&dev: 436 | train&test: 740 | dev&test: 220
query overlap train&dev: 0 | train&test: 0 | dev&test: 0
train query  tokens p10/50/90/max: 1899/5863/17328/61879 | sentences (median) 39 | start with 'PETITIONER': 34%
train cand   tokens p10/50/90/max: 1222/4100/13402

In [24]:
import os, pickle
P7_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_7"
C7 = pickle.load(open(f"{P7_DIR}/pcr_tok_cache.pkl", "rb"))
print({k: len(v) for k, v in C7.items()})

{'train': 827, 'train_candidates': 4320, 'dev': 118, 'dev_candidates': 1023, 'test': 237, 'test_candidates': 1727, 'gold': 2}


In [25]:
import time, numpy as np, torch, torch.nn.functional as F

@torch.no_grad()
def embed_split(bb, tok, ids=None, chunk=1024, bs=32):
    """Return {doc_id: (n_chunks, 2d) array}: one [last-token ; mean-token] vector per 1024-token chunk."""
    bb.eval(); ids = list(ids or tok.keys()); pieces = []
    for di, k in enumerate(ids):
        a = tok[k]
        for s in range(0, len(a), chunk): pieces.append((di, a[s:s + chunk]))
    order = sorted(range(len(pieces)), key=lambda i: len(pieces[i][1]))        # similar lengths together = little padding
    out = [None] * len(pieces)
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]; lens = [len(pieces[i][1]) for i in idx]; T = max(lens)
        x = torch.zeros(len(idx), T, dtype=torch.long)
        for j, i in enumerate(idx): x[j, :lens[j]] = torch.from_numpy(pieces[i][1].astype(np.int64))
        with torch.autocast("cuda", dtype=torch.bfloat16):
            f = bb.features(x.to(device), torch.tensor(lens, device=device))
        f = f.float().cpu().numpy()
        for j, i in enumerate(idx): out[i] = f[j]
    per_doc = {k: [] for k in ids}
    for (di, _), v in zip(pieces, out): per_doc[ids[di]].append(v)
    return {k: np.stack(v) for k, v in per_doc.items()}

def doc_matrix(ch, ids, part, pool, d):
    sl = slice(0, d) if part == "last" else slice(d, 2 * d)
    rows = []
    for k in ids:
        m = ch[k][:, sl]; rows.append(m.mean(0) if pool == "mean" else m.max(0) if pool == "max" else m[0])
    return np.stack(rows)

def f1_at_k(Qm, Cm, qids, cids, gold, center=False, k=7):
    Q, C = torch.tensor(Qm, device=device), torch.tensor(Cm, device=device)
    if center: mu = C.mean(0, keepdim=True); Q, C = Q - mu, C - mu
    S = F.normalize(Q, dim=-1) @ F.normalize(C, dim=-1).T
    pos = {c: i for i, c in enumerate(cids)}
    for qi, q in enumerate(qids):
        if q in pos: S[qi, pos[q]] = -1e9                      # a query must never retrieve itself
    top = S.topk(k, dim=1).indices.cpu().numpy()
    tp = sum(len({cids[j] for j in top[qi]} & set(gold[q])) for qi, q in enumerate(qids))
    P, R = tp / (k * len(qids)), tp / sum(len(gold[q]) for q in qids)
    return (2 * P * R / (P + R) if tp else 0.0), P, R

# ---- zero-shot baseline on dev ----
bb = make_backbone(); d = bb.d; t0 = time.time()
chq, chc = embed_split(bb, C7["dev"]), embed_split(bb, C7["dev_candidates"])
print(f"embedded dev queries + pool in {(time.time()-t0)/60:.1f} min")
qids, cids, gold = list(C7["dev"]), list(C7["dev_candidates"]), C7["gold"]["dev"]
print("random-ranking floor on dev is about 0.0074\n")
print(f"{'feature':8s} {'chunk pool':10s} {'centred':8s}   F1@7    P@7     R@7")
for part in ("mean", "last"):
    for pool in ("mean", "max", "first"):
        for center in (False, True):
            f, p, r = f1_at_k(doc_matrix(chq, qids, part, pool, d), doc_matrix(chc, cids, part, pool, d), qids, cids, gold, center)
            print(f"{part:8s} {pool:10s} {str(center):8s}  {f:.4f}  {p:.4f}  {r:.4f}")

embedded dev queries + pool in 1.2 min
random-ranking floor on dev is about 0.0074

feature  chunk pool centred    F1@7    P@7     R@7
mean     mean       False     0.2946  0.3208  0.2724
mean     mean       True      0.2979  0.3245  0.2754
mean     max        False     0.2235  0.2433  0.2066
mean     max        True      0.2368  0.2579  0.2189
mean     first      False     0.2090  0.2276  0.1932
mean     first      True      0.2135  0.2324  0.1973
last     mean       False     0.1056  0.1150  0.0976
last     mean       True      0.0978  0.1065  0.0904
last     max        False     0.0233  0.0254  0.0216
last     max        True      0.0311  0.0339  0.0288
last     first      False     0.0189  0.0206  0.0175
last     first      True      0.0222  0.0242  0.0206


In [26]:
import random, json, torch.nn as nn
R7 = globals().get("R7", {}); HN = pickle.load(open(f"{P7_DIR}/hardneg_zs.pkl", "rb"))
TRQ = [q for q, g in C7["gold"]["train"].items() if g]; print("training queries:", len(TRQ))

class Proj(nn.Module):                                # identity at init: starts exactly at the zero-shot embedding
    def __init__(self, d):
        super().__init__(); self.lin = nn.Linear(d, d); nn.init.eye_(self.lin.weight); nn.init.zeros_(self.lin.bias)
    def forward(self, x): return self.lin(x)

def rand_chunk(a, L, rng):
    if len(a) <= L: return a
    s = rng.randint(0, len(a) - L); return a[s:s + L]

def feats(bb, proj, docs, L, K, rng):                 # differentiable: K random L-token windows per doc, mean-pooled
    chunks = [rand_chunk(a, L, rng) for a in docs for _ in range(K)]
    lens = [len(c) for c in chunks]; x = torch.zeros(len(chunks), max(lens), dtype=torch.long)
    for j, c in enumerate(chunks): x[j, :lens[j]] = torch.from_numpy(c.astype(np.int64))
    with torch.autocast("cuda", dtype=torch.bfloat16):
        f = bb.features(x.to(device), torch.tensor(lens, device=device))[:, bb.d:]
    return F.normalize(proj(f.float().view(len(docs), K, -1).mean(1)), dim=-1)

@torch.no_grad()
def embed_final(bb, proj, tok):
    ch = embed_split(bb, tok); ids = list(tok.keys())
    return ids, proj(torch.tensor(doc_matrix(ch, ids, "mean", "mean", bb.d), device=device)).cpu().numpy()

def eval_dev(bb, proj):
    bb.eval(); qi, Qm = embed_final(bb, proj, C7["dev"]); ci, Cm = embed_final(bb, proj, C7["dev_candidates"])
    return f1_at_k(Qm, Cm, qi, ci, C7["gold"]["dev"], center=False)

def train_pcr(bb, name, lr, steps=600, B=8, L=512, K=1, tau=0.05, hn_top=30, lr_head=1e-3, eval_every=200, debug=False):
    proj = Proj(bb.d).to(device); rng = random.Random(0); params = [p for p in bb.parameters() if p.requires_grad]
    ntr = sum(p.numel() for p in params) + sum(p.numel() for p in proj.parameters())
    opt = torch.optim.AdamW([dict(params=list(proj.parameters()), lr=lr_head)] + ([dict(params=params, lr=lr)] if params else []), weight_decay=0.01)
    steps, eval_every = (4, 4) if debug else (steps, eval_every); warm = max(1, int(0.06 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    gold, pool, qtok = C7["gold"]["train"], C7["train_candidates"], C7["train"]
    best, t0, run, path = -1.0, time.time(), [], f"{P7_DIR}/pcr_{name}_best.pt"; bb.train(); torch.cuda.reset_peak_memory_stats()
    print(f"[{name}] trainable {ntr/1e6:.2f}M | steps {steps} | B {B} | window {L} x {K}")
    for step in range(1, steps + 1):
        qs = rng.sample(TRQ, B); pos_ids = [rng.choice(gold[q]) for q in qs]; hn_ids = [rng.choice(HN[q][:hn_top]) for q in qs]
        z = feats(bb, proj, [qtok[q] for q in qs] + [pool[p] for p in pos_ids] + [pool[h] for h in hn_ids], L, K, rng)
        logits = z[:B] @ z[B:].T / tau; cols = pos_ids + hn_ids
        mask = torch.zeros_like(logits, dtype=torch.bool)
        for i, q in enumerate(qs):
            gs = set(gold[q])
            for j, c in enumerate(cols):
                if j != i and c in gs: mask[i, j] = True          # also gold for this query: not a negative
        loss = F.cross_entropy(logits.masked_fill(mask, -1e4), torch.arange(B, device=device))
        loss.backward(); torch.nn.utils.clip_grad_norm_(params + list(proj.parameters()), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True); run.append(loss.item())
        if step % eval_every == 0 or step == steps:
            f, p, r = eval_dev(bb, proj); bb.train()
            print(f"step {step}/{steps} | loss {np.mean(run):.3f} | dev F1@7 {f:.4f} (P {p:.4f} R {r:.4f}) | {(time.time()-t0)/60:.1f} min"); run = []
            if f > best:
                best = f; torch.save(dict(bb={n: q.detach().cpu() for n, q in bb.named_parameters() if q.requires_grad}, proj=proj.state_dict()), path)
    ck = torch.load(path); bb.load_state_dict(ck["bb"], strict=False); proj.load_state_dict(ck["proj"])
    R7[name] = dict(dev_f1=best, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best dev F1@7 {best:.4f} | peak {R7[name]['peak_gb']:.1f} GB | {R7[name]['minutes']:.1f} min"); return bb, proj

@torch.no_grad()
def write_pred_pcr(bb, proj, name):
    bb.eval(); qi, Qm = embed_final(bb, proj, C7["test"]); ci, Cm = embed_final(bb, proj, C7["test_candidates"])
    S = F.normalize(torch.tensor(Qm, device=device), dim=-1) @ F.normalize(torch.tensor(Cm, device=device), dim=-1).T
    pos = {c: i for i, c in enumerate(ci)}
    for i, q in enumerate(qi):
        if q in pos: S[i, pos[q]] = -1e9                                  # never retrieve the query itself
    top = S.topk(7, dim=1).indices.cpu().numpy(); pred = {q: [ci[j] for j in top[i]] for i, q in enumerate(qi)}
    assert len(pred) == len(C7["test"]) and all(len(set(v)) == 7 and set(v) <= set(ci) and q not in v for q, v in pred.items())
    p = f"{P7_DIR}/prediction_pcr_{name}.json"; json.dump(pred, open(p, "w")); print("✅ wrote", p, "-> rename to prediction.json, zip, upload")
print("✅ retrieval toolkit ready")

training queries: 817
✅ retrieval toolkit ready


In [27]:
bb = make_backbone(); inject(bb, "lorma", 64, 128)
bb, proj = train_pcr(bb, "strided_lorma", lr=5e-4, steps=1500, B=16, eval_every=250)
write_pred_pcr(bb, proj, "strided_lorma")

[strided_lorma] trainable 3.21M | steps 1500 | B 16 | window 512 x 1
step 250/1500 | loss 2.203 | dev F1@7 0.3068 (P 0.3341 R 0.2837) | 2.2 min
step 500/1500 | loss 1.987 | dev F1@7 0.3079 (P 0.3354 R 0.2847) | 4.5 min
step 750/1500 | loss 1.810 | dev F1@7 0.3313 (P 0.3608 R 0.3063) | 6.7 min
step 1000/1500 | loss 1.668 | dev F1@7 0.3446 (P 0.3753 R 0.3186) | 9.0 min
step 1250/1500 | loss 1.645 | dev F1@7 0.3357 (P 0.3656 R 0.3104) | 11.2 min
step 1500/1500 | loss 1.546 | dev F1@7 0.3413 (P 0.3717 R 0.3155) | 13.5 min

✅ [strided_lorma] best dev F1@7 0.3446 | peak 18.2 GB | 13.5 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/prediction_pcr_strided_lorma.json -> rename to prediction.json, zip, upload
